# Library

In [ ]:

import os
from tqdm import tqdm
from datetime import datetime, timedelta
import ee

ee.Authenticate()
ee.Initialize(project='fates-mrv')

# GLO30

In [ ]:

def resample_to_10m(image):
    """
    Resample image to 10m using bilinear interpolation.
    """
    return (image
            .resample('bilinear')   # interpolation method
            )


def add_slope_aspect(img):

    """
    Adds slope and aspect bands derived from a DEM band in the input image.

    Parameters:
        img (ee.Image): An Earth Engine Image containing a band named 'DEM' representing elevation.

    Returns:
        ee.Image: An image with exactly three bands:
                  - 'Ele': elevation (renamed from 'DEM')
                  - 'Slope': slope in degrees calculated from the DEM
                  - 'Aspect': aspect in degrees calculated from the DEM
                  The original 'DEM' band is renamed to 'Ele'.
    """
    
    # Select the 'DEM' band from the input image and rename it to 'Ele' (Elevation)
    dem = img.select('DEM').rename('Ele')

    # Calculate slope (in degrees) from the elevation band
    slope = ee.Terrain.slope(dem).rename('Slope')

    # Calculate aspect (in degrees) from the elevation band
    aspect = ee.Terrain.aspect(dem).rename('Aspect')

    # Add the slope and aspect bands to the original image (with renamed  Ele band)
    # Return the image with 'Ele', 'Slope', and 'Aspect' bands
    return img.addBands([slope, aspect]).rename(['Ele', 'Slope', 'Aspect'])



if __name__ == "__main__":

    # define chunk size
    CHUNK_SIZE = 1 # per cell per chunk
    
    # Output directory name on Google Drive for exported images    
    outdir = '_glo30_signal' 
    
    # gee assets path
    parent = 'projects/fates-mrv/assets'
    
    # Ask EE for the list of *direct* children of that folder
    resp = ee.data.listAssets({'parent': parent})  # dict with key 'assets'
    assets_meta = resp.get('assets', [])           # list of dictionaries
    
    # Keep only assets whose type == 'TABLE'   (vectors/shapefiles)
    table_ids = [a['name'] for a in assets_meta if a['type'] == 'TABLE']
    
    print('Found', len(table_ids), 'vector assets')
    
    # Print all found asset IDs with index
    for i, aid in enumerate(table_ids):
        print(f'{i:2d}. {aid}')

    #  Read them in a loop
    for asset_id in tqdm(table_ids):
        #print(table_ids)

        # Filter to process only assets
        if not asset_id.endswith('2019'):
            continue  # Skip if it does not match this pattern

        # Extract basename from asset ID for naming outputs
        basename = os.path.basename(asset_id)      # ← fishnet
        print('\n───────────────────────────────────────────────')
        print('Processing:', asset_id)

        # Load the vector asset as a FeatureCollection
        fishnet = ee.FeatureCollection(asset_id)
        # Get number of features (grid cells) in the collection (client-side)
        total_cells = fishnet.size().getInfo()
        print('\nNumber of grid cells :', total_cells)
        print('Schema columns:', fishnet.first().propertyNames().getInfo())

        # spatial tiling : loop over the grid in chunks
        for offset in range(0, total_cells, CHUNK_SIZE):
            chunk_idx = offset // CHUNK_SIZE
            chunk_fc  = ee.FeatureCollection(fishnet.toList(CHUNK_SIZE, offset))
            
            # Extract geometry of chunk_fc
            chunk_geom = chunk_fc.geometry()

            # Filter the GLO-30 DEM ImageCollection to images intersecting the polygon geometry
            glo30_collection = (ee.ImageCollection('COPERNICUS/DEM/GLO30')
                               .filterBounds(chunk_geom)  # geom touched scene
                               .select('DEM')
                               # .map(resample_to_10m)
                               .map(add_slope_aspect)
                               .map(lambda image: image.toFloat())
                               # .map(lambda image: image.toDouble())
                               # .select(['Ele','Slope','Aspect'])
                              )


            # Check bands before reduction: get the number of images in the filtered collection (client-side)
            collection_size = glo30_collection.size().getInfo()
            
            if collection_size == 0:
                 # No images found for this polygon, skip processing
                print(f"No images found for chunk {chunk_idx}. Skipping...")
                continue  # skip to next chunk


            else:

                # Map over the ImageCollection to perform reduceRegions on each image
                # and add a 'date' property to each resulting feature
                def addDateAndReduce(image):
                    # Perform reduceRegions on the current image
                    reduced_features = image.reduceRegions(
                        collection=chunk_fc,  # The chunk of features
                        reducer=ee.Reducer.mean(),  # Mean per feature (polygon)
                        scale=10,  # Resolution,
                        crs='EPSG:3857',
                        tileScale=1
                    )
                    # Add the date as a property to each feature
                    return reduced_features.map(lambda feature: feature.set('date', image.date().format('YYYY-MM-dd')))
                
                # Apply the function to each image in the collection and flatten the result
                glo30_samples = (glo30_collection.map(lambda image: image)  # Fix: Map clip to each image
                                                .map(addDateAndReduce)
                                                .flatten())

                # Create a descriptive task name for export
                task_desc = f'glo30_{basename}_chunk_{chunk_idx}'
    
                # Set up and start an export task to Google Drive
                task = ee.batch.Export.table.toDrive(**{        
                    'collection': glo30_samples,
                    'description': task_desc, # set file name
                    'folder': outdir,  # folder on google drive
                    'fileFormat': 'csv'  # format
                })
    
                task.start()
                
                print(f'\n    → task: {task_desc} submitted '
                      f'({offset:,} – {min(offset+CHUNK_SIZE, total_cells)-1:,})')
                    
                # Optional: sleep to avoid submitting too many tasks too quickly
                # Each project's queue supports a maximum of 3,000 tasks
                import time
                time.sleep(3) # sleep interval to aviod creating multi outdir in google drive
                # do a pre-check of time/chunk to set this sleep value

        

# sentinel-2 

## sentinel-2 Raw signal 

In [ ]:
def scale_s2(image):
    # Select all spectral bands (B1 through B12)
    optical_bands = image.select('B.*').divide(10000)
    # Replace the original bands with scaled ones, but keep the QA and 'cs' bands as they are
    return image.addBands(optical_bands, overwrite=True)

def resample_to_10m(image):
    """
    Resample image to 10m using bilinear interpolation.
    """
    return (image
            .resample('bilinear')   # interpolation method
            )

def add_NDVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    ndvi = (nir.subtract(red).divide(nir.add(red))).rename(['NDVI'])   
    return image.addBands(ndvi)

def add_NIRv(image):
    red = image.select('B4')
    nir = image.select('B8')
    ndvi = (nir.subtract(red).divide(nir.add(red))).rename(['NDVI'])  
    nirv = (nir.multiply(ndvi)).rename(['NIRv'])   
    return image.addBands(nirv)

def add_GNDVI(image):
    """
    Calculate the Green Normalized Difference Vegetation Index (GNDVI) and add it as a new band to the input image.
    
    GNDVI is a vegetation index similar to NDVI but uses the green band instead of the red band,
    which can provide better sensitivity to chlorophyll concentration.
    
    The formula is:
    GNDVI = (NIR - Green) / (NIR + Green)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B3 (green) and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'GNDVI' containing the Green NDVI values.
    """
    
    green = image.select('B3')
    nir = image.select('B8')
    gndvi = (nir.subtract(green).divide(nir.add(green))).rename(['GNDVI'])   
    return image.addBands(gndvi)

def add_EVI1(image):
    """
    Calculate the Enhanced Vegetation Index (EVI1) and add it as a new band to the input image.
      
    The formula used here is:
    EVI1 = 2.5 * ( (NIR - Red) / (NIR + 6 * Red - 7.5 * Blue + 1) )
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B2 (blue), B4 (red), and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'EVI1' containing the Enhanced Vegetation Index values.
    """
    
    blue = image.select('B2')
    red = image.select('B4')
    nir = image.select('B8')
    evi1 = ((((nir.subtract(red))).divide(nir.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5)).rename(['EVI1'])   
    return image.addBands(evi1)


def add_EVI2(image):
    red = image.select('B4')
    nir = image.select('B8')
    evi2 = ((((nir.subtract(red))).divide(nir.add(red.multiply(2.4)).add(1))).multiply(2.5)).rename(['EVI2'])   
    return image.addBands(evi2)

def add_CIgreen(image):
    """
    Calculate the Chlorophyll Index Green (CIgreen) and add it as a new band to the input image.
    CIgreen = (NIR / Green) - 1
     
    Parameters:
    image (ee.Image): Input satellite image containing bands B3 (green) and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'CIgreen' containing the Chlorophyll Index Green values.
    """
    
    green = image.select('B3')
    nir = image.select('B8')
    cigreen = ((nir.divide(green)).subtract(1)).rename(['CIgreen'])   
    return image.addBands(cigreen)

def add_SR(image):
    red = image.select('B4')
    nir = image.select('B8')
    sr = (nir.divide(red)).rename(['SR'])   
    return image.addBands(sr)

def add_DVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    dvi = (nir.subtract(red)).rename(['DVI'])   
    return image.addBands(dvi)

def add_SAVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    savi = (((nir.subtract(red)).divide(nir.add(red).add(0.5))).multiply(1.5)).rename(['SAVI'])   
    return image.addBands(savi)

def add_OSAVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    osavi = ((nir.subtract(red)).divide(nir.add(red).add(0.16))).rename(['OSAVI'])   
    return image.addBands(osavi)
    
def add_NDWI1(image):
    swir1 = image.select('B11')
    nir = image.select('B8')
    ndwi1 = nir.subtract(swir1).divide(nir.add(swir1)).rename(['NDWI1'])   
    return image.addBands(ndwi1)

def add_NDWI2(image):
    swir2 = image.select('B12')
    nir = image.select('B8')
    ndwi2 = nir.subtract(swir2).divide(nir.add(swir2)).rename(['NDWI2'])   
    return image.addBands(ndwi2)

def add_EVIre1(image):
    blue = image.select('B2')
    red = image.select('B4')
    rededge1 = image.select('B5')
    evire1 = (((rededge1.subtract(red))).divide(rededge1.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5).rename(['EVIre1'])   
    return image.addBands(evire1)

def add_EVIre2(image):
    blue = image.select('B2')
    red = image.select('B4')
    rededge2 = image.select('B6')
    evire2 = (((rededge2.subtract(red))).divide(rededge2.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5).rename(['EVIre2'])   
    return image.addBands(evire2)

def add_EVIre3(image):
    blue = image.select('B2')
    red = image.select('B4')
    rededge3 = image.select('B7')
    evire3 = (((rededge3.subtract(red))).divide(rededge3.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5).rename(['EVIre3'])   
    return image.addBands(evire3)

def add_NDRE1(image):
    rededge1 = image.select('B5')
    nir = image.select('B8')
    ndre1 = nir.subtract(rededge1).divide(nir.add(rededge1)).rename(['NDRE1'])   
    return image.addBands(ndre1)

def add_NDRE2(image):
    rededge2 = image.select('B6')
    nir = image.select('B8')
    ndre2 = nir.subtract(rededge2).divide(nir.add(rededge2)).rename(['NDRE2'])   
    return image.addBands(ndre2)

def add_NDRE3(image):
    rededge3 = image.select('B7')
    nir = image.select('B8')
    ndre3 = nir.subtract(rededge3).divide(nir.add(rededge3)).rename(['NDRE3'])   
    return image.addBands(ndre3)

    
def add_CIre(image):
    """
    Calculate the Chlorophyll Index Red Edge (CIre) and add it as a new band to the input image.

    Parameters:
    image (ee.Image): Input satellite image containing bands B5 and B7.
    
    Returns:
    ee.Image: Original image with an additional band named 'CIre' containing the Chlorophyll Index Red Edge values.
    """
    
    rededge1 = image.select('B5')
    rededge3 = image.select('B7')
    cire = ((rededge3.divide(rededge1)).subtract(1)).rename(['CIre'])   
    return image.addBands(cire)


def add_MTCI1(image):
    """
    Calculate the MERIS Terrestrial Chlorophyll Index 1 (MTCI1) and add it as a new band to the input image.
    
    MTCI1 is used to estimate chlorophyll content in vegetation by combining near-infrared and red edge bands.
    The formula is:
    MTCI1 = (NIR - RedEdge1) / (RedEdge1 - Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red), B5 (red edge 1), and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'MTCI1' containing the MTCI1 values.
    """   
    rededge1 = image.select('B5')
    nir = image.select('B8')
    red = image.select('B4')
    mtci1 = ((nir.subtract(rededge1)).divide(rededge1.subtract(red))).rename(['MTCI1'])   
    return image.addBands(mtci1)
def add_MTCI2(image):
    rededge2 = image.select('B6')
    nir = image.select('B8')
    red = image.select('B4')
    mtci2 = ((nir.subtract(rededge2)).divide(rededge2.subtract(red))).rename(['MTCI2'])   
    return image.addBands(mtci2)

def add_MTCI3(image):
    rededge3 = image.select('B7')
    nir = image.select('B8')
    red = image.select('B4')
    mtci3 = ((nir.subtract(rededge3)).divide(rededge3.subtract(red))).rename(['MTCI3'])   
    return image.addBands(mtci3)

def add_MCARI1(image):
    """
    Calculate the Modified Chlorophyll Absorption in Reflectance Index 1 (MCARI1) and add it as a new band to the input image.
    
    MCARI1 is an index designed to estimate chlorophyll content by minimizing the effects of soil background and non-photosynthetic vegetation.
    The formula is:
    MCARI1 = [(RedEdge1 - Red) - 0.2 * (RedEdge1 - Green)] * (RedEdge1 / Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B3 (green), B4 (red), and B5 (red edge 1).
    
    Returns:
    ee.Image: Original image with an additional band named 'MCARI1' containing the MCARI1 values.
    """    
    rededge1 = image.select('B5')
    red = image.select('B4')
    green = image.select('B3')
    mcari1 = (rededge1.subtract(red).subtract(rededge1.subtract(green).multiply(0.2)).multiply(rededge1.divide(red))).rename(['MCARI1'])   
    return image.addBands(mcari1)

def add_MCARI2(image):
    rededge2 = image.select('B6')
    red = image.select('B4')
    green = image.select('B3')
    mcari2 = (rededge2.subtract(red).subtract(rededge2.subtract(green).multiply(0.2)).multiply(rededge2.divide(red))).rename(['MCARI2'])   
    return image.addBands(mcari2)

def add_MCARI3(image):
    rededge3 = image.select('B7')
    red = image.select('B4')
    green = image.select('B3')
    mcari3 = (rededge3.subtract(red).subtract(rededge3.subtract(green).multiply(0.2)).multiply(rededge3.divide(red))).rename(['MCARI3'])   
    return image.addBands(mcari3)

def add_NDI45(image):
    """
    Calculate the Normalized Difference Index between bands 4 and 5 (NDI45) and add it as a new band to the input image.
    
    NDI45 is a normalized difference index calculated using red edge 1 (B5) and red (B4) bands.
    The formula is similar to NDVI but uses different bands:
    NDI45 = (RedEdge1 - Red) / (RedEdge1 + Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red) and B5 (red edge 1).
    
    Returns:
    ee.Image: Original image with an additional band named 'NDI45' containing the NDI45 values.
    """
        
    rededge1 = image.select('B5')
    red = image.select('B4')
    ndi45 = (rededge1.subtract(red).divide(rededge1.add(red))).rename(['NDI45'])   
    return image.addBands(ndi45)

def add_PSSRa(image):
    """
    Calculate the (Pigment Specific Simple Ratio for Chlorophyll a (PSSRa) and add it as a new band to the input image.

    PSSRa = RedEdge / Red
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B7 (red edge 3) and B4 (red).
    
    Returns:
    ee.Image: Original image with an additional band named 'PSSRa' containing the PSSRa values.
    """
    rededge3 = image.select('B7')
    red = image.select('B4')
    pssra = (rededge3.divide(red)).rename(['PSSRa'])   
    return image.addBands(pssra)


def add_IRECI(image):
    rededge1 = image.select('B5')
    rededge2 = image.select('B6')
    nir = image.select('B8')
    red = image.select('B4')
    ndvi56 = (nir.subtract(red).divide(rededge1.divide(rededge2))).rename(['IRECI'])   
    return image.addBands(ndvi56)

    
def add_NDVI56(image):
    """
    Calculate the Normalized Difference Vegetation Index between bands 5 and 6 (NDVI56) and add it as a new band to the input image.

    The formula is:
    NDVI56 = (RedEdge2 - RedEdge1) / (RedEdge2 + RedEdge1)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B5 (red edge 1) and B6 (red edge 2).
    
    Returns:
    ee.Image: Original image with an additional band named 'NDVI56' containing the NDVI56 values.
    """    
    rededge1 = image.select('B5')
    rededge2 = image.select('B6')
    ndvi56 = (rededge2.subtract(rededge1).divide(rededge2.add(rededge1))).rename(['NDVI56'])   
    return image.addBands(ndvi56)

def add_NDVI57(image):
    rededge1 = image.select('B5')
    rededge3 = image.select('B7')
    ndvi57 = (rededge3.subtract(rededge1).divide(rededge3.add(rededge1))).rename(['NDVI57'])   
    return image.addBands(ndvi57)

def add_NDVI68a(image):
    nira = image.select('B8A')
    rededge2 = image.select('B6')
    ndvi68a = (nira.subtract(rededge2).divide(nira.add(rededge2))).rename(['NDVI68a'])   
    return image.addBands(ndvi68a)

def add_NDVI78a(image):
    nira = image.select('B8A')
    rededge3 = image.select('B7')
    ndvi78a = (nira.subtract(rededge3).divide(nira.add(rededge3))).rename(['NDVI78a'])   
    return image.addBands(ndvi78a)

    
def add_NLI(image):
    """
    Calculate the Normalized Leaf Index (NLI) and add it as a new band to the input image.
    
    NLI is an index used to assess leaf characteristics by combining red and red edge bands.
    The formula used here is:
    NLI = (RedEdge^2 - Red) / (RedEdge^2 + Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red) and B5 (red edge 1).
    
    Returns:
    ee.Image: Original image with an additional band named 'NLI' containing the NLI values.
    """
    red = image.select('B4')
    rededge1 = image.select('B5')
    nli = ((rededge1.multiply(rededge1)).subtract(red).divide(rededge1.multiply(rededge1).add(red))).rename(['NLI'])   
    return image.addBands(nli)


def add_kNDVI(image):
    """
    Calculate the kernel Normalized Difference Vegetation Index (kNDVI) and add it as a new band to the input image.
    The steps are:
    1. Calculate NDVI = (NIR - Red) / (NIR + Red)
    2. Apply kNDVI = tanh(NDVI2)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red) and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'kNDVI' containing the kNDVI values.
    """
        
    red = image.select('B4')
    nir = image.select('B8')
    ndvi = (nir.subtract(red).divide(nir.add(red))).rename(['NDVI'])      
    kndvi = (ndvi.expression('tanh(b*b)', {'b': ndvi})).rename(['kNDVI'])   
    return image.addBands(kndvi)


def applyQA60Mask(image):
    """
    Apply a cloud and cirrus mask to the input image using the QA60 band.
    
    The QA60 band contains quality assessment information where:
    - Bit 10 indicates clouds
    - Bit 11 indicates cirrus clouds
    
    This function masks out pixels where either clouds or cirrus are detected.
    
    Parameters:
    image (ee.Image): Input satellite image containing the QA60 band.
    
    Returns:
    ee.Image: Masked image with clouds and cirrus pixels removed.
    """    
    qa = image.select('QA60')
    # Bits 10 and 11 are clouds and cirrus, respectively.
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    # Both flags should be set to zero, indicating clear conditions.
    mask = (qa.bitwiseAnd(cloud_bit_mask).eq(0).And(qa.bitwiseAnd(cirrus_bit_mask).eq(0)))
    return image.updateMask(mask)


def applyCLDPRBMask(image):
    """
    Apply a cloud probability mask to the input image using the MSK_CLDPRB band.
    
    The MSK_CLDPRB band indicates cloud probability, where:
    - 0 means clear pixels
    - Non-zero values indicate varying probabilities of cloud presence
    
    This function masks out pixels with any cloud probability (i.e., keeps only pixels with cloud probability = 0).
    
    Parameters:
    image (ee.Image): Input satellite image containing the MSK_CLDPRB band.
    
    Returns:
    ee.Image: Masked image with probable cloud pixels removed.
    """
    
    # Select the MSK_CLDPRB band (cloud probability mask)
    cloudProb = image.select('MSK_CLDPRB')
    # Mask pixels where MSK_CLDPRB is not equal to 0 (clear pixels)
    mask = cloudProb.eq(0)
    # mask =  cloudProb.lte(0.000001)  
    return image.updateMask(mask)


def applyCloudScoreMask(image):
    """
    Applies Cloud Score+ masking to a Sentinel-2 image.
    Assumes the collection has already been linked with the csPlus collection.
    """
    # Define the threshold
    CLEAR_THRESHOLD = 0.60 
    
    # Select the 'cs' (or 'cs_cdf') band linked from Cloud Score+
    # Ensure the band name matches what you linked (QA_BAND)
    qa_mask = image.select('cs').gte(CLEAR_THRESHOLD)
    
    # Update the image mask and return
    return image.updateMask(qa_mask)


    

if __name__ == "__main__":

    # define chunk size
    CHUNK_SIZE = 1 # per cell per chunk
    
    # Output directory name on Google Drive for exported images   
    outdir = '_raw_sentinel2'
    
    # gee assets path
    parent = 'projects/fates-mrv/assets'
    
    # Ask EE for the list of *direct* children of that folder
    resp = ee.data.listAssets({'parent': parent})  # dict with key 'assets'
    assets_meta = resp.get('assets', [])           # list of dictionaries
    
    # Keep only assets whose type == 'TABLE'   (vectors/shapefiles)
    table_ids = [a['name'] for a in assets_meta if a['type'] == 'TABLE']
    print('Found', len(table_ids), 'vector assets')
    
    # Print all found asset IDs with index
    for i, aid in enumerate(table_ids):
        print(f'{i:2d}. {aid}')
   
    # 1. Define the Quarters
    year = 2019     
    START_DATE = f'{year}-01-01'
    END_DATE   = f'{year}-12-31'
    
    #  Read them in a loop
    for asset_id in tqdm(table_ids):
        #print(table_ids)

        # Filter to process only assets
        if not asset_id.endswith(f'{year}'):
            continue  # Skip if it does not end with 'fishnet'

        # Extract basename from asset ID for naming outputs
        basename = os.path.basename(asset_id)      # ← fishnet_25000m_2019_25m_190
        print('\n───────────────────────────────────────────────')
        print('Processing:', asset_id)

        # Load the vector asset as a FeatureCollection
        fishnet = ee.FeatureCollection(asset_id)
        total_cells = fishnet.size().getInfo()
        print('\nNumber of grid cells :', total_cells)
        print('Schema columns:', fishnet.first().propertyNames().getInfo())


        # spatial tiling : loop over the grid in chunks
        for offset in range(0, total_cells, CHUNK_SIZE):
            chunk_idx = offset // CHUNK_SIZE
            chunk_fc  = ee.FeatureCollection(fishnet.toList(CHUNK_SIZE, offset))
            chunk_geom = chunk_fc.geometry()
            
            #  Build the ImageCollection and reduce it with the compound reducer
            s2_collection = (ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
                                .filterBounds(chunk_geom)                        
                                .filterDate(START_DATE, END_DATE)
                                .linkCollection(ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED'), ['cs'])
                                .map(scale_s2)
                                # .map(resample_to_10m)
                                .map(applyQA60Mask)
                                .map(applyCloudScoreMask)
                                .map(add_NIRv)
                                .map(add_GNDVI)
                                .map(add_EVI1)
                                .map(add_EVI2)
                                .map(add_CIgreen)
                                .map(add_SR)
                                .map(add_DVI)
                                .map(add_SAVI)
                                .map(add_OSAVI)
                                .map(add_NDWI1)
                                .map(add_NDWI2)
                                .map(add_EVIre1)
                                .map(add_EVIre2)
                                .map(add_EVIre3)
                                .map(add_NDRE1)
                                .map(add_NDRE2)
                                .map(add_NDRE3)
                                .map(add_CIre)
                                .map(add_MTCI1)
                                .map(add_MTCI2)
                                .map(add_MTCI3)
                                .map(add_MCARI1)
                                .map(add_MCARI2)
                                .map(add_MCARI3)
                                .map(add_NDI45)
                                .map(add_PSSRa)
                                .map(add_IRECI)
                                .map(add_NDVI56)
                                .map(add_NDVI57)
                                .map(add_NDVI68a)
                                .map(add_NDVI78a)
                                .map(add_NLI)
                                .map(add_kNDVI)
                                .select(['B1','B2', 'B3', 'B4', 'B5', 'B6','B7','B8','B8A', 'B9','B10','B11','B12',
                                         'NIRv', 'GNDVI', 'EVI1', 'EVI2', 'CIgreen', 'SR', 'DVI', 'SAVI', 'OSAVI', 'NDWI1', 'NDWI2', 'EVIre1', 'EVIre2', 'EVIre3',
                                         'NDRE1','NDRE2','NDRE3', 'CIre','MTCI1', 'MTCI2', 'MTCI3', 'MCARI1', 'MCARI2', 'MCARI3', 'NDI45', 'PSSRa', 'IRECI', 'NDVI56', 'NDVI57', 'NDVI68a',
                                         'NDVI78a', 'NLI', 'kNDVI'                                      
                                        ])
                                .map(lambda image: image.toFloat())
                                # .map(lambda image: image.toDouble())
                            )
                                

            # Check bands before reduction
            collection_size = s2_collection.size().getInfo()
            
            if collection_size == 0:
                # No images found for this feature, skip processing
                print(f"No images found for chunk {chunk_idx}. Skipping...")
                continue  # skip to next chunk
                
            else:

            
                # Map over the ImageCollection to perform reduceRegions on each image
                # and add a 'date' property to each resulting feature
                def addDateAndReduce(image):
                    # Perform reduceRegions on the current image
                    reduced_features = image.reduceRegions(
                        collection=chunk_fc,  # The chunk of features
                        reducer=ee.Reducer.mean(),  # Mean per feature (polygon)
                        scale=10,  # Resolution/ use near
                        crs='EPSG:3857',
                        tileScale=1
                    )
                    # Add the date as a property to each feature
                    return reduced_features.map(lambda feature: feature.set('date', image.date().format('YYYY-MM-dd')))
                
                # Apply the function to each image in the collection and flatten the result
                s2_rawTimeSeries = (s2_collection.map(lambda image: image)  # Fix: Map clip to each image
                                                .map(addDateAndReduce)
                                                .flatten())
                
                # Create a descriptive task name for export
                task_desc = f'sentinel-2_{basename}_rawTimeSeries_chunk_{chunk_idx}'
                
                # Set up and start an export task to Google Drive
                task = ee.batch.Export.table.toDrive(**{
                    'collection': s2_rawTimeSeries,  # The combined FeatureCollection with time series
                    'description': task_desc,  # Set file name
                    'folder': outdir,          # Folder on Google Drive
                    'fileFormat': 'CSV',       # Format (use uppercase for consistency)
                    # 'selectors': ['date'] + list(s2_collection.first().bandNames().getInfo())  # Include date and bands
                })
                
                task.start()
                
                print(f'\n    → Task: {task_desc} submitted '
                      f'({offset:,} – {min(offset+CHUNK_SIZE, total_cells)-1:,})')
    
                # Optional: sleep to avoid submitting too many tasks too quickly
                # Each project's queue supports a maximum of 3,000 tasks
                import time
                time.sleep(3) # sleep interval to aviod creating multi outdir in google drive
                # do a pre-check of time/chunk to set this sleep value

        

## sentinel-2 monthly median

In [ ]:
def scale_s2(image):
    # Select all spectral bands (B1 through B12)
    optical_bands = image.select('B.*').divide(10000)
    # Replace the original bands with scaled ones, but keep the QA and 'cs' bands as they are
    return image.addBands(optical_bands, overwrite=True)

def resample_to_10m(image):
    """
    Resample image to 10m using bilinear interpolation.
    """
    return (image
            .resample('bilinear')   # interpolation method
            )

def add_NDVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    ndvi = (nir.subtract(red).divide(nir.add(red))).rename(['NDVI'])   
    return image.addBands(ndvi)

def add_NIRv(image):
    red = image.select('B4')
    nir = image.select('B8')
    ndvi = (nir.subtract(red).divide(nir.add(red))).rename(['NDVI'])  
    nirv = (nir.multiply(ndvi)).rename(['NIRv'])   
    return image.addBands(nirv)

def add_GNDVI(image):
    """
    Calculate the Green Normalized Difference Vegetation Index (GNDVI) and add it as a new band to the input image.
    
    GNDVI is a vegetation index similar to NDVI but uses the green band instead of the red band,
    which can provide better sensitivity to chlorophyll concentration.
    
    The formula is:
    GNDVI = (NIR - Green) / (NIR + Green)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B3 (green) and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'GNDVI' containing the Green NDVI values.
    """
    
    green = image.select('B3')
    nir = image.select('B8')
    gndvi = (nir.subtract(green).divide(nir.add(green))).rename(['GNDVI'])   
    return image.addBands(gndvi)

def add_EVI1(image):
    """
    Calculate the Enhanced Vegetation Index (EVI1) and add it as a new band to the input image.
      
    The formula used here is:
    EVI1 = 2.5 * ( (NIR - Red) / (NIR + 6 * Red - 7.5 * Blue + 1) )
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B2 (blue), B4 (red), and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'EVI1' containing the Enhanced Vegetation Index values.
    """
    
    blue = image.select('B2')
    red = image.select('B4')
    nir = image.select('B8')
    evi1 = ((((nir.subtract(red))).divide(nir.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5)).rename(['EVI1'])   
    return image.addBands(evi1)


def add_EVI2(image):
    red = image.select('B4')
    nir = image.select('B8')
    evi2 = ((((nir.subtract(red))).divide(nir.add(red.multiply(2.4)).add(1))).multiply(2.5)).rename(['EVI2'])   
    return image.addBands(evi2)

def add_CIgreen(image):
    """
    Calculate the Chlorophyll Index Green (CIgreen) and add it as a new band to the input image.
    CIgreen = (NIR / Green) - 1
     
    Parameters:
    image (ee.Image): Input satellite image containing bands B3 (green) and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'CIgreen' containing the Chlorophyll Index Green values.
    """
    
    green = image.select('B3')
    nir = image.select('B8')
    cigreen = ((nir.divide(green)).subtract(1)).rename(['CIgreen'])   
    return image.addBands(cigreen)

def add_SR(image):
    red = image.select('B4')
    nir = image.select('B8')
    sr = (nir.divide(red)).rename(['SR'])   
    return image.addBands(sr)

def add_DVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    dvi = (nir.subtract(red)).rename(['DVI'])   
    return image.addBands(dvi)

def add_SAVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    savi = (((nir.subtract(red)).divide(nir.add(red).add(0.5))).multiply(1.5)).rename(['SAVI'])   
    return image.addBands(savi)

def add_OSAVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    osavi = ((nir.subtract(red)).divide(nir.add(red).add(0.16))).rename(['OSAVI'])   
    return image.addBands(osavi)
    
def add_NDWI1(image):
    swir1 = image.select('B11')
    nir = image.select('B8')
    ndwi1 = nir.subtract(swir1).divide(nir.add(swir1)).rename(['NDWI1'])   
    return image.addBands(ndwi1)

def add_NDWI2(image):
    swir2 = image.select('B12')
    nir = image.select('B8')
    ndwi2 = nir.subtract(swir2).divide(nir.add(swir2)).rename(['NDWI2'])   
    return image.addBands(ndwi2)

def add_EVIre1(image):
    blue = image.select('B2')
    red = image.select('B4')
    rededge1 = image.select('B5')
    evire1 = (((rededge1.subtract(red))).divide(rededge1.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5).rename(['EVIre1'])   
    return image.addBands(evire1)

def add_EVIre2(image):
    blue = image.select('B2')
    red = image.select('B4')
    rededge2 = image.select('B6')
    evire2 = (((rededge2.subtract(red))).divide(rededge2.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5).rename(['EVIre2'])   
    return image.addBands(evire2)

def add_EVIre3(image):
    blue = image.select('B2')
    red = image.select('B4')
    rededge3 = image.select('B7')
    evire3 = (((rededge3.subtract(red))).divide(rededge3.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5).rename(['EVIre3'])   
    return image.addBands(evire3)

def add_NDRE1(image):
    rededge1 = image.select('B5')
    nir = image.select('B8')
    ndre1 = nir.subtract(rededge1).divide(nir.add(rededge1)).rename(['NDRE1'])   
    return image.addBands(ndre1)

def add_NDRE2(image):
    rededge2 = image.select('B6')
    nir = image.select('B8')
    ndre2 = nir.subtract(rededge2).divide(nir.add(rededge2)).rename(['NDRE2'])   
    return image.addBands(ndre2)

def add_NDRE3(image):
    rededge3 = image.select('B7')
    nir = image.select('B8')
    ndre3 = nir.subtract(rededge3).divide(nir.add(rededge3)).rename(['NDRE3'])   
    return image.addBands(ndre3)

    
def add_CIre(image):
    """
    Calculate the Chlorophyll Index Red Edge (CIre) and add it as a new band to the input image.

    Parameters:
    image (ee.Image): Input satellite image containing bands B5 and B7.
    
    Returns:
    ee.Image: Original image with an additional band named 'CIre' containing the Chlorophyll Index Red Edge values.
    """
    
    rededge1 = image.select('B5')
    rededge3 = image.select('B7')
    cire = ((rededge3.divide(rededge1)).subtract(1)).rename(['CIre'])   
    return image.addBands(cire)


def add_MTCI1(image):
    """
    Calculate the MERIS Terrestrial Chlorophyll Index 1 (MTCI1) and add it as a new band to the input image.
    
    MTCI1 is used to estimate chlorophyll content in vegetation by combining near-infrared and red edge bands.
    The formula is:
    MTCI1 = (NIR - RedEdge1) / (RedEdge1 - Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red), B5 (red edge 1), and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'MTCI1' containing the MTCI1 values.
    """   
    rededge1 = image.select('B5')
    nir = image.select('B8')
    red = image.select('B4')
    mtci1 = ((nir.subtract(rededge1)).divide(rededge1.subtract(red))).rename(['MTCI1'])   
    return image.addBands(mtci1)
def add_MTCI2(image):
    rededge2 = image.select('B6')
    nir = image.select('B8')
    red = image.select('B4')
    mtci2 = ((nir.subtract(rededge2)).divide(rededge2.subtract(red))).rename(['MTCI2'])   
    return image.addBands(mtci2)

def add_MTCI3(image):
    rededge3 = image.select('B7')
    nir = image.select('B8')
    red = image.select('B4')
    mtci3 = ((nir.subtract(rededge3)).divide(rededge3.subtract(red))).rename(['MTCI3'])   
    return image.addBands(mtci3)

def add_MCARI1(image):
    """
    Calculate the Modified Chlorophyll Absorption in Reflectance Index 1 (MCARI1) and add it as a new band to the input image.
    
    MCARI1 is an index designed to estimate chlorophyll content by minimizing the effects of soil background and non-photosynthetic vegetation.
    The formula is:
    MCARI1 = [(RedEdge1 - Red) - 0.2 * (RedEdge1 - Green)] * (RedEdge1 / Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B3 (green), B4 (red), and B5 (red edge 1).
    
    Returns:
    ee.Image: Original image with an additional band named 'MCARI1' containing the MCARI1 values.
    """    
    rededge1 = image.select('B5')
    red = image.select('B4')
    green = image.select('B3')
    mcari1 = (rededge1.subtract(red).subtract(rededge1.subtract(green).multiply(0.2)).multiply(rededge1.divide(red))).rename(['MCARI1'])   
    return image.addBands(mcari1)

def add_MCARI2(image):
    rededge2 = image.select('B6')
    red = image.select('B4')
    green = image.select('B3')
    mcari2 = (rededge2.subtract(red).subtract(rededge2.subtract(green).multiply(0.2)).multiply(rededge2.divide(red))).rename(['MCARI2'])   
    return image.addBands(mcari2)

def add_MCARI3(image):
    rededge3 = image.select('B7')
    red = image.select('B4')
    green = image.select('B3')
    mcari3 = (rededge3.subtract(red).subtract(rededge3.subtract(green).multiply(0.2)).multiply(rededge3.divide(red))).rename(['MCARI3'])   
    return image.addBands(mcari3)

def add_NDI45(image):
    """
    Calculate the Normalized Difference Index between bands 4 and 5 (NDI45) and add it as a new band to the input image.
    
    NDI45 is a normalized difference index calculated using red edge 1 (B5) and red (B4) bands.
    The formula is similar to NDVI but uses different bands:
    NDI45 = (RedEdge1 - Red) / (RedEdge1 + Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red) and B5 (red edge 1).
    
    Returns:
    ee.Image: Original image with an additional band named 'NDI45' containing the NDI45 values.
    """
        
    rededge1 = image.select('B5')
    red = image.select('B4')
    ndi45 = (rededge1.subtract(red).divide(rededge1.add(red))).rename(['NDI45'])   
    return image.addBands(ndi45)

def add_PSSRa(image):
    """
    Calculate the (Pigment Specific Simple Ratio for Chlorophyll a (PSSRa) and add it as a new band to the input image.

    PSSRa = RedEdge / Red
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B7 (red edge 3) and B4 (red).
    
    Returns:
    ee.Image: Original image with an additional band named 'PSSRa' containing the PSSRa values.
    """
    rededge3 = image.select('B7')
    red = image.select('B4')
    pssra = (rededge3.divide(red)).rename(['PSSRa'])   
    return image.addBands(pssra)


def add_IRECI(image):
    rededge1 = image.select('B5')
    rededge2 = image.select('B6')
    nir = image.select('B8')
    red = image.select('B4')
    ndvi56 = (nir.subtract(red).divide(rededge1.divide(rededge2))).rename(['IRECI'])   
    return image.addBands(ndvi56)

    
def add_NDVI56(image):
    """
    Calculate the Normalized Difference Vegetation Index between bands 5 and 6 (NDVI56) and add it as a new band to the input image.

    The formula is:
    NDVI56 = (RedEdge2 - RedEdge1) / (RedEdge2 + RedEdge1)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B5 (red edge 1) and B6 (red edge 2).
    
    Returns:
    ee.Image: Original image with an additional band named 'NDVI56' containing the NDVI56 values.
    """    
    rededge1 = image.select('B5')
    rededge2 = image.select('B6')
    ndvi56 = (rededge2.subtract(rededge1).divide(rededge2.add(rededge1))).rename(['NDVI56'])   
    return image.addBands(ndvi56)

def add_NDVI57(image):
    rededge1 = image.select('B5')
    rededge3 = image.select('B7')
    ndvi57 = (rededge3.subtract(rededge1).divide(rededge3.add(rededge1))).rename(['NDVI57'])   
    return image.addBands(ndvi57)

def add_NDVI68a(image):
    nira = image.select('B8A')
    rededge2 = image.select('B6')
    ndvi68a = (nira.subtract(rededge2).divide(nira.add(rededge2))).rename(['NDVI68a'])   
    return image.addBands(ndvi68a)

def add_NDVI78a(image):
    nira = image.select('B8A')
    rededge3 = image.select('B7')
    ndvi78a = (nira.subtract(rededge3).divide(nira.add(rededge3))).rename(['NDVI78a'])   
    return image.addBands(ndvi78a)

    
def add_NLI(image):
    """
    Calculate the Normalized Leaf Index (NLI) and add it as a new band to the input image.
    
    NLI is an index used to assess leaf characteristics by combining red and red edge bands.
    The formula used here is:
    NLI = (RedEdge^2 - Red) / (RedEdge^2 + Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red) and B5 (red edge 1).
    
    Returns:
    ee.Image: Original image with an additional band named 'NLI' containing the NLI values.
    """
    red = image.select('B4')
    rededge1 = image.select('B5')
    nli = ((rededge1.multiply(rededge1)).subtract(red).divide(rededge1.multiply(rededge1).add(red))).rename(['NLI'])   
    return image.addBands(nli)


def add_kNDVI(image):
    """
    Calculate the kernel Normalized Difference Vegetation Index (kNDVI) and add it as a new band to the input image.
    The steps are:
    1. Calculate NDVI = (NIR - Red) / (NIR + Red)
    2. Apply kNDVI = tanh(NDVI2)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red) and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'kNDVI' containing the kNDVI values.
    """
        
    red = image.select('B4')
    nir = image.select('B8')
    ndvi = (nir.subtract(red).divide(nir.add(red))).rename(['NDVI'])      
    kndvi = (ndvi.expression('tanh(b*b)', {'b': ndvi})).rename(['kNDVI'])   
    return image.addBands(kndvi)


def applyQA60Mask(image):
    """
    Apply a cloud and cirrus mask to the input image using the QA60 band.
    
    The QA60 band contains quality assessment information where:
    - Bit 10 indicates clouds
    - Bit 11 indicates cirrus clouds
    
    This function masks out pixels where either clouds or cirrus are detected.
    
    Parameters:
    image (ee.Image): Input satellite image containing the QA60 band.
    
    Returns:
    ee.Image: Masked image with clouds and cirrus pixels removed.
    """    
    qa = image.select('QA60')
    # Bits 10 and 11 are clouds and cirrus, respectively.
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    # Both flags should be set to zero, indicating clear conditions.
    mask = (qa.bitwiseAnd(cloud_bit_mask).eq(0).And(qa.bitwiseAnd(cirrus_bit_mask).eq(0)))
    return image.updateMask(mask)


def applyCLDPRBMask(image):
    """
    Apply a cloud probability mask to the input image using the MSK_CLDPRB band.
    
    The MSK_CLDPRB band indicates cloud probability, where:
    - 0 means clear pixels
    - Non-zero values indicate varying probabilities of cloud presence
    
    This function masks out pixels with any cloud probability (i.e., keeps only pixels with cloud probability = 0).
    
    Parameters:
    image (ee.Image): Input satellite image containing the MSK_CLDPRB band.
    
    Returns:
    ee.Image: Masked image with probable cloud pixels removed.
    """
    
    # Select the MSK_CLDPRB band (cloud probability mask)
    cloudProb = image.select('MSK_CLDPRB')
    # Mask pixels where MSK_CLDPRB is not equal to 0 (clear pixels)
    mask = cloudProb.eq(0)
    # mask =  cloudProb.lte(0.000001)  
    return image.updateMask(mask)


def applyCloudScoreMask(image):
    """
    Applies Cloud Score+ masking to a Sentinel-2 image.
    Assumes the collection has already been linked with the csPlus collection.
    """
    # Define the threshold
    CLEAR_THRESHOLD = 0.60 
    
    # Select the 'cs' (or 'cs_cdf') band linked from Cloud Score+
    # Ensure the band name matches what you linked (QA_BAND)
    qa_mask = image.select('cs').gte(CLEAR_THRESHOLD)
    
    # Update the image mask and return
    return image.updateMask(qa_mask)

# New function to prefix computed indices with 'S2_' while keeping raw bands (B*) unchanged
def add_s2_prefix_to_all_bands(image):
    """
    Renames EVERY band in the image by adding 'S2_' prefix.
    This applies to both raw Sentinel-2 bands (e.g., 'B4' → 'S2_B4')
    and computed indices (e.g., 'NDVI' → 'S2_NDVI', 'EVI1' → 'S2_EVI1').
    """
    band_names = image.bandNames()
    new_names = band_names.map(lambda name: ee.String('S2_').cat(name))
    return image.rename(new_names)
    

if __name__ == "__main__":

    # define chunk size
    CHUNK_SIZE = 1 # per cell per chunk
    
    # Output directory name on Google Drive for exported images   
    outdir = '_sentinel2_monthly_median'
    
    # gee assets path
    parent = 'projects/fates-mrv/assets'
    
    # Ask EE for the list of *direct* children of that folder
    resp = ee.data.listAssets({'parent': parent})  # dict with key 'assets'
    assets_meta = resp.get('assets', [])           # list of dictionaries
    
    # Keep only assets whose type == 'TABLE'   (vectors/shapefiles)
    table_ids = [a['name'] for a in assets_meta if a['type'] == 'TABLE']
    print('Found', len(table_ids), 'vector assets')
    
    # Print all found asset IDs with index
    for i, aid in enumerate(table_ids):
        print(f'{i:2d}. {aid}')

    # 1. Define the Quarters
    year = 2019      
    START_DATE = f'{year}-01-01'
    END_DATE   = f'{year}-12-31'
      
    # Dynamic monthly
    months = [
        ('Jan', f'{year}-01-01', f'{year}-01-31'),
        ('Feb', f'{year}-02-01', f'{year}-02-28'),
        ('Mar', f'{year}-03-01', f'{year}-03-31'),
        ('Apr', f'{year}-04-01', f'{year}-04-30'),
        ('May', f'{year}-05-01', f'{year}-05-31'),
        ('Jun', f'{year}-06-01', f'{year}-06-30'),
        ('Jul', f'{year}-07-01', f'{year}-07-31'),
        ('Aug', f'{year}-08-01', f'{year}-08-31'),
        ('Sep', f'{year}-09-01', f'{year}-09-30'),
        ('Oct', f'{year}-10-01', f'{year}-10-31'),
        ('Nov', f'{year}-11-01', f'{year}-11-30'),
        ('Dec', f'{year}-12-01', f'{year}-12-31'),
    ]

    #  Read them in a loop
    for asset_id in tqdm(table_ids):
        #print(table_ids)

        # Filter to process only assets
        if not asset_id.endswith(f'{year}'):
            continue  # Skip if it does not end with 'fishnet'

        # Extract basename from asset ID for naming outputs
        basename = os.path.basename(asset_id)      # ← fishnet_25000m_2019_25m_190
        print('\n───────────────────────────────────────────────')
        print('Processing:', asset_id)

        # Load the vector asset as a FeatureCollection
        fishnet = ee.FeatureCollection(asset_id)
        total_cells = fishnet.size().getInfo()
        print('\nNumber of grid cells :', total_cells)
        print('Schema columns:', fishnet.first().propertyNames().getInfo())


        # spatial tiling : loop over the grid in chunks
        # for offset in range(0, total_cells, CHUNK_SIZE):
        for offset in range(2471, total_cells):
            chunk_idx = offset // CHUNK_SIZE
            chunk_fc  = ee.FeatureCollection(fishnet.toList(CHUNK_SIZE, offset))
            chunk_geom = chunk_fc.geometry()
            
            #  Build the ImageCollection and reduce it with the compound reducer
            s2_collection = (ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
                                .filterBounds(chunk_geom)                        
                                .filterDate(START_DATE, END_DATE)
                                .linkCollection(ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED'), ['cs'])
                                .map(scale_s2)
                                # .map(resample_to_10m)
                                .map(applyQA60Mask)
                                .map(applyCloudScoreMask)
                                .map(add_NIRv)
                                .map(add_GNDVI)
                                .map(add_EVI1)
                                .map(add_EVI2)
                                .map(add_CIgreen)
                                .map(add_SR)
                                .map(add_DVI)
                                .map(add_SAVI)
                                .map(add_OSAVI)
                                .map(add_NDWI1)
                                .map(add_NDWI2)
                                .map(add_EVIre1)
                                .map(add_EVIre2)
                                .map(add_EVIre3)
                                .map(add_NDRE1)
                                .map(add_NDRE2)
                                .map(add_NDRE3)
                                .map(add_CIre)
                                .map(add_MTCI1)
                                .map(add_MTCI2)
                                .map(add_MTCI3)
                                .map(add_MCARI1)
                                .map(add_MCARI2)
                                .map(add_MCARI3)
                                .map(add_NDI45)
                                .map(add_PSSRa)
                                .map(add_IRECI)
                                .map(add_NDVI56)
                                .map(add_NDVI57)
                                .map(add_NDVI68a)
                                .map(add_NDVI78a)
                                .map(add_NLI)
                                .map(add_kNDVI)
                                .select(['B1','B2', 'B3', 'B4', 'B5', 'B6','B7','B8','B8A', 'B9','B10','B11','B12',
                                         'NIRv', 'GNDVI', 'EVI1', 'EVI2', 'CIgreen', 'SR', 'DVI', 'SAVI', 'OSAVI', 'NDWI1', 'NDWI2', 'EVIre1', 'EVIre2', 'EVIre3',
                                         'NDRE1','NDRE2','NDRE3', 'CIre','MTCI1', 'MTCI2', 'MTCI3', 'MCARI1', 'MCARI2', 'MCARI3', 'NDI45', 'PSSRa', 'IRECI', 'NDVI56', 'NDVI57', 'NDVI68a',
                                         'NDVI78a', 'NLI', 'kNDVI'                                      
                                        ])
                                .map(lambda image: image.toFloat())
                                # .map(lambda image: image.toDouble())
                                .map(add_s2_prefix_to_all_bands)
                            )
                                

            if s2_collection.size().getInfo() == 0:
                # No images found for this feature, skip processing
                print(f"No images found for chunk {chunk_idx} in {year}. Skipping...")
                continue  # skip to next chunk

            # monthly processing
            all_monthly_fc = None
            has_month_data = False

            for m_name, m_start, m_end in months:
                m_col = s2_collection.filterDate(m_start, m_end)
                if m_col.size().getInfo() == 0:
                    print(f"  No images in {m_name} for chunk {chunk_idx}")
                    continue

                # composite for the month
                composite = m_col.median()

                # Zonal mean extraction
                reduced = composite.reduceRegions(
                    collection=chunk_fc,
                    reducer=ee.Reducer.mean(),
                    scale=10,
                    crs='EPSG:3857',
                    tileScale=1
                )

                reduced_q = reduced.map(lambda f: f.set({
                    'quarter': m_name,
                    'year': year,
                    'start_date': m_start,
                    'end_date': m_end
                }))

                if all_monthly_fc is None:
                    all_monthly_fc = reduced_q
                else:
                    all_monthly_fc = allmonthly_fc.merge(reduced_q)
                
                has_month_data = True

            if not has_month_data:
                print(f"No valid monthly data for chunk {chunk_idx}. Skipping export.")
                continue        

                
            # Create a descriptive task name for export
            task_desc = f'sentinel-2_{basename}_monthly_chunk_{chunk_idx}'
            
            # Set up and start an export task to Google Drive
            task = ee.batch.Export.table.toDrive(**{
                'collection': all_monthly_fc,  # The combined FeatureCollection with time series
                'description': task_desc,  # Set file name
                'folder': outdir,          # Folder on Google Drive
                'fileFormat': 'CSV',       # Format (use uppercase for consistency)
                # 'selectors': ['date'] + list(s2_collection.first().bandNames().getInfo())  # Include date and bands
            })
            
            task.start()
            
            print(f'\n    → Task: {task_desc} submitted '
                  f'({offset:,} – {min(offset+CHUNK_SIZE, total_cells)-1:,})')

            # Optional: sleep to avoid submitting too many tasks too quickly
            # Each project's queue supports a maximum of 3,000 tasks
            import time
            # time.sleep(3) # sleep interval to aviod creating multi outdir in google drive
            # do a pre-check of time/chunk to set this sleep value

        

## sentinel-2 quarterly median

In [ ]:
def scale_s2(image):
    # Select all spectral bands (B1 through B12)
    optical_bands = image.select('B.*').divide(10000)
    # Replace the original bands with scaled ones, but keep the QA and 'cs' bands as they are
    return image.addBands(optical_bands, overwrite=True)

def resample_to_10m(image):
    """
    Resample image to 10m using bilinear interpolation.
    """
    return (image
            .resample('bilinear')   # interpolation method
            )

def add_NDVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    ndvi = (nir.subtract(red).divide(nir.add(red))).rename(['NDVI'])   
    return image.addBands(ndvi)

def add_NIRv(image):
    red = image.select('B4')
    nir = image.select('B8')
    ndvi = (nir.subtract(red).divide(nir.add(red))).rename(['NDVI'])  
    nirv = (nir.multiply(ndvi)).rename(['NIRv'])   
    return image.addBands(nirv)

def add_GNDVI(image):
    """
    Calculate the Green Normalized Difference Vegetation Index (GNDVI) and add it as a new band to the input image.
    
    GNDVI is a vegetation index similar to NDVI but uses the green band instead of the red band,
    which can provide better sensitivity to chlorophyll concentration.
    
    The formula is:
    GNDVI = (NIR - Green) / (NIR + Green)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B3 (green) and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'GNDVI' containing the Green NDVI values.
    """
    
    green = image.select('B3')
    nir = image.select('B8')
    gndvi = (nir.subtract(green).divide(nir.add(green))).rename(['GNDVI'])   
    return image.addBands(gndvi)

def add_EVI1(image):
    """
    Calculate the Enhanced Vegetation Index (EVI1) and add it as a new band to the input image.
      
    The formula used here is:
    EVI1 = 2.5 * ( (NIR - Red) / (NIR + 6 * Red - 7.5 * Blue + 1) )
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B2 (blue), B4 (red), and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'EVI1' containing the Enhanced Vegetation Index values.
    """
    
    blue = image.select('B2')
    red = image.select('B4')
    nir = image.select('B8')
    evi1 = ((((nir.subtract(red))).divide(nir.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5)).rename(['EVI1'])   
    return image.addBands(evi1)


def add_EVI2(image):
    red = image.select('B4')
    nir = image.select('B8')
    evi2 = ((((nir.subtract(red))).divide(nir.add(red.multiply(2.4)).add(1))).multiply(2.5)).rename(['EVI2'])   
    return image.addBands(evi2)

def add_CIgreen(image):
    """
    Calculate the Chlorophyll Index Green (CIgreen) and add it as a new band to the input image.
    CIgreen = (NIR / Green) - 1
     
    Parameters:
    image (ee.Image): Input satellite image containing bands B3 (green) and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'CIgreen' containing the Chlorophyll Index Green values.
    """
    
    green = image.select('B3')
    nir = image.select('B8')
    cigreen = ((nir.divide(green)).subtract(1)).rename(['CIgreen'])   
    return image.addBands(cigreen)

def add_SR(image):
    red = image.select('B4')
    nir = image.select('B8')
    sr = (nir.divide(red)).rename(['SR'])   
    return image.addBands(sr)

def add_DVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    dvi = (nir.subtract(red)).rename(['DVI'])   
    return image.addBands(dvi)

def add_SAVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    savi = (((nir.subtract(red)).divide(nir.add(red).add(0.5))).multiply(1.5)).rename(['SAVI'])   
    return image.addBands(savi)

def add_OSAVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    osavi = ((nir.subtract(red)).divide(nir.add(red).add(0.16))).rename(['OSAVI'])   
    return image.addBands(osavi)
    
def add_NDWI1(image):
    swir1 = image.select('B11')
    nir = image.select('B8')
    ndwi1 = nir.subtract(swir1).divide(nir.add(swir1)).rename(['NDWI1'])   
    return image.addBands(ndwi1)

def add_NDWI2(image):
    swir2 = image.select('B12')
    nir = image.select('B8')
    ndwi2 = nir.subtract(swir2).divide(nir.add(swir2)).rename(['NDWI2'])   
    return image.addBands(ndwi2)

def add_EVIre1(image):
    blue = image.select('B2')
    red = image.select('B4')
    rededge1 = image.select('B5')
    evire1 = (((rededge1.subtract(red))).divide(rededge1.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5).rename(['EVIre1'])   
    return image.addBands(evire1)

def add_EVIre2(image):
    blue = image.select('B2')
    red = image.select('B4')
    rededge2 = image.select('B6')
    evire2 = (((rededge2.subtract(red))).divide(rededge2.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5).rename(['EVIre2'])   
    return image.addBands(evire2)

def add_EVIre3(image):
    blue = image.select('B2')
    red = image.select('B4')
    rededge3 = image.select('B7')
    evire3 = (((rededge3.subtract(red))).divide(rededge3.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5).rename(['EVIre3'])   
    return image.addBands(evire3)

def add_NDRE1(image):
    rededge1 = image.select('B5')
    nir = image.select('B8')
    ndre1 = nir.subtract(rededge1).divide(nir.add(rededge1)).rename(['NDRE1'])   
    return image.addBands(ndre1)

def add_NDRE2(image):
    rededge2 = image.select('B6')
    nir = image.select('B8')
    ndre2 = nir.subtract(rededge2).divide(nir.add(rededge2)).rename(['NDRE2'])   
    return image.addBands(ndre2)

def add_NDRE3(image):
    rededge3 = image.select('B7')
    nir = image.select('B8')
    ndre3 = nir.subtract(rededge3).divide(nir.add(rededge3)).rename(['NDRE3'])   
    return image.addBands(ndre3)

    
def add_CIre(image):
    """
    Calculate the Chlorophyll Index Red Edge (CIre) and add it as a new band to the input image.

    Parameters:
    image (ee.Image): Input satellite image containing bands B5 and B7.
    
    Returns:
    ee.Image: Original image with an additional band named 'CIre' containing the Chlorophyll Index Red Edge values.
    """
    
    rededge1 = image.select('B5')
    rededge3 = image.select('B7')
    cire = ((rededge3.divide(rededge1)).subtract(1)).rename(['CIre'])   
    return image.addBands(cire)


def add_MTCI1(image):
    """
    Calculate the MERIS Terrestrial Chlorophyll Index 1 (MTCI1) and add it as a new band to the input image.
    
    MTCI1 is used to estimate chlorophyll content in vegetation by combining near-infrared and red edge bands.
    The formula is:
    MTCI1 = (NIR - RedEdge1) / (RedEdge1 - Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red), B5 (red edge 1), and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'MTCI1' containing the MTCI1 values.
    """   
    rededge1 = image.select('B5')
    nir = image.select('B8')
    red = image.select('B4')
    mtci1 = ((nir.subtract(rededge1)).divide(rededge1.subtract(red))).rename(['MTCI1'])   
    return image.addBands(mtci1)
def add_MTCI2(image):
    rededge2 = image.select('B6')
    nir = image.select('B8')
    red = image.select('B4')
    mtci2 = ((nir.subtract(rededge2)).divide(rededge2.subtract(red))).rename(['MTCI2'])   
    return image.addBands(mtci2)

def add_MTCI3(image):
    rededge3 = image.select('B7')
    nir = image.select('B8')
    red = image.select('B4')
    mtci3 = ((nir.subtract(rededge3)).divide(rededge3.subtract(red))).rename(['MTCI3'])   
    return image.addBands(mtci3)

def add_MCARI1(image):
    """
    Calculate the Modified Chlorophyll Absorption in Reflectance Index 1 (MCARI1) and add it as a new band to the input image.
    
    MCARI1 is an index designed to estimate chlorophyll content by minimizing the effects of soil background and non-photosynthetic vegetation.
    The formula is:
    MCARI1 = [(RedEdge1 - Red) - 0.2 * (RedEdge1 - Green)] * (RedEdge1 / Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B3 (green), B4 (red), and B5 (red edge 1).
    
    Returns:
    ee.Image: Original image with an additional band named 'MCARI1' containing the MCARI1 values.
    """    
    rededge1 = image.select('B5')
    red = image.select('B4')
    green = image.select('B3')
    mcari1 = (rededge1.subtract(red).subtract(rededge1.subtract(green).multiply(0.2)).multiply(rededge1.divide(red))).rename(['MCARI1'])   
    return image.addBands(mcari1)

def add_MCARI2(image):
    rededge2 = image.select('B6')
    red = image.select('B4')
    green = image.select('B3')
    mcari2 = (rededge2.subtract(red).subtract(rededge2.subtract(green).multiply(0.2)).multiply(rededge2.divide(red))).rename(['MCARI2'])   
    return image.addBands(mcari2)

def add_MCARI3(image):
    rededge3 = image.select('B7')
    red = image.select('B4')
    green = image.select('B3')
    mcari3 = (rededge3.subtract(red).subtract(rededge3.subtract(green).multiply(0.2)).multiply(rededge3.divide(red))).rename(['MCARI3'])   
    return image.addBands(mcari3)

def add_NDI45(image):
    """
    Calculate the Normalized Difference Index between bands 4 and 5 (NDI45) and add it as a new band to the input image.
    
    NDI45 is a normalized difference index calculated using red edge 1 (B5) and red (B4) bands.
    The formula is similar to NDVI but uses different bands:
    NDI45 = (RedEdge1 - Red) / (RedEdge1 + Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red) and B5 (red edge 1).
    
    Returns:
    ee.Image: Original image with an additional band named 'NDI45' containing the NDI45 values.
    """
        
    rededge1 = image.select('B5')
    red = image.select('B4')
    ndi45 = (rededge1.subtract(red).divide(rededge1.add(red))).rename(['NDI45'])   
    return image.addBands(ndi45)

def add_PSSRa(image):
    """
    Calculate the (Pigment Specific Simple Ratio for Chlorophyll a (PSSRa) and add it as a new band to the input image.

    PSSRa = RedEdge / Red
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B7 (red edge 3) and B4 (red).
    
    Returns:
    ee.Image: Original image with an additional band named 'PSSRa' containing the PSSRa values.
    """
    rededge3 = image.select('B7')
    red = image.select('B4')
    pssra = (rededge3.divide(red)).rename(['PSSRa'])   
    return image.addBands(pssra)


def add_IRECI(image):
    rededge1 = image.select('B5')
    rededge2 = image.select('B6')
    nir = image.select('B8')
    red = image.select('B4')
    ndvi56 = (nir.subtract(red).divide(rededge1.divide(rededge2))).rename(['IRECI'])   
    return image.addBands(ndvi56)

    
def add_NDVI56(image):
    """
    Calculate the Normalized Difference Vegetation Index between bands 5 and 6 (NDVI56) and add it as a new band to the input image.

    The formula is:
    NDVI56 = (RedEdge2 - RedEdge1) / (RedEdge2 + RedEdge1)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B5 (red edge 1) and B6 (red edge 2).
    
    Returns:
    ee.Image: Original image with an additional band named 'NDVI56' containing the NDVI56 values.
    """    
    rededge1 = image.select('B5')
    rededge2 = image.select('B6')
    ndvi56 = (rededge2.subtract(rededge1).divide(rededge2.add(rededge1))).rename(['NDVI56'])   
    return image.addBands(ndvi56)

def add_NDVI57(image):
    rededge1 = image.select('B5')
    rededge3 = image.select('B7')
    ndvi57 = (rededge3.subtract(rededge1).divide(rededge3.add(rededge1))).rename(['NDVI57'])   
    return image.addBands(ndvi57)

def add_NDVI68a(image):
    nira = image.select('B8A')
    rededge2 = image.select('B6')
    ndvi68a = (nira.subtract(rededge2).divide(nira.add(rededge2))).rename(['NDVI68a'])   
    return image.addBands(ndvi68a)

def add_NDVI78a(image):
    nira = image.select('B8A')
    rededge3 = image.select('B7')
    ndvi78a = (nira.subtract(rededge3).divide(nira.add(rededge3))).rename(['NDVI78a'])   
    return image.addBands(ndvi78a)

    
def add_NLI(image):
    """
    Calculate the Normalized Leaf Index (NLI) and add it as a new band to the input image.
    
    NLI is an index used to assess leaf characteristics by combining red and red edge bands.
    The formula used here is:
    NLI = (RedEdge^2 - Red) / (RedEdge^2 + Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red) and B5 (red edge 1).
    
    Returns:
    ee.Image: Original image with an additional band named 'NLI' containing the NLI values.
    """
    red = image.select('B4')
    rededge1 = image.select('B5')
    nli = ((rededge1.multiply(rededge1)).subtract(red).divide(rededge1.multiply(rededge1).add(red))).rename(['NLI'])   
    return image.addBands(nli)


def add_kNDVI(image):
    """
    Calculate the kernel Normalized Difference Vegetation Index (kNDVI) and add it as a new band to the input image.
    The steps are:
    1. Calculate NDVI = (NIR - Red) / (NIR + Red)
    2. Apply kNDVI = tanh(NDVI2)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red) and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'kNDVI' containing the kNDVI values.
    """
        
    red = image.select('B4')
    nir = image.select('B8')
    ndvi = (nir.subtract(red).divide(nir.add(red))).rename(['NDVI'])      
    kndvi = (ndvi.expression('tanh(b*b)', {'b': ndvi})).rename(['kNDVI'])   
    return image.addBands(kndvi)


def applyQA60Mask(image):
    """
    Apply a cloud and cirrus mask to the input image using the QA60 band.
    
    The QA60 band contains quality assessment information where:
    - Bit 10 indicates clouds
    - Bit 11 indicates cirrus clouds
    
    This function masks out pixels where either clouds or cirrus are detected.
    
    Parameters:
    image (ee.Image): Input satellite image containing the QA60 band.
    
    Returns:
    ee.Image: Masked image with clouds and cirrus pixels removed.
    """    
    qa = image.select('QA60')
    # Bits 10 and 11 are clouds and cirrus, respectively.
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    # Both flags should be set to zero, indicating clear conditions.
    mask = (qa.bitwiseAnd(cloud_bit_mask).eq(0).And(qa.bitwiseAnd(cirrus_bit_mask).eq(0)))
    return image.updateMask(mask)


def applyCLDPRBMask(image):
    """
    Apply a cloud probability mask to the input image using the MSK_CLDPRB band.
    
    The MSK_CLDPRB band indicates cloud probability, where:
    - 0 means clear pixels
    - Non-zero values indicate varying probabilities of cloud presence
    
    This function masks out pixels with any cloud probability (i.e., keeps only pixels with cloud probability = 0).
    
    Parameters:
    image (ee.Image): Input satellite image containing the MSK_CLDPRB band.
    
    Returns:
    ee.Image: Masked image with probable cloud pixels removed.
    """
    
    # Select the MSK_CLDPRB band (cloud probability mask)
    cloudProb = image.select('MSK_CLDPRB')
    # Mask pixels where MSK_CLDPRB is not equal to 0 (clear pixels)
    mask = cloudProb.eq(0)
    # mask =  cloudProb.lte(0.000001)  
    return image.updateMask(mask)


def applyCloudScoreMask(image):
    """
    Applies Cloud Score+ masking to a Sentinel-2 image.
    Assumes the collection has already been linked with the csPlus collection.
    """
    # Define the threshold
    CLEAR_THRESHOLD = 0.60 
    
    # Select the 'cs' (or 'cs_cdf') band linked from Cloud Score+
    # Ensure the band name matches what you linked (QA_BAND)
    qa_mask = image.select('cs').gte(CLEAR_THRESHOLD)
    
    # Update the image mask and return
    return image.updateMask(qa_mask)

# New function to prefix computed indices with 'S2_' while keeping raw bands (B*) unchanged
def add_s2_prefix_to_all_bands(image):
    """
    Renames EVERY band in the image by adding 'S2_' prefix.
    This applies to both raw Sentinel-2 bands (e.g., 'B4' → 'S2_B4')
    and computed indices (e.g., 'NDVI' → 'S2_NDVI', 'EVI1' → 'S2_EVI1').
    """
    band_names = image.bandNames()
    new_names = band_names.map(lambda name: ee.String('S2_').cat(name))
    return image.rename(new_names)
    

if __name__ == "__main__":

    # define chunk size
    CHUNK_SIZE = 1 # per cell per chunk
    
    # Output directory name on Google Drive for exported images   
    outdir = '_sentinel2_quarterly_median'
    
    # gee assets path
    parent = 'projects/fates-mrv/assets'
    
    # Ask EE for the list of *direct* children of that folder
    resp = ee.data.listAssets({'parent': parent})  # dict with key 'assets'
    assets_meta = resp.get('assets', [])           # list of dictionaries
    
    # Keep only assets whose type == 'TABLE'   (vectors/shapefiles)
    table_ids = [a['name'] for a in assets_meta if a['type'] == 'TABLE']
    print('Found', len(table_ids), 'vector assets')
    
    # Print all found asset IDs with index
    for i, aid in enumerate(table_ids):
        print(f'{i:2d}. {aid}')

    # 1. Define the Quarters
    year = 2019      
    START_DATE = f'{year}-01-01'
    END_DATE   = f'{year}-12-31'
      
    # Dynamic quarters
    quarters = [
            ('Q1', f'{year}-01-01', f'{year}-03-31'),
            ('Q2', f'{year}-04-01', f'{year}-06-30'),
            ('Q3', f'{year}-07-01', f'{year}-09-30'),
            ('Q4', f'{year}-10-01', f'{year}-12-31'),
        ]

    #  Read them in a loop
    for asset_id in tqdm(table_ids):
        #print(table_ids)

        # Filter to process only assets
        if not asset_id.endswith(f'{year}'):
            continue  # Skip if it does not end with 'fishnet'

        # Extract basename from asset ID for naming outputs
        basename = os.path.basename(asset_id)      # ← fishnet_25000m_2019_25m_190
        print('\n───────────────────────────────────────────────')
        print('Processing:', asset_id)

        # Load the vector asset as a FeatureCollection
        fishnet = ee.FeatureCollection(asset_id)
        total_cells = fishnet.size().getInfo()
        print('\nNumber of grid cells :', total_cells)
        print('Schema columns:', fishnet.first().propertyNames().getInfo())


        # spatial tiling : loop over the grid in chunks
        for offset in range(0, total_cells, CHUNK_SIZE):
            chunk_idx = offset // CHUNK_SIZE
            chunk_fc  = ee.FeatureCollection(fishnet.toList(CHUNK_SIZE, offset))
            chunk_geom = chunk_fc.geometry()
            
            #  Build the ImageCollection and reduce it with the compound reducer
            s2_collection = (ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
                                .filterBounds(chunk_geom)                        
                                .filterDate(START_DATE, END_DATE)
                                .linkCollection(ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED'), ['cs'])
                                .map(scale_s2)
                                # .map(resample_to_10m)
                                .map(applyQA60Mask)
                                .map(applyCloudScoreMask)
                                .map(add_NIRv)
                                .map(add_GNDVI)
                                .map(add_EVI1)
                                .map(add_EVI2)
                                .map(add_CIgreen)
                                .map(add_SR)
                                .map(add_DVI)
                                .map(add_SAVI)
                                .map(add_OSAVI)
                                .map(add_NDWI1)
                                .map(add_NDWI2)
                                .map(add_EVIre1)
                                .map(add_EVIre2)
                                .map(add_EVIre3)
                                .map(add_NDRE1)
                                .map(add_NDRE2)
                                .map(add_NDRE3)
                                .map(add_CIre)
                                .map(add_MTCI1)
                                .map(add_MTCI2)
                                .map(add_MTCI3)
                                .map(add_MCARI1)
                                .map(add_MCARI2)
                                .map(add_MCARI3)
                                .map(add_NDI45)
                                .map(add_PSSRa)
                                .map(add_IRECI)
                                .map(add_NDVI56)
                                .map(add_NDVI57)
                                .map(add_NDVI68a)
                                .map(add_NDVI78a)
                                .map(add_NLI)
                                .map(add_kNDVI)
                                .select(['B1','B2', 'B3', 'B4', 'B5', 'B6','B7','B8','B8A', 'B9','B10','B11','B12',
                                         'NIRv', 'GNDVI', 'EVI1', 'EVI2', 'CIgreen', 'SR', 'DVI', 'SAVI', 'OSAVI', 'NDWI1', 'NDWI2', 'EVIre1', 'EVIre2', 'EVIre3',
                                         'NDRE1','NDRE2','NDRE3', 'CIre','MTCI1', 'MTCI2', 'MTCI3', 'MCARI1', 'MCARI2', 'MCARI3', 'NDI45', 'PSSRa', 'IRECI', 'NDVI56', 'NDVI57', 'NDVI68a',
                                         'NDVI78a', 'NLI', 'kNDVI'                                      
                                        ])
                                .map(lambda image: image.toFloat())
                                # .map(lambda image: image.toDouble())
                                .map(add_s2_prefix_to_all_bands)
                            )
                                

            if s2_collection.size().getInfo() == 0:
                # No images found for this feature, skip processing
                print(f"No images found for chunk {chunk_idx} in {year}. Skipping...")
                continue  # skip to next chunk

            # Quarterly processing
            all_quarterly_fc = None
            has_quarter_data = False

            for q_name, q_start, q_end in quarters:
                q_col = s2_collection.filterDate(q_start, q_end)
                if q_col.size().getInfo() == 0:
                    print(f"  No images in {q_name} for chunk {chunk_idx}")
                    continue

                # composite for the quarter
                composite = q_col.median()

                # Zonal mean extraction
                reduced = composite.reduceRegions(
                    collection=chunk_fc,
                    reducer=ee.Reducer.mean(),
                    scale=10,
                    crs='EPSG:3857',
                    tileScale=1
                )

                reduced_q = reduced.map(lambda f: f.set({
                    'quarter': q_name,
                    'year': year,
                    'start_date': q_start,
                    'end_date': q_end
                }))

                if all_quarterly_fc is None:
                    all_quarterly_fc = reduced_q
                else:
                    all_quarterly_fc = all_quarterly_fc.merge(reduced_q)
                
                has_quarter_data = True

            if not has_quarter_data:
                print(f"No valid quarterly data for chunk {chunk_idx}. Skipping export.")
                continue        

                
            # Create a descriptive task name for export
            task_desc = f'sentinel-2_{basename}_quarterly_chunk_{chunk_idx}'
            
            # Set up and start an export task to Google Drive
            task = ee.batch.Export.table.toDrive(**{
                'collection': all_quarterly_fc,  # The combined FeatureCollection with time series
                'description': task_desc,  # Set file name
                'folder': outdir,          # Folder on Google Drive
                'fileFormat': 'CSV',       # Format (use uppercase for consistency)
                # 'selectors': ['date'] + list(s2_collection.first().bandNames().getInfo())  # Include date and bands
            })
            
            task.start()
            
            print(f'\n    → Task: {task_desc} submitted '
                  f'({offset:,} – {min(offset+CHUNK_SIZE, total_cells)-1:,})')

            # Optional: sleep to avoid submitting too many tasks too quickly
            # Each project's queue supports a maximum of 3,000 tasks
            import time
            time.sleep(3) # sleep interval to aviod creating multi outdir in google drive
            # do a pre-check of time/chunk to set this sleep value

        

## sentinel-2 semiannual median

In [ ]:
def scale_s2(image):
    # Select all spectral bands (B1 through B12)
    optical_bands = image.select('B.*').divide(10000)
    # Replace the original bands with scaled ones, but keep the QA and 'cs' bands as they are
    return image.addBands(optical_bands, overwrite=True)

def resample_to_10m(image):
    """
    Resample image to 10m using bilinear interpolation.
    """
    return (image
            .resample('bilinear')   # interpolation method
            )

def add_NDVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    ndvi = (nir.subtract(red).divide(nir.add(red))).rename(['NDVI'])   
    return image.addBands(ndvi)

def add_NIRv(image):
    red = image.select('B4')
    nir = image.select('B8')
    ndvi = (nir.subtract(red).divide(nir.add(red))).rename(['NDVI'])  
    nirv = (nir.multiply(ndvi)).rename(['NIRv'])   
    return image.addBands(nirv)

def add_GNDVI(image):
    """
    Calculate the Green Normalized Difference Vegetation Index (GNDVI) and add it as a new band to the input image.
    
    GNDVI is a vegetation index similar to NDVI but uses the green band instead of the red band,
    which can provide better sensitivity to chlorophyll concentration.
    
    The formula is:
    GNDVI = (NIR - Green) / (NIR + Green)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B3 (green) and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'GNDVI' containing the Green NDVI values.
    """
    
    green = image.select('B3')
    nir = image.select('B8')
    gndvi = (nir.subtract(green).divide(nir.add(green))).rename(['GNDVI'])   
    return image.addBands(gndvi)

def add_EVI1(image):
    """
    Calculate the Enhanced Vegetation Index (EVI1) and add it as a new band to the input image.
      
    The formula used here is:
    EVI1 = 2.5 * ( (NIR - Red) / (NIR + 6 * Red - 7.5 * Blue + 1) )
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B2 (blue), B4 (red), and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'EVI1' containing the Enhanced Vegetation Index values.
    """
    
    blue = image.select('B2')
    red = image.select('B4')
    nir = image.select('B8')
    evi1 = ((((nir.subtract(red))).divide(nir.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5)).rename(['EVI1'])   
    return image.addBands(evi1)


def add_EVI2(image):
    red = image.select('B4')
    nir = image.select('B8')
    evi2 = ((((nir.subtract(red))).divide(nir.add(red.multiply(2.4)).add(1))).multiply(2.5)).rename(['EVI2'])   
    return image.addBands(evi2)

def add_CIgreen(image):
    """
    Calculate the Chlorophyll Index Green (CIgreen) and add it as a new band to the input image.
    CIgreen = (NIR / Green) - 1
     
    Parameters:
    image (ee.Image): Input satellite image containing bands B3 (green) and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'CIgreen' containing the Chlorophyll Index Green values.
    """
    
    green = image.select('B3')
    nir = image.select('B8')
    cigreen = ((nir.divide(green)).subtract(1)).rename(['CIgreen'])   
    return image.addBands(cigreen)

def add_SR(image):
    red = image.select('B4')
    nir = image.select('B8')
    sr = (nir.divide(red)).rename(['SR'])   
    return image.addBands(sr)

def add_DVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    dvi = (nir.subtract(red)).rename(['DVI'])   
    return image.addBands(dvi)

def add_SAVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    savi = (((nir.subtract(red)).divide(nir.add(red).add(0.5))).multiply(1.5)).rename(['SAVI'])   
    return image.addBands(savi)

def add_OSAVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    osavi = ((nir.subtract(red)).divide(nir.add(red).add(0.16))).rename(['OSAVI'])   
    return image.addBands(osavi)
    
def add_NDWI1(image):
    swir1 = image.select('B11')
    nir = image.select('B8')
    ndwi1 = nir.subtract(swir1).divide(nir.add(swir1)).rename(['NDWI1'])   
    return image.addBands(ndwi1)

def add_NDWI2(image):
    swir2 = image.select('B12')
    nir = image.select('B8')
    ndwi2 = nir.subtract(swir2).divide(nir.add(swir2)).rename(['NDWI2'])   
    return image.addBands(ndwi2)

def add_EVIre1(image):
    blue = image.select('B2')
    red = image.select('B4')
    rededge1 = image.select('B5')
    evire1 = (((rededge1.subtract(red))).divide(rededge1.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5).rename(['EVIre1'])   
    return image.addBands(evire1)

def add_EVIre2(image):
    blue = image.select('B2')
    red = image.select('B4')
    rededge2 = image.select('B6')
    evire2 = (((rededge2.subtract(red))).divide(rededge2.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5).rename(['EVIre2'])   
    return image.addBands(evire2)

def add_EVIre3(image):
    blue = image.select('B2')
    red = image.select('B4')
    rededge3 = image.select('B7')
    evire3 = (((rededge3.subtract(red))).divide(rededge3.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5).rename(['EVIre3'])   
    return image.addBands(evire3)

def add_NDRE1(image):
    rededge1 = image.select('B5')
    nir = image.select('B8')
    ndre1 = nir.subtract(rededge1).divide(nir.add(rededge1)).rename(['NDRE1'])   
    return image.addBands(ndre1)

def add_NDRE2(image):
    rededge2 = image.select('B6')
    nir = image.select('B8')
    ndre2 = nir.subtract(rededge2).divide(nir.add(rededge2)).rename(['NDRE2'])   
    return image.addBands(ndre2)

def add_NDRE3(image):
    rededge3 = image.select('B7')
    nir = image.select('B8')
    ndre3 = nir.subtract(rededge3).divide(nir.add(rededge3)).rename(['NDRE3'])   
    return image.addBands(ndre3)

    
def add_CIre(image):
    """
    Calculate the Chlorophyll Index Red Edge (CIre) and add it as a new band to the input image.

    Parameters:
    image (ee.Image): Input satellite image containing bands B5 and B7.
    
    Returns:
    ee.Image: Original image with an additional band named 'CIre' containing the Chlorophyll Index Red Edge values.
    """
    
    rededge1 = image.select('B5')
    rededge3 = image.select('B7')
    cire = ((rededge3.divide(rededge1)).subtract(1)).rename(['CIre'])   
    return image.addBands(cire)


def add_MTCI1(image):
    """
    Calculate the MERIS Terrestrial Chlorophyll Index 1 (MTCI1) and add it as a new band to the input image.
    
    MTCI1 is used to estimate chlorophyll content in vegetation by combining near-infrared and red edge bands.
    The formula is:
    MTCI1 = (NIR - RedEdge1) / (RedEdge1 - Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red), B5 (red edge 1), and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'MTCI1' containing the MTCI1 values.
    """   
    rededge1 = image.select('B5')
    nir = image.select('B8')
    red = image.select('B4')
    mtci1 = ((nir.subtract(rededge1)).divide(rededge1.subtract(red))).rename(['MTCI1'])   
    return image.addBands(mtci1)
def add_MTCI2(image):
    rededge2 = image.select('B6')
    nir = image.select('B8')
    red = image.select('B4')
    mtci2 = ((nir.subtract(rededge2)).divide(rededge2.subtract(red))).rename(['MTCI2'])   
    return image.addBands(mtci2)

def add_MTCI3(image):
    rededge3 = image.select('B7')
    nir = image.select('B8')
    red = image.select('B4')
    mtci3 = ((nir.subtract(rededge3)).divide(rededge3.subtract(red))).rename(['MTCI3'])   
    return image.addBands(mtci3)

def add_MCARI1(image):
    """
    Calculate the Modified Chlorophyll Absorption in Reflectance Index 1 (MCARI1) and add it as a new band to the input image.
    
    MCARI1 is an index designed to estimate chlorophyll content by minimizing the effects of soil background and non-photosynthetic vegetation.
    The formula is:
    MCARI1 = [(RedEdge1 - Red) - 0.2 * (RedEdge1 - Green)] * (RedEdge1 / Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B3 (green), B4 (red), and B5 (red edge 1).
    
    Returns:
    ee.Image: Original image with an additional band named 'MCARI1' containing the MCARI1 values.
    """    
    rededge1 = image.select('B5')
    red = image.select('B4')
    green = image.select('B3')
    mcari1 = (rededge1.subtract(red).subtract(rededge1.subtract(green).multiply(0.2).multiply(rededge1.divide(red))).rename(['MCARI1'])   
    return image.addBands(mcari1)

def add_MCARI2(image):
    rededge2 = image.select('B6')
    red = image.select('B4')
    green = image.select('B3')
    mcari2 = (rededge2.subtract(red).subtract(rededge2.subtract(green).multiply(0.2).multiply(rededge2.divide(red))).rename(['MCARI2'])   
    return image.addBands(mcari2)

def add_MCARI3(image):
    rededge3 = image.select('B7')
    red = image.select('B4')
    green = image.select('B3')
    mcari3 = (rededge3.subtract(red).subtract(rededge3.subtract(green).multiply(0.2)).multiply(rededge3.divide(red))).rename(['MCARI3'])   
    return image.addBands(mcari3)

def add_NDI45(image):
    """
    Calculate the Normalized Difference Index between bands 4 and 5 (NDI45) and add it as a new band to the input image.
    
    NDI45 is a normalized difference index calculated using red edge 1 (B5) and red (B4) bands.
    The formula is similar to NDVI but uses different bands:
    NDI45 = (RedEdge1 - Red) / (RedEdge1 + Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red) and B5 (red edge 1).
    
    Returns:
    ee.Image: Original image with an additional band named 'NDI45' containing the NDI45 values.
    """
        
    rededge1 = image.select('B5')
    red = image.select('B4')
    ndi45 = (rededge1.subtract(red).divide(rededge1.add(red))).rename(['NDI45'])   
    return image.addBands(ndi45)

def add_PSSRa(image):
    """
    Calculate the (Pigment Specific Simple Ratio for Chlorophyll a (PSSRa) and add it as a new band to the input image.

    PSSRa = RedEdge / Red
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B7 (red edge 3) and B4 (red).
    
    Returns:
    ee.Image: Original image with an additional band named 'PSSRa' containing the PSSRa values.
    """
    rededge3 = image.select('B7')
    red = image.select('B4')
    pssra = (rededge3.divide(red)).rename(['PSSRa'])   
    return image.addBands(pssra)


def add_IRECI(image):
    rededge1 = image.select('B5')
    rededge2 = image.select('B6')
    nir = image.select('B8')
    red = image.select('B4')
    ndvi56 = (nir.subtract(red).divide(rededge1.divide(rededge2))).rename(['IRECI'])   
    return image.addBands(ndvi56)

    
def add_NDVI56(image):
    """
    Calculate the Normalized Difference Vegetation Index between bands 5 and 6 (NDVI56) and add it as a new band to the input image.

    The formula is:
    NDVI56 = (RedEdge2 - RedEdge1) / (RedEdge2 + RedEdge1)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B5 (red edge 1) and B6 (red edge 2).
    
    Returns:
    ee.Image: Original image with an additional band named 'NDVI56' containing the NDVI56 values.
    """    
    rededge1 = image.select('B5')
    rededge2 = image.select('B6')
    ndvi56 = (rededge2.subtract(rededge1).divide(rededge2.add(rededge1))).rename(['NDVI56'])   
    return image.addBands(ndvi56)

def add_NDVI57(image):
    rededge1 = image.select('B5')
    rededge3 = image.select('B7')
    ndvi57 = (rededge3.subtract(rededge1).divide(rededge3.add(rededge1))).rename(['NDVI57'])   
    return image.addBands(ndvi57)

def add_NDVI68a(image):
    nira = image.select('B8A')
    rededge2 = image.select('B6')
    ndvi68a = (nira.subtract(rededge2).divide(nira.add(rededge2))).rename(['NDVI68a'])   
    return image.addBands(ndvi68a)

def add_NDVI78a(image):
    nira = image.select('B8A')
    rededge3 = image.select('B7')
    ndvi78a = (nira.subtract(rededge3).divide(nira.add(rededge3))).rename(['NDVI78a'])   
    return image.addBands(ndvi78a)

    
def add_NLI(image):
    """
    Calculate the Normalized Leaf Index (NLI) and add it as a new band to the input image.
    
    NLI is an index used to assess leaf characteristics by combining red and red edge bands.
    The formula used here is:
    NLI = (RedEdge^2 - Red) / (RedEdge^2 + Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red) and B5 (red edge 1).
    
    Returns:
    ee.Image: Original image with an additional band named 'NLI' containing the NLI values.
    """
    red = image.select('B4')
    rededge1 = image.select('B5')
    nli = ((rededge1.multiply(rededge1)).subtract(red).divide(rededge1.multiply(rededge1).add(red))).rename(['NLI'])   
    return image.addBands(nli)


def add_kNDVI(image):
    """
    Calculate the kernel Normalized Difference Vegetation Index (kNDVI) and add it as a new band to the input image.
    The steps are:
    1. Calculate NDVI = (NIR - Red) / (NIR + Red)
    2. Apply kNDVI = tanh(NDVI2)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red) and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'kNDVI' containing the kNDVI values.
    """
        
    red = image.select('B4')
    nir = image.select('B8')
    ndvi = (nir.subtract(red).divide(nir.add(red))).rename(['NDVI'])      
    kndvi = (ndvi.expression('tanh(b*b)', {'b': ndvi})).rename(['kNDVI'])   
    return image.addBands(kndvi)


def applyQA60Mask(image):
    """
    Apply a cloud and cirrus mask to the input image using the QA60 band.
    
    The QA60 band contains quality assessment information where:
    - Bit 10 indicates clouds
    - Bit 11 indicates cirrus clouds
    
    This function masks out pixels where either clouds or cirrus are detected.
    
    Parameters:
    image (ee.Image): Input satellite image containing the QA60 band.
    
    Returns:
    ee.Image: Masked image with clouds and cirrus pixels removed.
    """    
    qa = image.select('QA60')
    # Bits 10 and 11 are clouds and cirrus, respectively.
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    # Both flags should be set to zero, indicating clear conditions.
    mask = (qa.bitwiseAnd(cloud_bit_mask).eq(0).And(qa.bitwiseAnd(cirrus_bit_mask).eq(0)))
    return image.updateMask(mask)


def applyCLDPRBMask(image):
    """
    Apply a cloud probability mask to the input image using the MSK_CLDPRB band.
    
    The MSK_CLDPRB band indicates cloud probability, where:
    - 0 means clear pixels
    - Non-zero values indicate varying probabilities of cloud presence
    
    This function masks out pixels with any cloud probability (i.e., keeps only pixels with cloud probability = 0).
    
    Parameters:
    image (ee.Image): Input satellite image containing the MSK_CLDPRB band.
    
    Returns:
    ee.Image: Masked image with probable cloud pixels removed.
    """
    
    # Select the MSK_CLDPRB band (cloud probability mask)
    cloudProb = image.select('MSK_CLDPRB')
    # Mask pixels where MSK_CLDPRB is not equal to 0 (clear pixels)
    mask = cloudProb.eq(0)
    # mask =  cloudProb.lte(0.000001)  
    return image.updateMask(mask)


def applyCloudScoreMask(image):
    """
    Applies Cloud Score+ masking to a Sentinel-2 image.
    Assumes the collection has already been linked with the csPlus collection.
    """
    # Define the threshold
    CLEAR_THRESHOLD = 0.60 
    
    # Select the 'cs' (or 'cs_cdf') band linked from Cloud Score+
    # Ensure the band name matches what you linked (QA_BAND)
    qa_mask = image.select('cs').gte(CLEAR_THRESHOLD)
    
    # Update the image mask and return
    return image.updateMask(qa_mask)

# New function to prefix computed indices with 'S2_' while keeping raw bands (B*) unchanged
def add_s2_prefix_to_all_bands(image):
    """
    Renames EVERY band in the image by adding 'S2_' prefix.
    This applies to both raw Sentinel-2 bands (e.g., 'B4' → 'S2_B4')
    and computed indices (e.g., 'NDVI' → 'S2_NDVI', 'EVI1' → 'S2_EVI1').
    """
    band_names = image.bandNames()
    new_names = band_names.map(lambda name: ee.String('S2_').cat(name))
    return image.rename(new_names)
    

if __name__ == "__main__":

    # define chunk size
    CHUNK_SIZE = 1 # per cell per chunk
    
    # Output directory name on Google Drive for exported images   
    outdir = '_sentinel2_semiannually_median'
    
    # gee assets path
    parent = 'projects/fates-mrv/assets'
    
    # Ask EE for the list of *direct* children of that folder
    resp = ee.data.listAssets({'parent': parent})  # dict with key 'assets'
    assets_meta = resp.get('assets', [])           # list of dictionaries
    
    # Keep only assets whose type == 'TABLE'   (vectors/shapefiles)
    table_ids = [a['name'] for a in assets_meta if a['type'] == 'TABLE']
    print('Found', len(table_ids), 'vector assets')
    
    # Print all found asset IDs with index
    for i, aid in enumerate(table_ids):
        print(f'{i:2d}. {aid}')

    # 1. Define the semiannual
    year = 2019      
    START_DATE = f'{year}-01-01'
    END_DATE   = f'{year}-12-31'
      
    # Dynamic semiannual
    semiannual = [
            ('Semiannual1', f'{year}-01-01', f'{year}-06-30'),
            ('Semiannual2', f'{year}-07-01', f'{year}-12-31')
        ]

    #  Read them in a loop
    for asset_id in tqdm(table_ids):
        #print(table_ids)

        # Filter to process only assets
        if not asset_id.endswith(f'{year}'):
            continue  # Skip if it does not end with 'fishnet'

        # Extract basename from asset ID for naming outputs
        basename = os.path.basename(asset_id)      # ← fishnet_25000m_2019_25m_190
        print('\n───────────────────────────────────────────────')
        print('Processing:', asset_id)

        # Load the vector asset as a FeatureCollection
        fishnet = ee.FeatureCollection(asset_id)
        total_cells = fishnet.size().getInfo()
        print('\nNumber of grid cells :', total_cells)
        print('Schema columns:', fishnet.first().propertyNames().getInfo())


        # spatial tiling : loop over the grid in chunks
        # for offset in range(0, total_cells, CHUNK_SIZE):
        for offset in range(5337, total_cells, CHUNK_SIZE):
            chunk_idx = offset // CHUNK_SIZE
            chunk_fc  = ee.FeatureCollection(fishnet.toList(CHUNK_SIZE, offset))
            chunk_geom = chunk_fc.geometry()
            
            #  Build the ImageCollection and reduce it with the compound reducer
            s2_collection = (ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
                                .filterBounds(chunk_geom)                        
                                .filterDate(START_DATE, END_DATE)
                                .linkCollection(ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED'), ['cs'])
                                .map(scale_s2)
                                # .map(resample_to_10m)
                                .map(applyQA60Mask)
                                .map(applyCloudScoreMask)
                                .map(add_NIRv)
                                .map(add_GNDVI)
                                .map(add_EVI1)
                                .map(add_EVI2)
                                .map(add_CIgreen)
                                .map(add_SR)
                                .map(add_DVI)
                                .map(add_SAVI)
                                .map(add_OSAVI)
                                .map(add_NDWI1)
                                .map(add_NDWI2)
                                .map(add_EVIre1)
                                .map(add_EVIre2)
                                .map(add_EVIre3)
                                .map(add_NDRE1)
                                .map(add_NDRE2)
                                .map(add_NDRE3)
                                .map(add_CIre)
                                .map(add_MTCI1)
                                .map(add_MTCI2)
                                .map(add_MTCI3)
                                .map(add_MCARI1)
                                .map(add_MCARI2)
                                .map(add_MCARI3)
                                .map(add_NDI45)
                                .map(add_PSSRa)
                                .map(add_IRECI)
                                .map(add_NDVI56)
                                .map(add_NDVI57)
                                .map(add_NDVI68a)
                                .map(add_NDVI78a)
                                .map(add_NLI)
                                .map(add_kNDVI)
                                .select(['B1','B2', 'B3', 'B4', 'B5', 'B6','B7','B8','B8A', 'B9','B10','B11','B12',
                                         'NIRv', 'GNDVI', 'EVI1', 'EVI2', 'CIgreen', 'SR', 'DVI', 'SAVI', 'OSAVI', 'NDWI1', 'NDWI2', 'EVIre1', 'EVIre2', 'EVIre3',
                                         'NDRE1','NDRE2','NDRE3', 'CIre','MTCI1', 'MTCI2', 'MTCI3', 'MCARI1', 'MCARI2', 'MCARI3', 'NDI45', 'PSSRa', 'IRECI', 'NDVI56', 'NDVI57', 'NDVI68a',
                                         'NDVI78a', 'NLI', 'kNDVI'                                      
                                        ])
                                .map(lambda image: image.toFloat())
                                # .map(lambda image: image.toDouble())
                                .map(add_s2_prefix_to_all_bands)
                            )
                                

            if s2_collection.size().getInfo() == 0:
                # No images found for this feature, skip processing
                print(f"No images found for chunk {chunk_idx} in {year}. Skipping...")
                continue  # skip to next chunk

            # semiannual processing
            all_semiannual_fc = None
            has_semiannual_data = False

            for name, start, end in semiannual:
                col = s2_collection.filterDate(start, end)
                if col.size().getInfo() == 0:
                    print(f"  No images in {name} for chunk {chunk_idx}")
                    continue

                # composite for the semiannual
                composite = col.median()

                # Zonal mean extraction
                reduced = composite.reduceRegions(
                    collection=chunk_fc,
                    reducer=ee.Reducer.mean(),
                    scale=10,
                    crs='EPSG:3857',
                    tileScale=1
                )

                reduced_q = reduced.map(lambda f: f.set({
                    'quarter': name,
                    'year': year,
                    'start_date': start,
                    'end_date': end
                }))

                if all_semiannual_fc is None:
                    all_semiannual_fc = reduced_q
                else:
                    all_semiannual_fc = all_semiannual_fc.merge(reduced_q)
                
                has_semiannual_data = True

            if not has_semiannual_data:
                print(f"No valid semiannual data for chunk {chunk_idx}. Skipping export.")
                continue        

                
            # Create a descriptive task name for export
            task_desc = f'sentinel-2_{basename}_semiannually_chunk_{chunk_idx}'
            
            # Set up and start an export task to Google Drive
            task = ee.batch.Export.table.toDrive(**{
                'collection': all_semiannual_fc,  # The combined FeatureCollection with time series
                'description': task_desc,  # Set file name
                'folder': outdir,          # Folder on Google Drive
                'fileFormat': 'CSV',       # Format (use uppercase for consistency)
                # 'selectors': ['date'] + list(s2_collection.first().bandNames().getInfo())  # Include date and bands
            })
            
            task.start()
            
            print(f'\n    → Task: {task_desc} submitted '
                  f'({offset:,} – {min(offset+CHUNK_SIZE, total_cells)-1:,})')

            # # Optional: sleep to avoid submitting too many tasks too quickly
            # # Each project's queue supports a maximum of 3,000 tasks
            # import time
            # time.sleep(3) # sleep interval to aviod creating multi outdir in google drive
            # # do a pre-check of time/chunk to set this sleep value

        

## sentinel-2 Yearly median

In [ ]:
def scale_s2(image):
    # Select all spectral bands (B1 through B12)
    optical_bands = image.select('B.*').divide(10000)
    # Replace the original bands with scaled ones, but keep the QA and 'cs' bands as they are
    return image.addBands(optical_bands, overwrite=True)

def resample_to_10m(image):
    """
    Resample image to 10m using bilinear interpolation.
    """
    return (image
            .resample('bilinear')   # interpolation method
            )

def add_NDVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    ndvi = (nir.subtract(red).divide(nir.add(red))).rename(['NDVI'])   
    return image.addBands(ndvi)

def add_NIRv(image):
    red = image.select('B4')
    nir = image.select('B8')
    ndvi = (nir.subtract(red).divide(nir.add(red))).rename(['NDVI'])  
    nirv = (nir.multiply(ndvi)).rename(['NIRv'])   
    return image.addBands(nirv)

def add_GNDVI(image):
    """
    Calculate the Green Normalized Difference Vegetation Index (GNDVI) and add it as a new band to the input image.
    
    GNDVI is a vegetation index similar to NDVI but uses the green band instead of the red band,
    which can provide better sensitivity to chlorophyll concentration.
    
    The formula is:
    GNDVI = (NIR - Green) / (NIR + Green)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B3 (green) and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'GNDVI' containing the Green NDVI values.
    """
    
    green = image.select('B3')
    nir = image.select('B8')
    gndvi = (nir.subtract(green).divide(nir.add(green))).rename(['GNDVI'])   
    return image.addBands(gndvi)

def add_EVI1(image):
    """
    Calculate the Enhanced Vegetation Index (EVI1) and add it as a new band to the input image.
      
    The formula used here is:
    EVI1 = 2.5 * ( (NIR - Red) / (NIR + 6 * Red - 7.5 * Blue + 1) )
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B2 (blue), B4 (red), and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'EVI1' containing the Enhanced Vegetation Index values.
    """
    
    blue = image.select('B2')
    red = image.select('B4')
    nir = image.select('B8')
    evi1 = ((((nir.subtract(red))).divide(nir.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5)).rename(['EVI1'])   
    return image.addBands(evi1)


def add_EVI2(image):
    red = image.select('B4')
    nir = image.select('B8')
    evi2 = ((((nir.subtract(red))).divide(nir.add(red.multiply(2.4)).add(1))).multiply(2.5)).rename(['EVI2'])   
    return image.addBands(evi2)

def add_CIgreen(image):
    """
    Calculate the Chlorophyll Index Green (CIgreen) and add it as a new band to the input image.
    CIgreen = (NIR / Green) - 1
     
    Parameters:
    image (ee.Image): Input satellite image containing bands B3 (green) and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'CIgreen' containing the Chlorophyll Index Green values.
    """
    
    green = image.select('B3')
    nir = image.select('B8')
    cigreen = ((nir.divide(green)).subtract(1)).rename(['CIgreen'])   
    return image.addBands(cigreen)

def add_SR(image):
    red = image.select('B4')
    nir = image.select('B8')
    sr = (nir.divide(red)).rename(['SR'])   
    return image.addBands(sr)

def add_DVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    dvi = (nir.subtract(red)).rename(['DVI'])   
    return image.addBands(dvi)

def add_SAVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    savi = (((nir.subtract(red)).divide(nir.add(red).add(0.5))).multiply(1.5)).rename(['SAVI'])   
    return image.addBands(savi)

def add_OSAVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    osavi = ((nir.subtract(red)).divide(nir.add(red).add(0.16))).rename(['OSAVI'])   
    return image.addBands(osavi)
    
def add_NDWI1(image):
    swir1 = image.select('B11')
    nir = image.select('B8')
    ndwi1 = nir.subtract(swir1).divide(nir.add(swir1)).rename(['NDWI1'])   
    return image.addBands(ndwi1)

def add_NDWI2(image):
    swir2 = image.select('B12')
    nir = image.select('B8')
    ndwi2 = nir.subtract(swir2).divide(nir.add(swir2)).rename(['NDWI2'])   
    return image.addBands(ndwi2)

def add_EVIre1(image):
    blue = image.select('B2')
    red = image.select('B4')
    rededge1 = image.select('B5')
    evire1 = (((rededge1.subtract(red))).divide(rededge1.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5).rename(['EVIre1'])   
    return image.addBands(evire1)

def add_EVIre2(image):
    blue = image.select('B2')
    red = image.select('B4')
    rededge2 = image.select('B6')
    evire2 = (((rededge2.subtract(red))).divide(rededge2.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5).rename(['EVIre2'])   
    return image.addBands(evire2)

def add_EVIre3(image):
    blue = image.select('B2')
    red = image.select('B4')
    rededge3 = image.select('B7')
    evire3 = (((rededge3.subtract(red))).divide(rededge3.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5).rename(['EVIre3'])   
    return image.addBands(evire3)

def add_NDRE1(image):
    rededge1 = image.select('B5')
    nir = image.select('B8')
    ndre1 = nir.subtract(rededge1).divide(nir.add(rededge1)).rename(['NDRE1'])   
    return image.addBands(ndre1)

def add_NDRE2(image):
    rededge2 = image.select('B6')
    nir = image.select('B8')
    ndre2 = nir.subtract(rededge2).divide(nir.add(rededge2)).rename(['NDRE2'])   
    return image.addBands(ndre2)

def add_NDRE3(image):
    rededge3 = image.select('B7')
    nir = image.select('B8')
    ndre3 = nir.subtract(rededge3).divide(nir.add(rededge3)).rename(['NDRE3'])   
    return image.addBands(ndre3)

    
def add_CIre(image):
    """
    Calculate the Chlorophyll Index Red Edge (CIre) and add it as a new band to the input image.

    Parameters:
    image (ee.Image): Input satellite image containing bands B5 and B7.
    
    Returns:
    ee.Image: Original image with an additional band named 'CIre' containing the Chlorophyll Index Red Edge values.
    """
    
    rededge1 = image.select('B5')
    rededge3 = image.select('B7')
    cire = ((rededge3.divide(rededge1)).subtract(1)).rename(['CIre'])   
    return image.addBands(cire)


def add_MTCI1(image):
    """
    Calculate the MERIS Terrestrial Chlorophyll Index 1 (MTCI1) and add it as a new band to the input image.
    
    MTCI1 is used to estimate chlorophyll content in vegetation by combining near-infrared and red edge bands.
    The formula is:
    MTCI1 = (NIR - RedEdge1) / (RedEdge1 - Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red), B5 (red edge 1), and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'MTCI1' containing the MTCI1 values.
    """   
    rededge1 = image.select('B5')
    nir = image.select('B8')
    red = image.select('B4')
    mtci1 = ((nir.subtract(rededge1)).divide(rededge1.subtract(red))).rename(['MTCI1'])   
    return image.addBands(mtci1)
def add_MTCI2(image):
    rededge2 = image.select('B6')
    nir = image.select('B8')
    red = image.select('B4')
    mtci2 = ((nir.subtract(rededge2)).divide(rededge2.subtract(red))).rename(['MTCI2'])   
    return image.addBands(mtci2)

def add_MTCI3(image):
    rededge3 = image.select('B7')
    nir = image.select('B8')
    red = image.select('B4')
    mtci3 = ((nir.subtract(rededge3)).divide(rededge3.subtract(red))).rename(['MTCI3'])   
    return image.addBands(mtci3)

def add_MCARI1(image):
    """
    Calculate the Modified Chlorophyll Absorption in Reflectance Index 1 (MCARI1) and add it as a new band to the input image.
    
    MCARI1 is an index designed to estimate chlorophyll content by minimizing the effects of soil background and non-photosynthetic vegetation.
    The formula is:
    MCARI1 = [(RedEdge1 - Red) - 0.2 * (RedEdge1 - Green)] * (RedEdge1 / Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B3 (green), B4 (red), and B5 (red edge 1).
    
    Returns:
    ee.Image: Original image with an additional band named 'MCARI1' containing the MCARI1 values.
    """    
    rededge1 = image.select('B5')
    red = image.select('B4')
    green = image.select('B3')
    mcari1 = (rededge1.subtract(red).subtract(rededge1.subtract(green).multiply(0.2).multiply(rededge1.divide(red))).rename(['MCARI1'])   
    return image.addBands(mcari1)

def add_MCARI2(image):
    rededge2 = image.select('B6')
    red = image.select('B4')
    green = image.select('B3')
    mcari2 = (rededge2.subtract(red).subtract(rededge2.subtract(green).multiply(0.2).multiply(rededge2.divide(red))).rename(['MCARI2'])   
    return image.addBands(mcari2)

def add_MCARI3(image):
    rededge3 = image.select('B7')
    red = image.select('B4')
    green = image.select('B3')
    mcari3 = (rededge3.subtract(red).subtract(rededge3.subtract(green).multiply(0.2).multiply(rededge3.divide(red))).rename(['MCARI3'])   
    return image.addBands(mcari3)

def add_NDI45(image):
    """
    Calculate the Normalized Difference Index between bands 4 and 5 (NDI45) and add it as a new band to the input image.
    
    NDI45 is a normalized difference index calculated using red edge 1 (B5) and red (B4) bands.
    The formula is similar to NDVI but uses different bands:
    NDI45 = (RedEdge1 - Red) / (RedEdge1 + Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red) and B5 (red edge 1).
    
    Returns:
    ee.Image: Original image with an additional band named 'NDI45' containing the NDI45 values.
    """
        
    rededge1 = image.select('B5')
    red = image.select('B4')
    ndi45 = (rededge1.subtract(red).divide(rededge1.add(red))).rename(['NDI45'])   
    return image.addBands(ndi45)

def add_PSSRa(image):
    """
    Calculate the (Pigment Specific Simple Ratio for Chlorophyll a (PSSRa) and add it as a new band to the input image.

    PSSRa = RedEdge / Red
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B7 (red edge 3) and B4 (red).
    
    Returns:
    ee.Image: Original image with an additional band named 'PSSRa' containing the PSSRa values.
    """
    rededge3 = image.select('B7')
    red = image.select('B4')
    pssra = (rededge3.divide(red)).rename(['PSSRa'])   
    return image.addBands(pssra)


def add_IRECI(image):
    rededge1 = image.select('B5')
    rededge2 = image.select('B6')
    nir = image.select('B8')
    red = image.select('B4')
    ndvi56 = (nir.subtract(red).divide(rededge1.divide(rededge2))).rename(['IRECI'])   
    return image.addBands(ndvi56)

    
def add_NDVI56(image):
    """
    Calculate the Normalized Difference Vegetation Index between bands 5 and 6 (NDVI56) and add it as a new band to the input image.

    The formula is:
    NDVI56 = (RedEdge2 - RedEdge1) / (RedEdge2 + RedEdge1)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B5 (red edge 1) and B6 (red edge 2).
    
    Returns:
    ee.Image: Original image with an additional band named 'NDVI56' containing the NDVI56 values.
    """    
    rededge1 = image.select('B5')
    rededge2 = image.select('B6')
    ndvi56 = (rededge2.subtract(rededge1).divide(rededge2.add(rededge1))).rename(['NDVI56'])   
    return image.addBands(ndvi56)

def add_NDVI57(image):
    rededge1 = image.select('B5')
    rededge3 = image.select('B7')
    ndvi57 = (rededge3.subtract(rededge1).divide(rededge3.add(rededge1))).rename(['NDVI57'])   
    return image.addBands(ndvi57)

def add_NDVI68a(image):
    nira = image.select('B8A')
    rededge2 = image.select('B6')
    ndvi68a = (nira.subtract(rededge2).divide(nira.add(rededge2))).rename(['NDVI68a'])   
    return image.addBands(ndvi68a)

def add_NDVI78a(image):
    nira = image.select('B8A')
    rededge3 = image.select('B7')
    ndvi78a = (nira.subtract(rededge3).divide(nira.add(rededge3))).rename(['NDVI78a'])   
    return image.addBands(ndvi78a)

    
def add_NLI(image):
    """
    Calculate the Normalized Leaf Index (NLI) and add it as a new band to the input image.
    
    NLI is an index used to assess leaf characteristics by combining red and red edge bands.
    The formula used here is:
    NLI = (RedEdge^2 - Red) / (RedEdge^2 + Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red) and B5 (red edge 1).
    
    Returns:
    ee.Image: Original image with an additional band named 'NLI' containing the NLI values.
    """
    red = image.select('B4')
    rededge1 = image.select('B5')
    nli = ((rededge1.multiply(rededge1)).subtract(red).divide(rededge1.multiply(rededge1).add(red))).rename(['NLI'])   
    return image.addBands(nli)


def add_kNDVI(image):
    """
    Calculate the kernel Normalized Difference Vegetation Index (kNDVI) and add it as a new band to the input image.
    The steps are:
    1. Calculate NDVI = (NIR - Red) / (NIR + Red)
    2. Apply kNDVI = tanh(NDVI2)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red) and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'kNDVI' containing the kNDVI values.
    """
        
    red = image.select('B4')
    nir = image.select('B8')
    ndvi = (nir.subtract(red).divide(nir.add(red))).rename(['NDVI'])      
    kndvi = (ndvi.expression('tanh(b*b)', {'b': ndvi})).rename(['kNDVI'])   
    return image.addBands(kndvi)


def applyQA60Mask(image):
    """
    Apply a cloud and cirrus mask to the input image using the QA60 band.
    
    The QA60 band contains quality assessment information where:
    - Bit 10 indicates clouds
    - Bit 11 indicates cirrus clouds
    
    This function masks out pixels where either clouds or cirrus are detected.
    
    Parameters:
    image (ee.Image): Input satellite image containing the QA60 band.
    
    Returns:
    ee.Image: Masked image with clouds and cirrus pixels removed.
    """    
    qa = image.select('QA60')
    # Bits 10 and 11 are clouds and cirrus, respectively.
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    # Both flags should be set to zero, indicating clear conditions.
    mask = (qa.bitwiseAnd(cloud_bit_mask).eq(0).And(qa.bitwiseAnd(cirrus_bit_mask).eq(0)))
    return image.updateMask(mask)


def applyCLDPRBMask(image):
    """
    Apply a cloud probability mask to the input image using the MSK_CLDPRB band.
    
    The MSK_CLDPRB band indicates cloud probability, where:
    - 0 means clear pixels
    - Non-zero values indicate varying probabilities of cloud presence
    
    This function masks out pixels with any cloud probability (i.e., keeps only pixels with cloud probability = 0).
    
    Parameters:
    image (ee.Image): Input satellite image containing the MSK_CLDPRB band.
    
    Returns:
    ee.Image: Masked image with probable cloud pixels removed.
    """
    
    # Select the MSK_CLDPRB band (cloud probability mask)
    cloudProb = image.select('MSK_CLDPRB')
    # Mask pixels where MSK_CLDPRB is not equal to 0 (clear pixels)
    mask = cloudProb.eq(0)
    # mask =  cloudProb.lte(0.000001)  
    return image.updateMask(mask)


def applyCloudScoreMask(image):
    """
    Applies Cloud Score+ masking to a Sentinel-2 image.
    Assumes the collection has already been linked with the csPlus collection.
    """
    # Define the threshold
    CLEAR_THRESHOLD = 0.60 
    
    # Select the 'cs' (or 'cs_cdf') band linked from Cloud Score+
    # Ensure the band name matches what you linked (QA_BAND)
    qa_mask = image.select('cs').gte(CLEAR_THRESHOLD)
    
    # Update the image mask and return
    return image.updateMask(qa_mask)

# New function to prefix computed indices with 'S2_' while keeping raw bands (B*) unchanged
def add_s2_prefix_to_all_bands(image):
    """
    Renames EVERY band in the image by adding 'S2_' prefix.
    This applies to both raw Sentinel-2 bands (e.g., 'B4' → 'S2_B4')
    and computed indices (e.g., 'NDVI' → 'S2_NDVI', 'EVI1' → 'S2_EVI1').
    """
    band_names = image.bandNames()
    new_names = band_names.map(lambda name: ee.String('S2_').cat(name))
    return image.rename(new_names)
    
    

if __name__ == "__main__":

    # define chunk size
    CHUNK_SIZE = 1 # per cell per chunk
    
    # Output directory name on Google Drive for exported images   
    outdir = '_sentinel2_yearly_median'
    
    # gee assets path
    parent = 'projects/fates-mrv/assets'
    
    # Ask EE for the list of *direct* children of that folder
    resp = ee.data.listAssets({'parent': parent})  # dict with key 'assets'
    assets_meta = resp.get('assets', [])           # list of dictionaries
    
    # Keep only assets whose type == 'TABLE'   (vectors/shapefiles)
    table_ids = [a['name'] for a in assets_meta if a['type'] == 'TABLE']
    print('Found', len(table_ids), 'vector assets')
    
    # Print all found asset IDs with index
    for i, aid in enumerate(table_ids):
        print(f'{i:2d}. {aid}')
     
    # 1. Define the Quarters
    year = 2017     
    START_DATE = f'{year}-01-01'
    END_DATE   = f'{year}-12-31'
    
    #  Read them in a loop
    for asset_id in tqdm(table_ids):
        #print(table_ids)

        # Filter to process only assets
        if not asset_id.endswith(f'{year}'):
            continue  # Skip if it does not end with 'fishnet'

        # Extract basename from asset ID for naming outputs
        basename = os.path.basename(asset_id)      # ← fishnet_25000m_2019_25m_190
        print('\n───────────────────────────────────────────────')
        print('Processing:', asset_id)

        # Load the vector asset as a FeatureCollection
        fishnet = ee.FeatureCollection(asset_id)
        total_cells = fishnet.size().getInfo()
        print('\nNumber of grid cells :', total_cells)
        print('Schema columns:', fishnet.first().propertyNames().getInfo())


        # spatial tiling : loop over the grid in chunks
        for offset in range(0, total_cells, CHUNK_SIZE):
        # for offset in range(2322, total_cells):
            chunk_idx = offset // CHUNK_SIZE
            chunk_fc  = ee.FeatureCollection(fishnet.toList(CHUNK_SIZE, offset))
            chunk_geom = chunk_fc.geometry()
            
            #  Build the ImageCollection and reduce it with the compound reducer
            s2_collection = (ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
                                .filterBounds(chunk_geom)                        
                                .filterDate(START_DATE, END_DATE)
                                .linkCollection(ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED'), ['cs'])
                                .map(scale_s2)
                                # .map(resample_to_10m)
                                .map(applyQA60Mask)
                                .map(applyCloudScoreMask)
                                .map(add_NIRv)
                                .map(add_GNDVI)
                                .map(add_EVI1)
                                .map(add_EVI2)
                                .map(add_CIgreen)
                                .map(add_SR)
                                .map(add_DVI)
                                .map(add_SAVI)
                                .map(add_OSAVI)
                                .map(add_NDWI1)
                                .map(add_NDWI2)
                                .map(add_EVIre1)
                                .map(add_EVIre2)
                                .map(add_EVIre3)
                                .map(add_NDRE1)
                                .map(add_NDRE2)
                                .map(add_NDRE3)
                                .map(add_CIre)
                                .map(add_MTCI1)
                                .map(add_MTCI2)
                                .map(add_MTCI3)
                                .map(add_MCARI1)
                                .map(add_MCARI2)
                                .map(add_MCARI3)
                                .map(add_NDI45)
                                .map(add_PSSRa)
                                .map(add_IRECI)
                                .map(add_NDVI56)
                                .map(add_NDVI57)
                                .map(add_NDVI68a)
                                .map(add_NDVI78a)
                                .map(add_NLI)
                                .map(add_kNDVI)
                                .select(['B1','B2', 'B3', 'B4', 'B5', 'B6','B7','B8','B8A', 'B9','B10','B11','B12',
                                         'NIRv', 'GNDVI', 'EVI1', 'EVI2', 'CIgreen', 'SR', 'DVI', 'SAVI', 'OSAVI', 'NDWI1', 'NDWI2', 'EVIre1', 'EVIre2', 'EVIre3',
                                         'NDRE1','NDRE2','NDRE3', 'CIre','MTCI1', 'MTCI2', 'MTCI3', 'MCARI1', 'MCARI2', 'MCARI3', 'NDI45', 'PSSRa', 'IRECI', 'NDVI56', 'NDVI57', 'NDVI68a',
                                         'NDVI78a', 'NLI', 'kNDVI'                                      
                                        ])
                                .map(lambda image: image.toFloat())
                                # .map(lambda image: image.toDouble())
                                .map(add_s2_prefix_to_all_bands)
                            )
                                

            # Check bands before reduction
            collection_size = s2_collection.size().getInfo()
            
            if collection_size == 0:
                # No images found for this feature, skip processing
                print(f"No images found for chunk {chunk_idx}. Skipping...")
                continue  # skip to next chunk
                
            else:

                # 1. Extract the year from the first image in the collection
                # use .first() to get one image, then format its date to 'YYYY'
                first_image = s2_collection.first()
                year_string = first_image.date().format('YYYY')
                
                # 1. Reduce the collection to a yearly mean image
                # This collapses the temporal dimension into one image
                yearly_median_image = s2_collection.median()

                # # 2. Perform reduceRegions on the yearly mean image
                # This calculates the mean pixel value within each polygon of the chunk
                s2_yearly_stats = yearly_median_image.reduceRegions(
                    collection=chunk_fc,
                    reducer=ee.Reducer.mean(),
                    scale=10,
                    crs='EPSG:3857',
                    tileScale=1)
                
                # 3. Add a property to indicate the year/period
                s2_yearly_stats = s2_yearly_stats.map(lambda f: f.set('system:1styear', year_string))
                
                # Create a descriptive task name for export
                task_desc = f'sentinel-2_{basename}_Yearly_chunk_{chunk_idx}'
                
                # Set up and start an export task to Google Drive
                task = ee.batch.Export.table.toDrive(**{
                    'collection': s2_yearly_stats,  # The combined FeatureCollection with time series
                    'description': task_desc,  # Set file name
                    'folder': outdir,          # Folder on Google Drive
                    'fileFormat': 'CSV',       # Format (use uppercase for consistency)
                    # 'selectors': ['date'] + list(s2_collection.first().bandNames().getInfo())  # Include date and bands
                })
                
                task.start()
                
                print(f'\n    → Task: {task_desc} submitted '
                      f'({offset:,} – {min(offset+CHUNK_SIZE, total_cells)-1:,})')
    
                # Optional: sleep to avoid submitting too many tasks too quickly
                # Each project's queue supports a maximum of 3,000 tasks
                import time
                time.sleep(3) # sleep interval to aviod creating multi outdir in google drive
                # do a pre-check of time/chunk to set this sleep value

        

## sentinel-2 Yearly summary stats

In [ ]:
def scale_s2(image):
    # Select all spectral bands (B1 through B12)
    optical_bands = image.select('B.*').divide(10000)
    # Replace the original bands with scaled ones, but keep the QA and 'cs' bands as they are
    return image.addBands(optical_bands, overwrite=True)

def resample_to_10m(image):
    """
    Resample image to 10m using bilinear interpolation.
    """
    return (image
            .resample('bilinear')   # interpolation method
            )

def add_NDVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    ndvi = (nir.subtract(red).divide(nir.add(red))).rename(['NDVI'])   
    return image.addBands(ndvi)

def add_NIRv(image):
    red = image.select('B4')
    nir = image.select('B8')
    ndvi = (nir.subtract(red).divide(nir.add(red))).rename(['NDVI'])  
    nirv = (nir.multiply(ndvi)).rename(['NIRv'])   
    return image.addBands(nirv)

def add_GNDVI(image):
    """
    Calculate the Green Normalized Difference Vegetation Index (GNDVI) and add it as a new band to the input image.
    
    GNDVI is a vegetation index similar to NDVI but uses the green band instead of the red band,
    which can provide better sensitivity to chlorophyll concentration.
    
    The formula is:
    GNDVI = (NIR - Green) / (NIR + Green)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B3 (green) and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'GNDVI' containing the Green NDVI values.
    """
    
    green = image.select('B3')
    nir = image.select('B8')
    gndvi = (nir.subtract(green).divide(nir.add(green))).rename(['GNDVI'])   
    return image.addBands(gndvi)

def add_EVI1(image):
    """
    Calculate the Enhanced Vegetation Index (EVI1) and add it as a new band to the input image.
      
    The formula used here is:
    EVI1 = 2.5 * ( (NIR - Red) / (NIR + 6 * Red - 7.5 * Blue + 1) )
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B2 (blue), B4 (red), and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'EVI1' containing the Enhanced Vegetation Index values.
    """
    
    blue = image.select('B2')
    red = image.select('B4')
    nir = image.select('B8')
    evi1 = ((((nir.subtract(red))).divide(nir.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5)).rename(['EVI1'])   
    return image.addBands(evi1)


def add_EVI2(image):
    red = image.select('B4')
    nir = image.select('B8')
    evi2 = ((((nir.subtract(red))).divide(nir.add(red.multiply(2.4)).add(1))).multiply(2.5)).rename(['EVI2'])   
    return image.addBands(evi2)

def add_CIgreen(image):
    """
    Calculate the Chlorophyll Index Green (CIgreen) and add it as a new band to the input image.
    CIgreen = (NIR / Green) - 1
     
    Parameters:
    image (ee.Image): Input satellite image containing bands B3 (green) and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'CIgreen' containing the Chlorophyll Index Green values.
    """
    
    green = image.select('B3')
    nir = image.select('B8')
    cigreen = ((nir.divide(green)).subtract(1)).rename(['CIgreen'])   
    return image.addBands(cigreen)

def add_SR(image):
    red = image.select('B4')
    nir = image.select('B8')
    sr = (nir.divide(red)).rename(['SR'])   
    return image.addBands(sr)

def add_DVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    dvi = (nir.subtract(red)).rename(['DVI'])   
    return image.addBands(dvi)

def add_SAVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    savi = (((nir.subtract(red)).divide(nir.add(red).add(0.5))).multiply(1.5)).rename(['SAVI'])   
    return image.addBands(savi)

def add_OSAVI(image):
    red = image.select('B4')
    nir = image.select('B8')
    osavi = ((nir.subtract(red)).divide(nir.add(red).add(0.16))).rename(['OSAVI'])   
    return image.addBands(osavi)
    
def add_NDWI1(image):
    swir1 = image.select('B11')
    nir = image.select('B8')
    ndwi1 = nir.subtract(swir1).divide(nir.add(swir1)).rename(['NDWI1'])   
    return image.addBands(ndwi1)

def add_NDWI2(image):
    swir2 = image.select('B12')
    nir = image.select('B8')
    ndwi2 = nir.subtract(swir2).divide(nir.add(swir2)).rename(['NDWI2'])   
    return image.addBands(ndwi2)

def add_EVIre1(image):
    blue = image.select('B2')
    red = image.select('B4')
    rededge1 = image.select('B5')
    evire1 = (((rededge1.subtract(red))).divide(rededge1.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5).rename(['EVIre1'])   
    return image.addBands(evire1)

def add_EVIre2(image):
    blue = image.select('B2')
    red = image.select('B4')
    rededge2 = image.select('B6')
    evire2 = (((rededge2.subtract(red))).divide(rededge2.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5).rename(['EVIre2'])   
    return image.addBands(evire2)

def add_EVIre3(image):
    blue = image.select('B2')
    red = image.select('B4')
    rededge3 = image.select('B7')
    evire3 = (((rededge3.subtract(red))).divide(rededge3.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))).multiply(2.5).rename(['EVIre3'])   
    return image.addBands(evire3)

def add_NDRE1(image):
    rededge1 = image.select('B5')
    nir = image.select('B8')
    ndre1 = nir.subtract(rededge1).divide(nir.add(rededge1)).rename(['NDRE1'])   
    return image.addBands(ndre1)

def add_NDRE2(image):
    rededge2 = image.select('B6')
    nir = image.select('B8')
    ndre2 = nir.subtract(rededge2).divide(nir.add(rededge2)).rename(['NDRE2'])   
    return image.addBands(ndre2)

def add_NDRE3(image):
    rededge3 = image.select('B7')
    nir = image.select('B8')
    ndre3 = nir.subtract(rededge3).divide(nir.add(rededge3)).rename(['NDRE3'])   
    return image.addBands(ndre3)

    
def add_CIre(image):
    """
    Calculate the Chlorophyll Index Red Edge (CIre) and add it as a new band to the input image.

    Parameters:
    image (ee.Image): Input satellite image containing bands B5 and B7.
    
    Returns:
    ee.Image: Original image with an additional band named 'CIre' containing the Chlorophyll Index Red Edge values.
    """
    
    rededge1 = image.select('B5')
    rededge3 = image.select('B7')
    cire = ((rededge3.divide(rededge1)).subtract(1)).rename(['CIre'])   
    return image.addBands(cire)


def add_MTCI1(image):
    """
    Calculate the MERIS Terrestrial Chlorophyll Index 1 (MTCI1) and add it as a new band to the input image.
    
    MTCI1 is used to estimate chlorophyll content in vegetation by combining near-infrared and red edge bands.
    The formula is:
    MTCI1 = (NIR - RedEdge1) / (RedEdge1 - Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red), B5 (red edge 1), and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'MTCI1' containing the MTCI1 values.
    """   
    rededge1 = image.select('B5')
    nir = image.select('B8')
    red = image.select('B4')
    mtci1 = ((nir.subtract(rededge1)).divide(rededge1.subtract(red))).rename(['MTCI1'])   
    return image.addBands(mtci1)
def add_MTCI2(image):
    rededge2 = image.select('B6')
    nir = image.select('B8')
    red = image.select('B4')
    mtci2 = ((nir.subtract(rededge2)).divide(rededge2.subtract(red))).rename(['MTCI2'])   
    return image.addBands(mtci2)

def add_MTCI3(image):
    rededge3 = image.select('B7')
    nir = image.select('B8')
    red = image.select('B4')
    mtci3 = ((nir.subtract(rededge3)).divide(rededge3.subtract(red))).rename(['MTCI3'])   
    return image.addBands(mtci3)

def add_MCARI1(image):
    """
    Calculate the Modified Chlorophyll Absorption in Reflectance Index 1 (MCARI1) and add it as a new band to the input image.
    
    MCARI1 is an index designed to estimate chlorophyll content by minimizing the effects of soil background and non-photosynthetic vegetation.
    The formula is:
    MCARI1 = [(RedEdge1 - Red) - 0.2 * (RedEdge1 - Green)] * (RedEdge1 / Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B3 (green), B4 (red), and B5 (red edge 1).
    
    Returns:
    ee.Image: Original image with an additional band named 'MCARI1' containing the MCARI1 values.
    """    
    rededge1 = image.select('B5')
    red = image.select('B4')
    green = image.select('B3')
    mcari1 = ((rededge1.subtract(red).subtract(rededge1.subtract(green)).multiply(0.2)).multiply(rededge1.divide(red))).rename(['MCARI1'])   
    return image.addBands(mcari1)

def add_MCARI2(image):
    rededge2 = image.select('B6')
    red = image.select('B4')
    green = image.select('B3')
    mcari2 = ((rededge2.subtract(red).subtract(rededge2.subtract(green)).multiply(0.2)).multiply(rededge2.divide(red))).rename(['MCARI2'])   
    return image.addBands(mcari2)

def add_MCARI3(image):
    rededge3 = image.select('B7')
    red = image.select('B4')
    green = image.select('B3')
    mcari3 = ((rededge3.subtract(red).subtract(rededge3.subtract(green)).multiply(0.2)).multiply(rededge3.divide(red))).rename(['MCARI3'])   
    return image.addBands(mcari3)

def add_NDI45(image):
    """
    Calculate the Normalized Difference Index between bands 4 and 5 (NDI45) and add it as a new band to the input image.
    
    NDI45 is a normalized difference index calculated using red edge 1 (B5) and red (B4) bands.
    The formula is similar to NDVI but uses different bands:
    NDI45 = (RedEdge1 - Red) / (RedEdge1 + Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red) and B5 (red edge 1).
    
    Returns:
    ee.Image: Original image with an additional band named 'NDI45' containing the NDI45 values.
    """
        
    rededge1 = image.select('B5')
    red = image.select('B4')
    ndi45 = (rededge1.subtract(red).divide(rededge1.add(red))).rename(['NDI45'])   
    return image.addBands(ndi45)

def add_PSSRa(image):
    """
    Calculate the (Pigment Specific Simple Ratio for Chlorophyll a (PSSRa) and add it as a new band to the input image.

    PSSRa = RedEdge / Red
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B7 (red edge 3) and B4 (red).
    
    Returns:
    ee.Image: Original image with an additional band named 'PSSRa' containing the PSSRa values.
    """
    rededge3 = image.select('B7')
    red = image.select('B4')
    pssra = (rededge3.divide(red)).rename(['PSSRa'])   
    return image.addBands(pssra)


def add_IRECI(image):
    rededge1 = image.select('B5')
    rededge2 = image.select('B6')
    nir = image.select('B8')
    red = image.select('B4')
    ndvi56 = (nir.subtract(red).divide(rededge1.divide(rededge2))).rename(['IRECI'])   
    return image.addBands(ndvi56)

    
def add_NDVI56(image):
    """
    Calculate the Normalized Difference Vegetation Index between bands 5 and 6 (NDVI56) and add it as a new band to the input image.

    The formula is:
    NDVI56 = (RedEdge2 - RedEdge1) / (RedEdge2 + RedEdge1)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B5 (red edge 1) and B6 (red edge 2).
    
    Returns:
    ee.Image: Original image with an additional band named 'NDVI56' containing the NDVI56 values.
    """    
    rededge1 = image.select('B5')
    rededge2 = image.select('B6')
    ndvi56 = (rededge2.subtract(rededge1).divide(rededge2.add(rededge1))).rename(['NDVI56'])   
    return image.addBands(ndvi56)

def add_NDVI57(image):
    rededge1 = image.select('B5')
    rededge3 = image.select('B7')
    ndvi57 = (rededge3.subtract(rededge1).divide(rededge3.add(rededge1))).rename(['NDVI57'])   
    return image.addBands(ndvi57)

def add_NDVI68a(image):
    nira = image.select('B8A')
    rededge2 = image.select('B6')
    ndvi68a = (nira.subtract(rededge2).divide(nira.add(rededge2))).rename(['NDVI68a'])   
    return image.addBands(ndvi68a)

def add_NDVI78a(image):
    nira = image.select('B8A')
    rededge3 = image.select('B7')
    ndvi78a = (nira.subtract(rededge3).divide(nira.add(rededge3))).rename(['NDVI78a'])   
    return image.addBands(ndvi78a)

    
def add_NLI(image):
    """
    Calculate the Normalized Leaf Index (NLI) and add it as a new band to the input image.
    
    NLI is an index used to assess leaf characteristics by combining red and red edge bands.
    The formula used here is:
    NLI = (RedEdge^2 - Red) / (RedEdge^2 + Red)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red) and B5 (red edge 1).
    
    Returns:
    ee.Image: Original image with an additional band named 'NLI' containing the NLI values.
    """
    red = image.select('B4')
    rededge1 = image.select('B5')
    nli = ((rededge1.multiply(rededge1)).subtract(red).divide(rededge1.multiply(rededge1).add(red))).rename(['NLI'])   
    return image.addBands(nli)


def add_kNDVI(image):
    """
    Calculate the kernel Normalized Difference Vegetation Index (kNDVI) and add it as a new band to the input image.
    The steps are:
    1. Calculate NDVI = (NIR - Red) / (NIR + Red)
    2. Apply kNDVI = tanh(NDVI2)
    
    Parameters:
    image (ee.Image): Input satellite image containing bands B4 (red) and B8 (near-infrared).
    
    Returns:
    ee.Image: Original image with an additional band named 'kNDVI' containing the kNDVI values.
    """
        
    red = image.select('B4')
    nir = image.select('B8')
    ndvi = (nir.subtract(red).divide(nir.add(red))).rename(['NDVI'])      
    kndvi = (ndvi.expression('tanh(b*b)', {'b': ndvi})).rename(['kNDVI'])   
    return image.addBands(kndvi)


def applyQA60Mask(image):
    """
    Apply a cloud and cirrus mask to the input image using the QA60 band.
    
    The QA60 band contains quality assessment information where:
    - Bit 10 indicates clouds
    - Bit 11 indicates cirrus clouds
    
    This function masks out pixels where either clouds or cirrus are detected.
    
    Parameters:
    image (ee.Image): Input satellite image containing the QA60 band.
    
    Returns:
    ee.Image: Masked image with clouds and cirrus pixels removed.
    """    
    qa = image.select('QA60')
    # Bits 10 and 11 are clouds and cirrus, respectively.
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    # Both flags should be set to zero, indicating clear conditions.
    mask = (qa.bitwiseAnd(cloud_bit_mask).eq(0).And(qa.bitwiseAnd(cirrus_bit_mask).eq(0)))
    return image.updateMask(mask)


def applyCLDPRBMask(image):
    """
    Apply a cloud probability mask to the input image using the MSK_CLDPRB band.
    
    The MSK_CLDPRB band indicates cloud probability, where:
    - 0 means clear pixels
    - Non-zero values indicate varying probabilities of cloud presence
    
    This function masks out pixels with any cloud probability (i.e., keeps only pixels with cloud probability = 0).
    
    Parameters:
    image (ee.Image): Input satellite image containing the MSK_CLDPRB band.
    
    Returns:
    ee.Image: Masked image with probable cloud pixels removed.
    """
    
    # Select the MSK_CLDPRB band (cloud probability mask)
    cloudProb = image.select('MSK_CLDPRB')
    # Mask pixels where MSK_CLDPRB is not equal to 0 (clear pixels)
    mask = cloudProb.eq(0)
    # mask =  cloudProb.lte(0.000001)  
    return image.updateMask(mask)


def applyCloudScoreMask(image):
    """
    Applies Cloud Score+ masking to a Sentinel-2 image.
    Assumes the collection has already been linked with the csPlus collection.
    """
    # Define the threshold
    CLEAR_THRESHOLD = 0.60 
    
    # Select the 'cs' (or 'cs_cdf') band linked from Cloud Score+
    # Ensure the band name matches what you linked (QA_BAND)
    qa_mask = image.select('cs').gte(CLEAR_THRESHOLD)
    
    # Update the image mask and return
    return image.updateMask(qa_mask)

# New function to prefix computed indices with 'S2_' while keeping raw bands (B*) unchanged
def add_s2_prefix_to_all_bands(image):
    """
    Renames EVERY band in the image by adding 'S2_' prefix.
    This applies to both raw Sentinel-2 bands (e.g., 'B4' → 'S2_B4')
    and computed indices (e.g., 'NDVI' → 'S2_NDVI', 'EVI1' → 'S2_EVI1').
    """
    band_names = image.bandNames()
    new_names = band_names.map(lambda name: ee.String('S2_').cat(name))
    return image.rename(new_names)
    
    

if __name__ == "__main__":

    # define chunk size
    CHUNK_SIZE = 1 # per cell per chunk
    
    # Output directory name on Google Drive for exported images   
    outdir = '_sentinel2_yearly_summary_stats'

    # gee assets path
    parent = 'projects/fates-mrv/assets'
    
    # Ask EE for the list of *direct* children of that folder
    resp = ee.data.listAssets({'parent': parent})  # dict with key 'assets'
    assets_meta = resp.get('assets', [])           # list of dictionaries
    
    # Keep only assets whose type == 'TABLE'   (vectors/shapefiles)
    table_ids = [a['name'] for a in assets_meta if a['type'] == 'TABLE']
    print('Found', len(table_ids), 'vector assets')
    
    # Print all found asset IDs with index
    for i, aid in enumerate(table_ids):
        print(f'{i:2d}. {aid}')
     
    # 1. Define the Quarters
    year = 2019     
    START_DATE = f'{year}-01-01'
    END_DATE   = f'{year}-12-31'
    
    #  Read them in a loop
    for asset_id in tqdm(table_ids):
        #print(table_ids)

        # Filter to process only assets
        if not asset_id.endswith(f'{year}'):
            continue  # Skip if it does not end with 'fishnet'

        # Extract basename from asset ID for naming outputs
        basename = os.path.basename(asset_id)      # ← fishnet_25000m_2019_25m_190
        print('\n───────────────────────────────────────────────')
        print('Processing:', asset_id)

        # Load the vector asset as a FeatureCollection
        fishnet = ee.FeatureCollection(asset_id)
        total_cells = fishnet.size().getInfo()
        print('\nNumber of grid cells :', total_cells)
        print('Schema columns:', fishnet.first().propertyNames().getInfo())


        # spatial tiling : loop over the grid in chunks
        for offset in range(0, total_cells, CHUNK_SIZE):
        # for offset in range(1660, total_cells):
            chunk_idx = offset // CHUNK_SIZE
            chunk_fc  = ee.FeatureCollection(fishnet.toList(CHUNK_SIZE, offset))
            chunk_geom = chunk_fc.geometry()
            
            #  Build the ImageCollection and reduce it with the compound reducer
            s2_collection = (ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
                                .filterBounds(chunk_geom)                        
                                .filterDate(START_DATE, END_DATE)
                                .linkCollection(ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED'), ['cs'])
                                .map(scale_s2)
                                # .map(resample_to_10m)
                                .map(applyQA60Mask)
                                .map(applyCloudScoreMask)
                                .map(add_NIRv)
                                .map(add_GNDVI)
                                .map(add_EVI1)
                                .map(add_EVI2)
                                .map(add_CIgreen)
                                .map(add_SR)
                                .map(add_DVI)
                                .map(add_SAVI)
                                .map(add_OSAVI)
                                .map(add_NDWI1)
                                .map(add_NDWI2)
                                .map(add_EVIre1)
                                .map(add_EVIre2)
                                .map(add_EVIre3)
                                .map(add_NDRE1)
                                .map(add_NDRE2)
                                .map(add_NDRE3)
                                .map(add_CIre)
                                .map(add_MTCI1)
                                .map(add_MTCI2)
                                .map(add_MTCI3)
                                .map(add_MCARI1)
                                .map(add_MCARI2)
                                .map(add_MCARI3)
                                .map(add_NDI45)
                                .map(add_PSSRa)
                                .map(add_IRECI)
                                .map(add_NDVI56)
                                .map(add_NDVI57)
                                .map(add_NDVI68a)
                                .map(add_NDVI78a)
                                .map(add_NLI)
                                .map(add_kNDVI)
                                .select(['B1','B2', 'B3', 'B4', 'B5', 'B6','B7','B8','B8A', 'B9','B10','B11','B12',
                                         'NIRv', 'GNDVI', 'EVI1', 'EVI2', 'CIgreen', 'SR', 'DVI', 'SAVI', 'OSAVI', 'NDWI1', 'NDWI2', 'EVIre1', 'EVIre2', 'EVIre3',
                                         'NDRE1','NDRE2','NDRE3', 'CIre','MTCI1', 'MTCI2', 'MTCI3', 'MCARI1', 'MCARI2', 'MCARI3', 'NDI45', 'PSSRa', 'IRECI', 'NDVI56', 'NDVI57', 'NDVI68a',
                                         'NDVI78a', 'NLI', 'kNDVI'                                      
                                        ])
                                .map(lambda image: image.toFloat())
                                # .map(lambda image: image.toDouble())
                                .map(add_s2_prefix_to_all_bands)
                            )
                                

            # Check bands before reduction
            collection_size = s2_collection.size().getInfo()
            
            if collection_size == 0:
                # No images found for this feature, skip processing
                print(f"No images found for chunk {chunk_idx}. Skipping...")
                continue  # skip to next chunk
                
            else:

                # 1. Extract the year from the first image in the collection
                # use .first() to get one image, then format its date to 'YYYY'
                first_image = s2_collection.first()
                year_string = first_image.date().format('YYYY')

                # Define a combined reducer that computes min, max, mean, median, and stdDev for every band
                stats_reducer = (ee.Reducer.minMax()
                                 .combine(reducer2=ee.Reducer.mean(), sharedInputs=True)
                                 .combine(reducer2=ee.Reducer.median(), sharedInputs=True)
                                 .combine(reducer2=ee.Reducer.stdDev(), sharedInputs=True))
               
                # Reduce the ImageCollection to single image with stats_reducer, with each band in Float consistently
                s2_stats = s2_collection.reduce(stats_reducer)
                bands = s2_collection.first().bandNames()
                
                # # check band before reduction
                # print("\nBands before ee.Reducer:", s2_collection.first().bandNames().getInfo())
                # # Check bands after reduction
                # print("Bands after ee.Reducer:", s2_stats.bandNames().getInfo())
                
                # # 2. Perform reduceRegions on the yearly mean image
                # This calculates the mean pixel value within each polygon of the chunk
                s2_yearly_stats = s2_stats.reduceRegions(
                    collection=chunk_fc,
                    reducer=ee.Reducer.mean(), 
                    scale=10,
                    crs='EPSG:3857',
                    tileScale=1)
                
                # Add range = max - min for every band
                s2_yearly_stats = s2_yearly_stats.map(lambda f: f.set(
                    ee.Dictionary.fromLists(
                        # Keys: e.g. "S2_B4_range"
                        bands.map(lambda b: ee.String(b).cat('_range')),
                        # Values: max - min (null if no valid pixels)
                        bands.map(lambda b: 
                            ee.Number(f.get(ee.String(b).cat('_max'))).subtract(
                                ee.Number(f.get(ee.String(b).cat('_min')))
                            )
                        )
                    )
                ))

                # Add a property with the year
                s2_yearly_stats = s2_yearly_stats.map(lambda f: f.set('system:1styear', year_string))
                
                # Create a descriptive task name for export
                task_desc = f'sentinel-2_{basename}_YearlyStats_chunk_{chunk_idx}'
                
                # Set up and start an export task to Google Drive
                task = ee.batch.Export.table.toDrive(**{
                    'collection': s2_yearly_stats,  # The combined FeatureCollection with time series
                    'description': task_desc,  # Set file name
                    'folder': outdir,          # Folder on Google Drive
                    'fileFormat': 'CSV',       # Format (use uppercase for consistency)
                    # 'selectors': ['date'] + list(s2_collection.first().bandNames().getInfo())  # Include date and bands
                })
                
                task.start()
                
                print(f'\n    → Task: {task_desc} submitted '
                      f'({offset:,} – {min(offset+CHUNK_SIZE, total_cells)-1:,})')
    
                # Optional: sleep to avoid submitting too many tasks too quickly
                # Each project's queue supports a maximum of 3,000 tasks
                import time
                time.sleep(6) # sleep interval to aviod creating multi outdir in google drive
                # do a pre-check of time/chunk to set this sleep value

        

# sentinel-1

## sentinel-1 monthly median

In [ ]:
    
def RefinedLee(img):
    """
    Applies the Refined Lee Speckle Filter to each band of an image individually.
    The image must be in natural units (not in dB).
    The processed bands will be named as 'original_band_name_lee'.
    
    Important:
    - The input image must be in natural units (linear scale), not in dB.
    - The output image will have the same number of bands as the input, with each band renamed
      by appending '_lee' to the original band name.
    
    Parameters:
    img (ee.Image): Input Earth Engine Image with one or more bands.
    
    Returns:
    ee.Image: Filtered image with bands named as 'original_band_name_lee'.
    """

    # Get the list of band names.
    band_names = img.bandNames()

    # Define the function to process each band.
    def per_band(bandName):
        bandName = ee.String(bandName)
        band = img.select(bandName)
        
        # Set up 5x5 kernels.
        weights5 = ee.List.repeat(ee.List.repeat(1, 5), 5)
        kernel5 = ee.Kernel.fixed(5, 5, weights5, 2, 2, False)
    
        # Compute mean and variance using the 5x5 kernel.
        mean5 = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=kernel5
        )
        variance5 = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=kernel5
        )
    
        # Use a sample of the 5x5 windows inside a 9x9 window to determine gradients and directions.
        sample_weights = ee.List([
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0]
        ])
        sample_kernel = ee.Kernel.fixed(9, 9, sample_weights, 4, 4, False)
    
        # Calculate mean and variance for the sampled windows and store as bands.
        sample_mean = mean5.neighborhoodToBands(kernel=sample_kernel)
        sample_var = variance5.neighborhoodToBands(kernel=sample_kernel)
    
        # Determine the gradients for the sampled windows.
        gradients = sample_mean.select(1).subtract(sample_mean.select(7)).abs()
        gradients = gradients.addBands(
            sample_mean.select(6).subtract(sample_mean.select(2)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(3).subtract(sample_mean.select(5)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(0).subtract(sample_mean.select(8)).abs()
        )
    
        # Find the maximum gradient among gradient bands.
        max_gradient = gradients.reduce(ee.Reducer.max())
    
        # Create a mask for pixels that have the maximum gradient.
        gradmask = gradients.eq(max_gradient)
        # Duplicate gradmask bands: each gradient represents 2 directions.
        gradmask = gradmask.addBands(gradmask)
    
        # Determine the 8 directions.
        directions = sample_mean.select(1).subtract(sample_mean.select(4)) \
            .gt(sample_mean.select(4).subtract(sample_mean.select(7))).multiply(1)
        directions = directions.addBands(
            sample_mean.select(6).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(2))).multiply(2)
        )
        directions = directions.addBands(
            sample_mean.select(3).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(5))).multiply(3)
        )
        directions = directions.addBands(
            sample_mean.select(0).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(8))).multiply(4)
        )
    
        # The next 4 are the Not() of the previous 4.
        directions = directions.addBands(directions.select(0).Not().multiply(5))
        directions = directions.addBands(directions.select(1).Not().multiply(6))
        directions = directions.addBands(directions.select(2).Not().multiply(7))
        directions = directions.addBands(directions.select(3).Not().multiply(8))
    
        # Mask all values that are not 1-8.
        directions = directions.updateMask(gradmask)
    
        # "Collapse" the stack into a single band image.
        directions = directions.reduce(ee.Reducer.sum())
    
        # Calculate local noise variance.
        sample_stats = sample_var.divide(sample_mean.multiply(sample_mean))
        sigmaV = sample_stats.toArray() \
            .arraySort() \
            .arraySlice(0, 0, 5) \
            .arrayReduce(ee.Reducer.mean(), [0])
    
        # Set up the 9x9 kernels for directional statistics.
        rect_weights = ee.List.repeat(ee.List.repeat(0, 9), 4) \
            .cat(ee.List.repeat(ee.List.repeat(1, 9), 5))
        diag_weights = ee.List([
            [1, 0, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 1]
        ])
    
        rect_kernel = ee.Kernel.fixed(9, 9, rect_weights, 4, 4, False)
        diag_kernel = ee.Kernel.fixed(9, 9, diag_weights, 4, 4, False)
    
        # Create stacks for mean and variance using the directional kernels.
        dir_mean = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
        dir_var = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
    
        dir_mean = dir_mean.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.mean(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
        dir_var = dir_var.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.variance(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
    
        # Add bands for rotated kernels.
        for i in range(1, 4):
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
    
        # "Collapse" the stack into single band images.
        dir_mean = dir_mean.reduce(ee.Reducer.sum())
        dir_var = dir_var.reduce(ee.Reducer.sum())
    
        # Generate the filtered value.
        varX = dir_var.subtract(
            dir_mean.multiply(dir_mean).multiply(sigmaV)
        ).divide(sigmaV.add(1.0))
    
        b = varX.divide(dir_var)
        result = dir_mean.add(
            b.multiply(band.subtract(dir_mean))
        )
    
        # Return the result with the new band name.
        result = result.arrayFlatten([['sum']]).rename(bandName.cat('_lee'))
        return result

    # Map over the band names using the per_band function.
    filtered_images = band_names.map(per_band)

    # Convert the list of images into an ImageCollection.
    filtered_collection = ee.ImageCollection(filtered_images)

    # Convert the ImageCollection to a single image.
    filtered_image = filtered_collection.toBands()

    # Collect the new band names.
    new_band_names = band_names.map(lambda name: ee.String(name).cat('_lee'))

    # Rename the bands of the filtered image.
    filtered_image = filtered_image.rename(new_band_names)

    return filtered_image



def apply_RefinedLee(image):
    """Applies the Refined Lee filter to the 'HH' and 'HV' bands.

    Parameters:
    image (ee.Image): Input image containing radar bands.
    
    Returns:
    ee.Image: Image containing original 'HH' and 'HV' bands plus their filtered versions
              named 'HH_lee' and 'HV_lee'.
    """
    # Get the list of bands in the image
    band_names = image.bandNames()
    
    # Define the target bands
    target_bands = ee.List(['VV', 'VH'])
    
    # Filter the target bands to those that exist in the image
    #existing_bands = target_bands.filter(lambda b: band_names.contains(b))
    
    # Function to process each existing band
    def process_band(band_name):
        band_name = ee.String(band_name)
        band = image.select(band_name)
        #band_natural = toNatural(band)
        band_filtered = RefinedLee(band)
        #band_filtered_db = todb(band_filtered)
        return band_filtered
    
    # Map the processing function over the existing bands
    filtered_bands_list = target_bands.map(process_band)
    
    # Convert the list of images to a single image
    filtered_bands_image = ee.ImageCollection(filtered_bands_list).toBands()

    # Add the filtered bands to the original image
    return image.addBands(filtered_bands_image).rename(['VV','VH','VV_lee','VH_lee'])
    


def todb(image):
    """
    Converts filtered 'HH_lee' and 'HV_lee' bands from natural units back to decibel (dB) scale.
    
    The conversion formula used is: 10 * log10((DN)^2- 83.0 db
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee' and 'HV_lee' bands in natural units.
    
    Returns:
    ee.Image: Image with additional bands 'HH_lee_db' and 'HV_lee_db' in dB scale.
    """

    vv = image.select('VV_lee')
    vvdb = vv.log10().multiply(10).rename('VV_lee_db')
    vh = image.select('VH_lee')
    vhdb = vh.log10().multiply(10).rename('VH_lee_db')
    return image.addBands(vvdb).addBands(vhdb)



def toInt(image):
    """
    Converts 'HH_lee_db' and 'HV_lee_db' bands from float dB values to 16-bit integer scale.
    
    The conversion:
    - Scales values from the range [-50, 10] dB to [0, 65535] integer range.
    - Converts scaled values to 32-bit integers.
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with additional integer bands 'HH_lee_db_int' and 'HV_lee_db_int'.
    """

    hh = image.select('VV_lee_db')
    hhint = hh.unitScale(-50, 1).multiply(65535).toUint16().rename('VV_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    hv = image.select('VH_lee_db')
    hvint = hv.unitScale(-50, 1).multiply(65535).toUint16().rename('VH_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    return image.addBands(hhint).addBands(hvint)



def addTexture(image, glcmsize=5):
    """
    Adds GLCM texture measures (average) to specified bands of an image.
    Applies texture calculation only to bands listed in 'bands_to_analyze'.
    
    Parameters:
    image (ee.Image): Input image containing bands to analyze.
    glcmsize (int): Size of the window (kernel) for GLCM calculation (default 5).
    
    Returns:
    ee.Image: Image with added texture bands for specified bands.
    """
    bands_to_analyze = ee.List(['VV_lee_db_int', 'VH_lee_db_int'])  # Bands to apply GLCM to
    
    def applyTexture(band, img):
        band = ee.String(band)
        condition = bands_to_analyze.contains(band)
        return ee.Image(ee.Algorithms.If(
            condition,
            ee.Image(img).addBands(
                ee.Image(img).select([band])
                .glcmTexture(size=glcmsize, average=True)
            ),
            img
        ))

    # Apply GLCM texture to HH and HV bands
    band_names = image.bandNames()
    textured_image = ee.Image(band_names.iterate(applyTexture, image))
    
    return textured_image


def VHVV_sum(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'HH&HV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH&HV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    vh_add_vv = vh.add(vv).rename(['VH&VV'])   
    return image.addBands(vh_add_vv)

def VHVV_sub(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'VH-VV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'VH-VV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    vhvv_sub = vh.subtract(vv).rename(['VH-VV'])   
    return image.addBands(vhvv_sub)


def VHVV_ratio(image):
    """
    Calculates the ratio of HH to HV bands and appends it as a new band.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH_HV_ratio'.
    """
    # 1. Select the bands
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    
    # 2. Perform the division (HH / HV)
    # Using .divide() handles the pixel-wise math on the server side
    vhvv_ratio = vh.divide(vv).rename('VH/VV')
    
    # 3. Return the image with the new band added
    return image.addBands(vhvv_ratio)


def RVI(image):
    """
    Calculates the ratio of HH to HV bands and appends it as a new band.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH_HV_ratio'.
    """
    # 1. Select the bands
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    
    # 2. Perform the division (HH / HV)
    # Using .divide() handles the pixel-wise math on the server side
    rvi = vh.divide(vh.add(vv)).rename('RVI')
    
    # 3. Return the image with the new band added
    return image.addBands(rvi)

    

# New function to prefix computed indices with 'S2_' while keeping raw bands (B*) unchanged
def add_s1_prefix_to_all_bands(image):
    """
    Renames EVERY band in the image by adding 'P2_' prefix.
    This applies to both raw Sentinel-2 bands (e.g., 'HH' → 'P2_HH')
    """
    band_names = image.bandNames()
    new_names = band_names.map(lambda name: ee.String('S1_').cat(name))
    return image.rename(new_names)


    
if __name__ == "__main__":

    # define chunk size
    CHUNK_SIZE = 1
    
    # Output directory name on Google Drive for exported images  
    outdir = '_sentinel1_monthly_median' 
    
    # gee assets path
    parent = 'projects/fates-mrv/assets'
    
    # Ask EE for the list of *direct* children of that folder
    resp = ee.data.listAssets({'parent': parent})  # dict with key 'assets'
    assets_meta = resp.get('assets', [])           # list of dictionaries
    
    # Keep only assets whose type == 'TABLE'   (vectors/shapefiles)
    table_ids = [a['name'] for a in assets_meta if a['type'] == 'TABLE']
    print('Found', len(table_ids), 'vector assets')

     # Print all found vector asset IDs with their index
    for i, aid in enumerate(table_ids):
        print(f'{i:2d}. {aid}')
    
    # 1. Define the monthss
    year = 2019     
    START_DATE = f'{year}-01-01'
    END_DATE   = f'{year}-12-31'
      
    # Dynamic monthly
    months = [
        ('Jan', f'{year}-01-01', f'{year}-01-31'),
        ('Feb', f'{year}-02-01', f'{year}-02-28'),
        ('Mar', f'{year}-03-01', f'{year}-03-31'),
        ('Apr', f'{year}-04-01', f'{year}-04-30'),
        ('May', f'{year}-05-01', f'{year}-05-31'),
        ('Jun', f'{year}-06-01', f'{year}-06-30'),
        ('Jul', f'{year}-07-01', f'{year}-07-31'),
        ('Aug', f'{year}-08-01', f'{year}-08-31'),
        ('Sep', f'{year}-09-01', f'{year}-09-30'),
        ('Oct', f'{year}-10-01', f'{year}-10-31'),
        ('Nov', f'{year}-11-01', f'{year}-11-30'),
        ('Dec', f'{year}-12-01', f'{year}-12-31'),
    ]
    # Loop over all vector assets
    for asset_id in tqdm(table_ids):
        #print(table_ids)

        # Filter to process only assets "fishnet"
        if not asset_id.endswith(f'{year}'):
            continue  # Skip if it does not end with ''

        # Extract basename from asset ID for naming outputs
        basename = os.path.basename(asset_id)      # ← fishnet
        print('\n───────────────────────────────────────────────')
        print('Processing:', asset_id)

        # Load the vector asset as a FeatureCollection
        fishnet = ee.FeatureCollection(asset_id)
        total_cells = fishnet.size().getInfo()
        print('\nNumber of grid cells :', total_cells)
        print('Schema columns:', fishnet.first().propertyNames().getInfo()) 

        # spatial tiling : loop over the grid in chunks
        # for offset in range(0, total_cells, CHUNK_SIZE):
        for offset in range(5690, total_cells):
            chunk_idx = offset // CHUNK_SIZE
            chunk_fc  = ee.FeatureCollection(fishnet.toList(CHUNK_SIZE, offset))
            chunk_geom = chunk_fc.geometry()

            # Build the ImageCollection with processing steps
            sar_collection = (ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')
                              .filterBounds(chunk_geom)
                              .filterDate(START_DATE, END_DATE)
                              .filter(ee.Filter.And(ee.Filter.listContains('system:band_names', 'VH'), 
                                                    ee.Filter.listContains('system:band_names', 'VV')))
                              .select(['VH', 'VV'])
                              .map(apply_RefinedLee)
                              .map(todb)
                              .map(toInt)
                              .map(addTexture)
                              .map(VHVV_sum)
                              .map(VHVV_sub)
                              .map(VHVV_ratio)
                              .map(RVI)
                              .map(add_s1_prefix_to_all_bands)
                             )
            
            # Check bands before reduction
            collection_size = sar_collection.size().getInfo()
            if collection_size == 0:
                # No images found for this feature, skip processing
                print(f"No images found for chunk {chunk_idx} in {year}. Skipping...")
                continue  # skip to next chunk

            # monthly processing
            all_monthly_fc = None
            has_month_data = False

            for m_name, m_start, m_end in months:
                m_col = sar_collection.filterDate(m_start, m_end)
                if m_col.size().getInfo() == 0:
                    print(f"  No images in {m_name} for chunk {chunk_idx}")
                    continue

                # composite for the month
                composite = m_col.median()

                # Zonal mean extraction
                reduced = composite.reduceRegions(
                    collection=chunk_fc,
                    reducer=ee.Reducer.mean(),
                    scale=10,
                    crs='EPSG:3857',
                    tileScale=1
                )

                reduced_q = reduced.map(lambda f: f.set({
                    'quarter': m_name,
                    'year': year,
                    'start_date': m_start,
                    'end_date': m_end
                }))

                if all_monthly_fc is None:
                    all_monthly_fc = reduced_q
                else:
                    all_monthly_fc = all_monthly_fc.merge(reduced_q)
                
                has_month_data = True

            if not has_month_data:
                print(f"No valid monthly data for chunk {chunk_idx}. Skipping export.")
                continue        

                
            # Create a descriptive task name for export
            task_desc = f'sentinel-1_{basename}_monthly_chunk_{chunk_idx}'
            
            # Set up and start an export task to Google Drive
            task = ee.batch.Export.table.toDrive(**{
                'collection': all_monthly_fc,  # The combined FeatureCollection with time series
                'description': task_desc,  # Set file name
                'folder': outdir,          # Folder on Google Drive
                'fileFormat': 'CSV',       # Format (use uppercase for consistency)
                # 'selectors': ['date'] + list(sar_collection.first().bandNames().getInfo())  # Include date and bands
            })
            
            task.start()
            
            print(f'\n    → Task: {task_desc} submitted '
                  f'({offset:,} – {min(offset+CHUNK_SIZE, total_cells)-1:,})')

            # Optional: sleep to avoid submitting too many tasks too quickly
            # Each project's queue supports a maximum of 3,000 tasks
            # import time
            # time.sleep(3) # sleep interval to aviod creating multi outdir in google drive
            # do a pre-check of time/chunk to set this sleep value


## sentinel-1 quarterly median

In [ ]:
    
def RefinedLee(img):
    """
    Applies the Refined Lee Speckle Filter to each band of an image individually.
    The image must be in natural units (not in dB).
    The processed bands will be named as 'original_band_name_lee'.
    
    Important:
    - The input image must be in natural units (linear scale), not in dB.
    - The output image will have the same number of bands as the input, with each band renamed
      by appending '_lee' to the original band name.
    
    Parameters:
    img (ee.Image): Input Earth Engine Image with one or more bands.
    
    Returns:
    ee.Image: Filtered image with bands named as 'original_band_name_lee'.
    """

    # Get the list of band names.
    band_names = img.bandNames()

    # Define the function to process each band.
    def per_band(bandName):
        bandName = ee.String(bandName)
        band = img.select(bandName)
        
        # Set up 5x5 kernels.
        weights5 = ee.List.repeat(ee.List.repeat(1, 5), 5)
        kernel5 = ee.Kernel.fixed(5, 5, weights5, 2, 2, False)
    
        # Compute mean and variance using the 5x5 kernel.
        mean5 = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=kernel5
        )
        variance5 = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=kernel5
        )
    
        # Use a sample of the 5x5 windows inside a 9x9 window to determine gradients and directions.
        sample_weights = ee.List([
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0]
        ])
        sample_kernel = ee.Kernel.fixed(9, 9, sample_weights, 4, 4, False)
    
        # Calculate mean and variance for the sampled windows and store as bands.
        sample_mean = mean5.neighborhoodToBands(kernel=sample_kernel)
        sample_var = variance5.neighborhoodToBands(kernel=sample_kernel)
    
        # Determine the gradients for the sampled windows.
        gradients = sample_mean.select(1).subtract(sample_mean.select(7)).abs()
        gradients = gradients.addBands(
            sample_mean.select(6).subtract(sample_mean.select(2)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(3).subtract(sample_mean.select(5)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(0).subtract(sample_mean.select(8)).abs()
        )
    
        # Find the maximum gradient among gradient bands.
        max_gradient = gradients.reduce(ee.Reducer.max())
    
        # Create a mask for pixels that have the maximum gradient.
        gradmask = gradients.eq(max_gradient)
        # Duplicate gradmask bands: each gradient represents 2 directions.
        gradmask = gradmask.addBands(gradmask)
    
        # Determine the 8 directions.
        directions = sample_mean.select(1).subtract(sample_mean.select(4)) \
            .gt(sample_mean.select(4).subtract(sample_mean.select(7))).multiply(1)
        directions = directions.addBands(
            sample_mean.select(6).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(2))).multiply(2)
        )
        directions = directions.addBands(
            sample_mean.select(3).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(5))).multiply(3)
        )
        directions = directions.addBands(
            sample_mean.select(0).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(8))).multiply(4)
        )
    
        # The next 4 are the Not() of the previous 4.
        directions = directions.addBands(directions.select(0).Not().multiply(5))
        directions = directions.addBands(directions.select(1).Not().multiply(6))
        directions = directions.addBands(directions.select(2).Not().multiply(7))
        directions = directions.addBands(directions.select(3).Not().multiply(8))
    
        # Mask all values that are not 1-8.
        directions = directions.updateMask(gradmask)
    
        # "Collapse" the stack into a single band image.
        directions = directions.reduce(ee.Reducer.sum())
    
        # Calculate local noise variance.
        sample_stats = sample_var.divide(sample_mean.multiply(sample_mean))
        sigmaV = sample_stats.toArray() \
            .arraySort() \
            .arraySlice(0, 0, 5) \
            .arrayReduce(ee.Reducer.mean(), [0])
    
        # Set up the 9x9 kernels for directional statistics.
        rect_weights = ee.List.repeat(ee.List.repeat(0, 9), 4) \
            .cat(ee.List.repeat(ee.List.repeat(1, 9), 5))
        diag_weights = ee.List([
            [1, 0, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 1]
        ])
    
        rect_kernel = ee.Kernel.fixed(9, 9, rect_weights, 4, 4, False)
        diag_kernel = ee.Kernel.fixed(9, 9, diag_weights, 4, 4, False)
    
        # Create stacks for mean and variance using the directional kernels.
        dir_mean = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
        dir_var = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
    
        dir_mean = dir_mean.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.mean(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
        dir_var = dir_var.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.variance(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
    
        # Add bands for rotated kernels.
        for i in range(1, 4):
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
    
        # "Collapse" the stack into single band images.
        dir_mean = dir_mean.reduce(ee.Reducer.sum())
        dir_var = dir_var.reduce(ee.Reducer.sum())
    
        # Generate the filtered value.
        varX = dir_var.subtract(
            dir_mean.multiply(dir_mean).multiply(sigmaV)
        ).divide(sigmaV.add(1.0))
    
        b = varX.divide(dir_var)
        result = dir_mean.add(
            b.multiply(band.subtract(dir_mean))
        )
    
        # Return the result with the new band name.
        result = result.arrayFlatten([['sum']]).rename(bandName.cat('_lee'))
        return result

    # Map over the band names using the per_band function.
    filtered_images = band_names.map(per_band)

    # Convert the list of images into an ImageCollection.
    filtered_collection = ee.ImageCollection(filtered_images)

    # Convert the ImageCollection to a single image.
    filtered_image = filtered_collection.toBands()

    # Collect the new band names.
    new_band_names = band_names.map(lambda name: ee.String(name).cat('_lee'))

    # Rename the bands of the filtered image.
    filtered_image = filtered_image.rename(new_band_names)

    return filtered_image



def apply_RefinedLee(image):
    """Applies the Refined Lee filter to the 'HH' and 'HV' bands.

    Parameters:
    image (ee.Image): Input image containing radar bands.
    
    Returns:
    ee.Image: Image containing original 'HH' and 'HV' bands plus their filtered versions
              named 'HH_lee' and 'HV_lee'.
    """
    # Get the list of bands in the image
    band_names = image.bandNames()
    
    # Define the target bands
    target_bands = ee.List(['VV', 'VH'])
    
    # Filter the target bands to those that exist in the image
    #existing_bands = target_bands.filter(lambda b: band_names.contains(b))
    
    # Function to process each existing band
    def process_band(band_name):
        band_name = ee.String(band_name)
        band = image.select(band_name)
        #band_natural = toNatural(band)
        band_filtered = RefinedLee(band)
        #band_filtered_db = todb(band_filtered)
        return band_filtered
    
    # Map the processing function over the existing bands
    filtered_bands_list = target_bands.map(process_band)
    
    # Convert the list of images to a single image
    filtered_bands_image = ee.ImageCollection(filtered_bands_list).toBands()

    # Add the filtered bands to the original image
    return image.addBands(filtered_bands_image).rename(['VV','VH','VV_lee','VH_lee'])
    


def todb(image):
    """
    Converts filtered 'HH_lee' and 'HV_lee' bands from natural units back to decibel (dB) scale.
    
    The conversion formula used is: 10 * log10((DN)^2- 83.0 db
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee' and 'HV_lee' bands in natural units.
    
    Returns:
    ee.Image: Image with additional bands 'HH_lee_db' and 'HV_lee_db' in dB scale.
    """

    vv = image.select('VV_lee')
    vvdb = vv.log10().multiply(10).rename('VV_lee_db')
    vh = image.select('VH_lee')
    vhdb = vh.log10().multiply(10).rename('VH_lee_db')
    return image.addBands(vvdb).addBands(vhdb)



def toInt(image):
    """
    Converts 'HH_lee_db' and 'HV_lee_db' bands from float dB values to 16-bit integer scale.
    
    The conversion:
    - Scales values from the range [-50, 10] dB to [0, 65535] integer range.
    - Converts scaled values to 32-bit integers.
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with additional integer bands 'HH_lee_db_int' and 'HV_lee_db_int'.
    """

    hh = image.select('VV_lee_db')
    hhint = hh.unitScale(-50, 1).multiply(65535).toUint16().rename('VV_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    hv = image.select('VH_lee_db')
    hvint = hv.unitScale(-50, 1).multiply(65535).toUint16().rename('VH_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    return image.addBands(hhint).addBands(hvint)



def addTexture(image, glcmsize=5):
    """
    Adds GLCM texture measures (average) to specified bands of an image.
    Applies texture calculation only to bands listed in 'bands_to_analyze'.
    
    Parameters:
    image (ee.Image): Input image containing bands to analyze.
    glcmsize (int): Size of the window (kernel) for GLCM calculation (default 5).
    
    Returns:
    ee.Image: Image with added texture bands for specified bands.
    """
    bands_to_analyze = ee.List(['VV_lee_db_int', 'VH_lee_db_int'])  # Bands to apply GLCM to
    
    def applyTexture(band, img):
        band = ee.String(band)
        condition = bands_to_analyze.contains(band)
        return ee.Image(ee.Algorithms.If(
            condition,
            ee.Image(img).addBands(
                ee.Image(img).select([band])
                .glcmTexture(size=glcmsize, average=True)
            ),
            img
        ))

    # Apply GLCM texture to HH and HV bands
    band_names = image.bandNames()
    textured_image = ee.Image(band_names.iterate(applyTexture, image))
    
    return textured_image


def VHVV_sum(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'HH&HV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH&HV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    vh_add_vv = vh.add(vv).rename(['VH&VV'])   
    return image.addBands(vh_add_vv)

def VHVV_sub(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'VH-VV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'VH-VV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    vhvv_sub = vh.subtract(vv).rename(['VH-VV'])   
    return image.addBands(vhvv_sub)


def VHVV_ratio(image):
    """
    Calculates the ratio of HH to HV bands and appends it as a new band.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH_HV_ratio'.
    """
    # 1. Select the bands
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    
    # 2. Perform the division (HH / HV)
    # Using .divide() handles the pixel-wise math on the server side
    vhvv_ratio = vh.divide(vv).rename('VH/VV')
    
    # 3. Return the image with the new band added
    return image.addBands(vhvv_ratio)


def RVI(image):
    """
    Calculates the ratio of HH to HV bands and appends it as a new band.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH_HV_ratio'.
    """
    # 1. Select the bands
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    
    # 2. Perform the division (HH / HV)
    # Using .divide() handles the pixel-wise math on the server side
    rvi = vh.divide(vh.add(vv)).rename('RVI')
    
    # 3. Return the image with the new band added
    return image.addBands(rvi)

    

# New function to prefix computed indices with 'S2_' while keeping raw bands (B*) unchanged
def add_s1_prefix_to_all_bands(image):
    """
    Renames EVERY band in the image by adding 'P2_' prefix.
    This applies to both raw Sentinel-2 bands (e.g., 'HH' → 'P2_HH')
    """
    band_names = image.bandNames()
    new_names = band_names.map(lambda name: ee.String('S1_').cat(name))
    return image.rename(new_names)


    
if __name__ == "__main__":

    # define chunk size
    CHUNK_SIZE = 1
    
    # Output directory name on Google Drive for exported images  
    outdir = '_sentinel1_quarterly_median' 
    
    # gee assets path
    parent = 'projects/fates-mrv/assets'
    
    # Ask EE for the list of *direct* children of that folder
    resp = ee.data.listAssets({'parent': parent})  # dict with key 'assets'
    assets_meta = resp.get('assets', [])           # list of dictionaries
    
    # Keep only assets whose type == 'TABLE'   (vectors/shapefiles)
    table_ids = [a['name'] for a in assets_meta if a['type'] == 'TABLE']
    print('Found', len(table_ids), 'vector assets')

     # Print all found vector asset IDs with their index
    for i, aid in enumerate(table_ids):
        print(f'{i:2d}. {aid}')
    
    # 1. Define the Quarters
    year = 2019     
    START_DATE = f'{year}-01-01'
    END_DATE   = f'{year}-12-31'
      
    # Dynamic quarters
    quarters = [
            ('Q1', f'{year}-01-01', f'{year}-03-31'),
            ('Q2', f'{year}-04-01', f'{year}-06-30'),
            ('Q3', f'{year}-07-01', f'{year}-09-30'),
            ('Q4', f'{year}-10-01', f'{year}-12-31'),
        ]
    
    # Loop over all vector assets
    for asset_id in tqdm(table_ids):
        #print(table_ids)

        # Filter to process only assets "fishnet"
        if not asset_id.endswith(f'{year}'):
            continue  # Skip if it does not end with ''

        # Extract basename from asset ID for naming outputs
        basename = os.path.basename(asset_id)      # ← fishnet
        print('\n───────────────────────────────────────────────')
        print('Processing:', asset_id)

        # Load the vector asset as a FeatureCollection
        fishnet = ee.FeatureCollection(asset_id)
        total_cells = fishnet.size().getInfo()
        print('\nNumber of grid cells :', total_cells)
        print('Schema columns:', fishnet.first().propertyNames().getInfo()) 

        # spatial tiling : loop over the grid in chunks
        for offset in range(0, total_cells, CHUNK_SIZE):
        # for offset in range(3108, total_cells):
            chunk_idx = offset // CHUNK_SIZE
            chunk_fc  = ee.FeatureCollection(fishnet.toList(CHUNK_SIZE, offset))
            chunk_geom = chunk_fc.geometry()

            # Build the ImageCollection with processing steps
            sar_collection = (ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')
                              .filterBounds(chunk_geom)
                              .filterDate(START_DATE, END_DATE)
                              .filter(ee.Filter.And(ee.Filter.listContains('system:band_names', 'VH'), 
                                                    ee.Filter.listContains('system:band_names', 'VV')))
                              .select(['VH', 'VV'])
                              .map(apply_RefinedLee)
                              .map(todb)
                              .map(toInt)
                              .map(addTexture)
                              .map(VHVV_sum)
                              .map(VHVV_sub)
                              .map(VHVV_ratio)
                              .map(RVI)
                              .map(add_s1_prefix_to_all_bands)
                             )
            
            # Check bands before reduction
            collection_size = sar_collection.size().getInfo()
            if collection_size == 0:
                # No images found for this feature, skip processing
                print(f"No images found for chunk {chunk_idx} in {year}. Skipping...")
                continue  # skip to next chunk

            # Quarterly processing
            all_quarterly_fc = None
            has_quarter_data = False

            for q_name, q_start, q_end in quarters:
                q_col = sar_collection.filterDate(q_start, q_end)
                if q_col.size().getInfo() == 0:
                    print(f"  No images in {q_name} for chunk {chunk_idx}")
                    continue

                # composite for the quarter
                composite = q_col.median()

                # Zonal mean extraction
                reduced = composite.reduceRegions(
                    collection=chunk_fc,
                    reducer=ee.Reducer.mean(),
                    scale=10,
                    crs='EPSG:3857',
                    tileScale=1
                )

                reduced_q = reduced.map(lambda f: f.set({
                    'quarter': q_name,
                    'year': year,
                    'start_date': q_start,
                    'end_date': q_end
                }))

                if all_quarterly_fc is None:
                    all_quarterly_fc = reduced_q
                else:
                    all_quarterly_fc = all_quarterly_fc.merge(reduced_q)
                
                has_quarter_data = True

            if not has_quarter_data:
                print(f"No valid quarterly data for chunk {chunk_idx}. Skipping export.")
                continue        

                
            # Create a descriptive task name for export
            task_desc = f'sentinel-1_{basename}_quarterly_chunk_{chunk_idx}'
            
            # Set up and start an export task to Google Drive
            task = ee.batch.Export.table.toDrive(**{
                'collection': all_quarterly_fc,  # The combined FeatureCollection with time series
                'description': task_desc,  # Set file name
                'folder': outdir,          # Folder on Google Drive
                'fileFormat': 'CSV',       # Format (use uppercase for consistency)
                # 'selectors': ['date'] + list(sar_collection.first().bandNames().getInfo())  # Include date and bands
            })
            
            task.start()
            
            print(f'\n    → Task: {task_desc} submitted '
                  f'({offset:,} – {min(offset+CHUNK_SIZE, total_cells)-1:,})')

            # Optional: sleep to avoid submitting too many tasks too quickly
            # Each project's queue supports a maximum of 3,000 tasks
            import time
            time.sleep(6) # sleep interval to aviod creating multi outdir in google drive
            # do a pre-check of time/chunk to set this sleep value


## sentinel-1 semiannual median

In [ ]:
    
def RefinedLee(img):
    """
    Applies the Refined Lee Speckle Filter to each band of an image individually.
    The image must be in natural units (not in dB).
    The processed bands will be named as 'original_band_name_lee'.
    
    Important:
    - The input image must be in natural units (linear scale), not in dB.
    - The output image will have the same number of bands as the input, with each band renamed
      by appending '_lee' to the original band name.
    
    Parameters:
    img (ee.Image): Input Earth Engine Image with one or more bands.
    
    Returns:
    ee.Image: Filtered image with bands named as 'original_band_name_lee'.
    """

    # Get the list of band names.
    band_names = img.bandNames()

    # Define the function to process each band.
    def per_band(bandName):
        bandName = ee.String(bandName)
        band = img.select(bandName)
        
        # Set up 5x5 kernels.
        weights5 = ee.List.repeat(ee.List.repeat(1, 5), 5)
        kernel5 = ee.Kernel.fixed(5, 5, weights5, 2, 2, False)
    
        # Compute mean and variance using the 5x5 kernel.
        mean5 = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=kernel5
        )
        variance5 = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=kernel5
        )
    
        # Use a sample of the 5x5 windows inside a 9x9 window to determine gradients and directions.
        sample_weights = ee.List([
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0]
        ])
        sample_kernel = ee.Kernel.fixed(9, 9, sample_weights, 4, 4, False)
    
        # Calculate mean and variance for the sampled windows and store as bands.
        sample_mean = mean5.neighborhoodToBands(kernel=sample_kernel)
        sample_var = variance5.neighborhoodToBands(kernel=sample_kernel)
    
        # Determine the gradients for the sampled windows.
        gradients = sample_mean.select(1).subtract(sample_mean.select(7)).abs()
        gradients = gradients.addBands(
            sample_mean.select(6).subtract(sample_mean.select(2)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(3).subtract(sample_mean.select(5)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(0).subtract(sample_mean.select(8)).abs()
        )
    
        # Find the maximum gradient among gradient bands.
        max_gradient = gradients.reduce(ee.Reducer.max())
    
        # Create a mask for pixels that have the maximum gradient.
        gradmask = gradients.eq(max_gradient)
        # Duplicate gradmask bands: each gradient represents 2 directions.
        gradmask = gradmask.addBands(gradmask)
    
        # Determine the 8 directions.
        directions = sample_mean.select(1).subtract(sample_mean.select(4)) \
            .gt(sample_mean.select(4).subtract(sample_mean.select(7))).multiply(1)
        directions = directions.addBands(
            sample_mean.select(6).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(2))).multiply(2)
        )
        directions = directions.addBands(
            sample_mean.select(3).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(5))).multiply(3)
        )
        directions = directions.addBands(
            sample_mean.select(0).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(8))).multiply(4)
        )
    
        # The next 4 are the Not() of the previous 4.
        directions = directions.addBands(directions.select(0).Not().multiply(5))
        directions = directions.addBands(directions.select(1).Not().multiply(6))
        directions = directions.addBands(directions.select(2).Not().multiply(7))
        directions = directions.addBands(directions.select(3).Not().multiply(8))
    
        # Mask all values that are not 1-8.
        directions = directions.updateMask(gradmask)
    
        # "Collapse" the stack into a single band image.
        directions = directions.reduce(ee.Reducer.sum())
    
        # Calculate local noise variance.
        sample_stats = sample_var.divide(sample_mean.multiply(sample_mean))
        sigmaV = sample_stats.toArray() \
            .arraySort() \
            .arraySlice(0, 0, 5) \
            .arrayReduce(ee.Reducer.mean(), [0])
    
        # Set up the 9x9 kernels for directional statistics.
        rect_weights = ee.List.repeat(ee.List.repeat(0, 9), 4) \
            .cat(ee.List.repeat(ee.List.repeat(1, 9), 5))
        diag_weights = ee.List([
            [1, 0, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 1]
        ])
    
        rect_kernel = ee.Kernel.fixed(9, 9, rect_weights, 4, 4, False)
        diag_kernel = ee.Kernel.fixed(9, 9, diag_weights, 4, 4, False)
    
        # Create stacks for mean and variance using the directional kernels.
        dir_mean = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
        dir_var = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
    
        dir_mean = dir_mean.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.mean(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
        dir_var = dir_var.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.variance(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
    
        # Add bands for rotated kernels.
        for i in range(1, 4):
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
    
        # "Collapse" the stack into single band images.
        dir_mean = dir_mean.reduce(ee.Reducer.sum())
        dir_var = dir_var.reduce(ee.Reducer.sum())
    
        # Generate the filtered value.
        varX = dir_var.subtract(
            dir_mean.multiply(dir_mean).multiply(sigmaV)
        ).divide(sigmaV.add(1.0))
    
        b = varX.divide(dir_var)
        result = dir_mean.add(
            b.multiply(band.subtract(dir_mean))
        )
    
        # Return the result with the new band name.
        result = result.arrayFlatten([['sum']]).rename(bandName.cat('_lee'))
        return result

    # Map over the band names using the per_band function.
    filtered_images = band_names.map(per_band)

    # Convert the list of images into an ImageCollection.
    filtered_collection = ee.ImageCollection(filtered_images)

    # Convert the ImageCollection to a single image.
    filtered_image = filtered_collection.toBands()

    # Collect the new band names.
    new_band_names = band_names.map(lambda name: ee.String(name).cat('_lee'))

    # Rename the bands of the filtered image.
    filtered_image = filtered_image.rename(new_band_names)

    return filtered_image



def apply_RefinedLee(image):
    """Applies the Refined Lee filter to the 'HH' and 'HV' bands.

    Parameters:
    image (ee.Image): Input image containing radar bands.
    
    Returns:
    ee.Image: Image containing original 'HH' and 'HV' bands plus their filtered versions
              named 'HH_lee' and 'HV_lee'.
    """
    # Get the list of bands in the image
    band_names = image.bandNames()
    
    # Define the target bands
    target_bands = ee.List(['VV', 'VH'])
    
    # Filter the target bands to those that exist in the image
    #existing_bands = target_bands.filter(lambda b: band_names.contains(b))
    
    # Function to process each existing band
    def process_band(band_name):
        band_name = ee.String(band_name)
        band = image.select(band_name)
        #band_natural = toNatural(band)
        band_filtered = RefinedLee(band)
        #band_filtered_db = todb(band_filtered)
        return band_filtered
    
    # Map the processing function over the existing bands
    filtered_bands_list = target_bands.map(process_band)
    
    # Convert the list of images to a single image
    filtered_bands_image = ee.ImageCollection(filtered_bands_list).toBands()

    # Add the filtered bands to the original image
    return image.addBands(filtered_bands_image).rename(['VV','VH','VV_lee','VH_lee'])
    


def todb(image):
    """
    Converts filtered 'HH_lee' and 'HV_lee' bands from natural units back to decibel (dB) scale.
    
    The conversion formula used is: 10 * log10((DN)^2- 83.0 db
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee' and 'HV_lee' bands in natural units.
    
    Returns:
    ee.Image: Image with additional bands 'HH_lee_db' and 'HV_lee_db' in dB scale.
    """

    vv = image.select('VV_lee')
    vvdb = vv.log10().multiply(10).rename('VV_lee_db')
    vh = image.select('VH_lee')
    vhdb = vh.log10().multiply(10).rename('VH_lee_db')
    return image.addBands(vvdb).addBands(vhdb)



def toInt(image):
    """
    Converts 'HH_lee_db' and 'HV_lee_db' bands from float dB values to 16-bit integer scale.
    
    The conversion:
    - Scales values from the range [-50, 10] dB to [0, 65535] integer range.
    - Converts scaled values to 32-bit integers.
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with additional integer bands 'HH_lee_db_int' and 'HV_lee_db_int'.
    """

    hh = image.select('VV_lee_db')
    hhint = hh.unitScale(-50, 1).multiply(65535).toUint16().rename('VV_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    hv = image.select('VH_lee_db')
    hvint = hv.unitScale(-50, 1).multiply(65535).toUint16().rename('VH_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    return image.addBands(hhint).addBands(hvint)



def addTexture(image, glcmsize=5):
    """
    Adds GLCM texture measures (average) to specified bands of an image.
    Applies texture calculation only to bands listed in 'bands_to_analyze'.
    
    Parameters:
    image (ee.Image): Input image containing bands to analyze.
    glcmsize (int): Size of the window (kernel) for GLCM calculation (default 5).
    
    Returns:
    ee.Image: Image with added texture bands for specified bands.
    """
    bands_to_analyze = ee.List(['VV_lee_db_int', 'VH_lee_db_int'])  # Bands to apply GLCM to
    
    def applyTexture(band, img):
        band = ee.String(band)
        condition = bands_to_analyze.contains(band)
        return ee.Image(ee.Algorithms.If(
            condition,
            ee.Image(img).addBands(
                ee.Image(img).select([band])
                .glcmTexture(size=glcmsize, average=True)
            ),
            img
        ))

    # Apply GLCM texture to HH and HV bands
    band_names = image.bandNames()
    textured_image = ee.Image(band_names.iterate(applyTexture, image))
    
    return textured_image


def VHVV_sum(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'HH&HV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH&HV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    vh_add_vv = vh.add(vv).rename(['VH&VV'])   
    return image.addBands(vh_add_vv)

def VHVV_sub(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'VH-VV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'VH-VV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    vhvv_sub = vh.subtract(vv).rename(['VH-VV'])   
    return image.addBands(vhvv_sub)


def VHVV_ratio(image):
    """
    Calculates the ratio of HH to HV bands and appends it as a new band.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH_HV_ratio'.
    """
    # 1. Select the bands
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    
    # 2. Perform the division (HH / HV)
    # Using .divide() handles the pixel-wise math on the server side
    vhvv_ratio = vh.divide(vv).rename('VH/VV')
    
    # 3. Return the image with the new band added
    return image.addBands(vhvv_ratio)


def RVI(image):
    """
    Calculates the ratio of HH to HV bands and appends it as a new band.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH_HV_ratio'.
    """
    # 1. Select the bands
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    
    # 2. Perform the division (HH / HV)
    # Using .divide() handles the pixel-wise math on the server side
    rvi = vh.divide(vh.add(vv)).rename('RVI')
    
    # 3. Return the image with the new band added
    return image.addBands(rvi)

    

# New function to prefix computed indices with 'S2_' while keeping raw bands (B*) unchanged
def add_s1_prefix_to_all_bands(image):
    """
    Renames EVERY band in the image by adding 'P2_' prefix.
    This applies to both raw Sentinel-2 bands (e.g., 'HH' → 'P2_HH')
    """
    band_names = image.bandNames()
    new_names = band_names.map(lambda name: ee.String('S1_').cat(name))
    return image.rename(new_names)


    
if __name__ == "__main__":

    # define chunk size
    CHUNK_SIZE = 1
    
    # Output directory name on Google Drive for exported images  
    outdir = '_sentinel1_semiannually_median' 
    
    # gee assets path
    parent = 'projects/fates-mrv/assets'
    
    # Ask EE for the list of *direct* children of that folder
    resp = ee.data.listAssets({'parent': parent})  # dict with key 'assets'
    assets_meta = resp.get('assets', [])           # list of dictionaries
    
    # Keep only assets whose type == 'TABLE'   (vectors/shapefiles)
    table_ids = [a['name'] for a in assets_meta if a['type'] == 'TABLE']
    print('Found', len(table_ids), 'vector assets')

     # Print all found vector asset IDs with their index
    for i, aid in enumerate(table_ids):
        print(f'{i:2d}. {aid}')
    
    # 1. Define the semiannual
    year = 2019     
    START_DATE = f'{year}-01-01'
    END_DATE   = f'{year}-12-31'
      
    # Dynamic semiannual
    semiannual = [
            ('Semiannual1', f'{year}-01-01', f'{year}-06-30'),
            ('Semiannual2', f'{year}-07-01', f'{year}-12-31')
        ]
    
    # Loop over all vector assets
    for asset_id in tqdm(table_ids):
        #print(table_ids)

        # Filter to process only assets "fishnet"
        if not asset_id.endswith(f'{year}'):
            continue  # Skip if it does not end with ''

        # Extract basename from asset ID for naming outputs
        basename = os.path.basename(asset_id)      # ← fishnet
        print('\n───────────────────────────────────────────────')
        print('Processing:', asset_id)

        # Load the vector asset as a FeatureCollection
        fishnet = ee.FeatureCollection(asset_id)
        total_cells = fishnet.size().getInfo()
        print('\nNumber of grid cells :', total_cells)
        print('Schema columns:', fishnet.first().propertyNames().getInfo()) 

        # spatial tiling : loop over the grid in chunks
        for offset in range(0, total_cells, CHUNK_SIZE):
        # for offset in range(713, total_cells):
            chunk_idx = offset // CHUNK_SIZE
            chunk_fc  = ee.FeatureCollection(fishnet.toList(CHUNK_SIZE, offset))
            chunk_geom = chunk_fc.geometry()

            # Build the ImageCollection with processing steps
            sar_collection = (ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')
                              .filterBounds(chunk_geom)
                              .filterDate(START_DATE, END_DATE)
                              .filter(ee.Filter.And(ee.Filter.listContains('system:band_names', 'VH'), 
                                                    ee.Filter.listContains('system:band_names', 'VV')))
                              .select(['VH', 'VV'])
                              .map(apply_RefinedLee)
                              .map(todb)
                              .map(toInt)
                              .map(addTexture)
                              .map(VHVV_sum)
                              .map(VHVV_sub)
                              .map(VHVV_ratio)
                              .map(RVI)
                              .map(add_s1_prefix_to_all_bands)
                             )
            
            # Check bands before reduction
            collection_size = sar_collection.size().getInfo()
            if collection_size == 0:
                # No images found for this feature, skip processing
                print(f"No images found for chunk {chunk_idx} in {year}. Skipping...")
                continue  # skip to next chunk

            # semiannual processing
            all_semiannual_fc = None
            has_semiannual_data = False

            for name, start, end in semiannual:
                col = sar_collection.filterDate(start, end)
                if col.size().getInfo() == 0:
                    print(f"  No images in {name} for chunk {chunk_idx}")
                    continue

                # composite for thesemiannual
                composite = col.median()

                # Zonal mean extraction
                reduced = composite.reduceRegions(
                    collection=chunk_fc,
                    reducer=ee.Reducer.mean(),
                    scale=10,
                    crs='EPSG:3857',
                    tileScale=1
                )

                reduced_q = reduced.map(lambda f: f.set({
                    'quarter': name,
                    'year': year,
                    'start_date': start,
                    'end_date': end
                }))

                if all_semiannual_fc is None:
                    all_semiannual_fc = reduced_q
                else:
                    all_semiannual_fc = all_semiannual_fc.merge(reduced_q)
                
                has_semiannual_data = True

            if not has_semiannual_data:
                print(f"No valid semiannually data for chunk {chunk_idx}. Skipping export.")
                continue        

                
            # Create a descriptive task name for export
            task_desc = f'sentinel-1_{basename}_semiannually_chunk_{chunk_idx}'
            
            # Set up and start an export task to Google Drive
            task = ee.batch.Export.table.toDrive(**{
                'collection': all_semiannual_fc,  # The combined FeatureCollection with time series
                'description': task_desc,  # Set file name
                'folder': outdir,          # Folder on Google Drive
                'fileFormat': 'CSV',       # Format (use uppercase for consistency)
                # 'selectors': ['date'] + list(sar_collection.first().bandNames().getInfo())  # Include date and bands
            })
            
            task.start()
            
            print(f'\n    → Task: {task_desc} submitted '
                  f'({offset:,} – {min(offset+CHUNK_SIZE, total_cells)-1:,})')

            # Optional: sleep to avoid submitting too many tasks too quickly
            # Each project's queue supports a maximum of 3,000 tasks
            import time
            time.sleep(1) # sleep interval to aviod creating multi outdir in google drive
            # do a pre-check of time/chunk to set this sleep value


## sentinel-1 Yearly median 

In [ ]:
    
def RefinedLee(img):
    """
    Applies the Refined Lee Speckle Filter to each band of an image individually.
    The image must be in natural units (not in dB).
    The processed bands will be named as 'original_band_name_lee'.
    
    Important:
    - The input image must be in natural units (linear scale), not in dB.
    - The output image will have the same number of bands as the input, with each band renamed
      by appending '_lee' to the original band name.
    
    Parameters:
    img (ee.Image): Input Earth Engine Image with one or more bands.
    
    Returns:
    ee.Image: Filtered image with bands named as 'original_band_name_lee'.
    """

    # Get the list of band names.
    band_names = img.bandNames()

    # Define the function to process each band.
    def per_band(bandName):
        bandName = ee.String(bandName)
        band = img.select(bandName)
        
        # Set up 5x5 kernels.
        weights5 = ee.List.repeat(ee.List.repeat(1, 5), 5)
        kernel5 = ee.Kernel.fixed(5, 5, weights5, 2, 2, False)
    
        # Compute mean and variance using the 5x5 kernel.
        mean5 = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=kernel5
        )
        variance5 = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=kernel5
        )
    
        # Use a sample of the 5x5 windows inside a 9x9 window to determine gradients and directions.
        sample_weights = ee.List([
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0]
        ])
        sample_kernel = ee.Kernel.fixed(9, 9, sample_weights, 4, 4, False)
    
        # Calculate mean and variance for the sampled windows and store as bands.
        sample_mean = mean5.neighborhoodToBands(kernel=sample_kernel)
        sample_var = variance5.neighborhoodToBands(kernel=sample_kernel)
    
        # Determine the gradients for the sampled windows.
        gradients = sample_mean.select(1).subtract(sample_mean.select(7)).abs()
        gradients = gradients.addBands(
            sample_mean.select(6).subtract(sample_mean.select(2)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(3).subtract(sample_mean.select(5)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(0).subtract(sample_mean.select(8)).abs()
        )
    
        # Find the maximum gradient among gradient bands.
        max_gradient = gradients.reduce(ee.Reducer.max())
    
        # Create a mask for pixels that have the maximum gradient.
        gradmask = gradients.eq(max_gradient)
        # Duplicate gradmask bands: each gradient represents 2 directions.
        gradmask = gradmask.addBands(gradmask)
    
        # Determine the 8 directions.
        directions = sample_mean.select(1).subtract(sample_mean.select(4)) \
            .gt(sample_mean.select(4).subtract(sample_mean.select(7))).multiply(1)
        directions = directions.addBands(
            sample_mean.select(6).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(2))).multiply(2)
        )
        directions = directions.addBands(
            sample_mean.select(3).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(5))).multiply(3)
        )
        directions = directions.addBands(
            sample_mean.select(0).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(8))).multiply(4)
        )
    
        # The next 4 are the Not() of the previous 4.
        directions = directions.addBands(directions.select(0).Not().multiply(5))
        directions = directions.addBands(directions.select(1).Not().multiply(6))
        directions = directions.addBands(directions.select(2).Not().multiply(7))
        directions = directions.addBands(directions.select(3).Not().multiply(8))
    
        # Mask all values that are not 1-8.
        directions = directions.updateMask(gradmask)
    
        # "Collapse" the stack into a single band image.
        directions = directions.reduce(ee.Reducer.sum())
    
        # Calculate local noise variance.
        sample_stats = sample_var.divide(sample_mean.multiply(sample_mean))
        sigmaV = sample_stats.toArray() \
            .arraySort() \
            .arraySlice(0, 0, 5) \
            .arrayReduce(ee.Reducer.mean(), [0])
    
        # Set up the 9x9 kernels for directional statistics.
        rect_weights = ee.List.repeat(ee.List.repeat(0, 9), 4) \
            .cat(ee.List.repeat(ee.List.repeat(1, 9), 5))
        diag_weights = ee.List([
            [1, 0, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 1]
        ])
    
        rect_kernel = ee.Kernel.fixed(9, 9, rect_weights, 4, 4, False)
        diag_kernel = ee.Kernel.fixed(9, 9, diag_weights, 4, 4, False)
    
        # Create stacks for mean and variance using the directional kernels.
        dir_mean = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
        dir_var = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
    
        dir_mean = dir_mean.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.mean(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
        dir_var = dir_var.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.variance(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
    
        # Add bands for rotated kernels.
        for i in range(1, 4):
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
    
        # "Collapse" the stack into single band images.
        dir_mean = dir_mean.reduce(ee.Reducer.sum())
        dir_var = dir_var.reduce(ee.Reducer.sum())
    
        # Generate the filtered value.
        varX = dir_var.subtract(
            dir_mean.multiply(dir_mean).multiply(sigmaV)
        ).divide(sigmaV.add(1.0))
    
        b = varX.divide(dir_var)
        result = dir_mean.add(
            b.multiply(band.subtract(dir_mean))
        )
    
        # Return the result with the new band name.
        result = result.arrayFlatten([['sum']]).rename(bandName.cat('_lee'))
        return result

    # Map over the band names using the per_band function.
    filtered_images = band_names.map(per_band)

    # Convert the list of images into an ImageCollection.
    filtered_collection = ee.ImageCollection(filtered_images)

    # Convert the ImageCollection to a single image.
    filtered_image = filtered_collection.toBands()

    # Collect the new band names.
    new_band_names = band_names.map(lambda name: ee.String(name).cat('_lee'))

    # Rename the bands of the filtered image.
    filtered_image = filtered_image.rename(new_band_names)

    return filtered_image



def apply_RefinedLee(image):
    """Applies the Refined Lee filter to the 'HH' and 'HV' bands.

    Parameters:
    image (ee.Image): Input image containing radar bands.
    
    Returns:
    ee.Image: Image containing original 'HH' and 'HV' bands plus their filtered versions
              named 'HH_lee' and 'HV_lee'.
    """
    # Get the list of bands in the image
    band_names = image.bandNames()
    
    # Define the target bands
    target_bands = ee.List(['VV', 'VH'])
    
    # Filter the target bands to those that exist in the image
    #existing_bands = target_bands.filter(lambda b: band_names.contains(b))
    
    # Function to process each existing band
    def process_band(band_name):
        band_name = ee.String(band_name)
        band = image.select(band_name)
        #band_natural = toNatural(band)
        band_filtered = RefinedLee(band)
        #band_filtered_db = todb(band_filtered)
        return band_filtered
    
    # Map the processing function over the existing bands
    filtered_bands_list = target_bands.map(process_band)
    
    # Convert the list of images to a single image
    filtered_bands_image = ee.ImageCollection(filtered_bands_list).toBands()

    # Add the filtered bands to the original image
    return image.addBands(filtered_bands_image).rename(['VV','VH','VV_lee','VH_lee'])
    


def todb(image):
    """
    Converts filtered 'HH_lee' and 'HV_lee' bands from natural units back to decibel (dB) scale.
    
    The conversion formula used is: 10 * log10((DN)^2- 83.0 db
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee' and 'HV_lee' bands in natural units.
    
    Returns:
    ee.Image: Image with additional bands 'HH_lee_db' and 'HV_lee_db' in dB scale.
    """

    vv = image.select('VV_lee')
    vvdb = vv.log10().multiply(10).rename('VV_lee_db')
    vh = image.select('VH_lee')
    vhdb = vh.log10().multiply(10).rename('VH_lee_db')
    return image.addBands(vvdb).addBands(vhdb)



def toInt(image):
    """
    Converts 'HH_lee_db' and 'HV_lee_db' bands from float dB values to 16-bit integer scale.
    
    The conversion:
    - Scales values from the range [-50, 10] dB to [0, 65535] integer range.
    - Converts scaled values to 32-bit integers.
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with additional integer bands 'HH_lee_db_int' and 'HV_lee_db_int'.
    """

    hh = image.select('VV_lee_db')
    hhint = hh.unitScale(-50, 1).multiply(65535).toUint16().rename('VV_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    hv = image.select('VH_lee_db')
    hvint = hv.unitScale(-50, 1).multiply(65535).toUint16().rename('VH_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    return image.addBands(hhint).addBands(hvint)



def addTexture(image, glcmsize=5):
    """
    Adds GLCM texture measures (average) to specified bands of an image.
    Applies texture calculation only to bands listed in 'bands_to_analyze'.
    
    Parameters:
    image (ee.Image): Input image containing bands to analyze.
    glcmsize (int): Size of the window (kernel) for GLCM calculation (default 5).
    
    Returns:
    ee.Image: Image with added texture bands for specified bands.
    """
    bands_to_analyze = ee.List(['VV_lee_db_int', 'VH_lee_db_int'])  # Bands to apply GLCM to
    
    def applyTexture(band, img):
        band = ee.String(band)
        condition = bands_to_analyze.contains(band)
        return ee.Image(ee.Algorithms.If(
            condition,
            ee.Image(img).addBands(
                ee.Image(img).select([band])
                .glcmTexture(size=glcmsize, average=True)
            ),
            img
        ))

    # Apply GLCM texture to HH and HV bands
    band_names = image.bandNames()
    textured_image = ee.Image(band_names.iterate(applyTexture, image))
    
    return textured_image


def VHVV_sum(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'HH&HV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH&HV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    vh_add_vv = vh.add(vv).rename(['VH&VV'])   
    return image.addBands(vh_add_vv)

def VHVV_sub(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'VH-VV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'VH-VV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    vhvv_sub = vh.subtract(vv).rename(['VH-VV'])   
    return image.addBands(vhvv_sub)


def VHVV_ratio(image):
    """
    Calculates the ratio of HH to HV bands and appends it as a new band.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH_HV_ratio'.
    """
    # 1. Select the bands
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    
    # 2. Perform the division (HH / HV)
    # Using .divide() handles the pixel-wise math on the server side
    vhvv_ratio = vh.divide(vv).rename('VH/VV')
    
    # 3. Return the image with the new band added
    return image.addBands(vhvv_ratio)


def RVI(image):
    """
    Calculates the ratio of HH to HV bands and appends it as a new band.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH_HV_ratio'.
    """
    # 1. Select the bands
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    
    # 2. Perform the division (HH / HV)
    # Using .divide() handles the pixel-wise math on the server side
    rvi = vh.divide(vh.add(vv)).rename('RVI')
    
    # 3. Return the image with the new band added
    return image.addBands(rvi)

    

# New function to prefix computed indices with 'S2_' while keeping raw bands (B*) unchanged
def add_s1_prefix_to_all_bands(image):
    """
    Renames EVERY band in the image by adding 'P2_' prefix.
    This applies to both raw Sentinel-2 bands (e.g., 'HH' → 'P2_HH')
    """
    band_names = image.bandNames()
    new_names = band_names.map(lambda name: ee.String('S1_').cat(name))
    return image.rename(new_names)


    
if __name__ == "__main__":

    # define chunk size
    CHUNK_SIZE = 1
    
    # Output directory name on Google Drive for exported images  
    outdir = '_sentinel1_yearly_median' 
    
    # gee assets path
    parent = 'projects/fates-mrv/assets'
    
    # Ask EE for the list of *direct* children of that folder
    resp = ee.data.listAssets({'parent': parent})  # dict with key 'assets'
    assets_meta = resp.get('assets', [])           # list of dictionaries
    
    # Keep only assets whose type == 'TABLE'   (vectors/shapefiles)
    table_ids = [a['name'] for a in assets_meta if a['type'] == 'TABLE']
    print('Found', len(table_ids), 'vector assets')

     # Print all found vector asset IDs with their index
    for i, aid in enumerate(table_ids):
        print(f'{i:2d}. {aid}')
    
    # 1. Define the Quarters
    year = 2019     
    START_DATE = f'{year}-01-01'
    END_DATE   = f'{year}-12-31'
    
    # Loop over all vector assets
    for asset_id in tqdm(table_ids):
        #print(table_ids)

        # Filter to process only assets "fishnet"
        if not asset_id.endswith(f'{year}'):
            continue  # Skip if it does not end with ''

        # Extract basename from asset ID for naming outputs
        basename = os.path.basename(asset_id)      # ← fishnet
        print('\n───────────────────────────────────────────────')
        print('Processing:', asset_id)

        # Load the vector asset as a FeatureCollection
        fishnet = ee.FeatureCollection(asset_id)
        total_cells = fishnet.size().getInfo()
        print('\nNumber of grid cells :', total_cells)
        print('Schema columns:', fishnet.first().propertyNames().getInfo()) 

        # spatial tiling : loop over the grid in chunks
        # for offset in range(0, total_cells, CHUNK_SIZE):
        for offset in range(4596, total_cells):
            chunk_idx = offset // CHUNK_SIZE
            chunk_fc  = ee.FeatureCollection(fishnet.toList(CHUNK_SIZE, offset))
            chunk_geom = chunk_fc.geometry()

            # Build the ImageCollection with processing steps
            sar_collection = (ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')
                              .filterBounds(chunk_geom)
                              .filterDate(START_DATE, END_DATE)
                              .filter(ee.Filter.And(ee.Filter.listContains('system:band_names', 'VH'), 
                                                    ee.Filter.listContains('system:band_names', 'VV')))
                              .select(['VH', 'VV'])
                              .map(apply_RefinedLee)
                              .map(todb)
                              .map(toInt)
                              .map(addTexture)
                              .map(VHVV_sum)
                              .map(VHVV_sub)
                              .map(VHVV_ratio)
                              .map(RVI)
                              # .map(resample_to_10m)
                              .map(add_s1_prefix_to_all_bands)
                             )
            
            # Check bands before reduction
            collection_size = sar_collection.size().getInfo()
            if collection_size == 0:
                # No images found for this feature, skip processing
                print(f"No images found for chunk {chunk_idx} in {year}. Skipping...")
                continue  # skip to next chunk

            else:

                # 1. Extract the year from the first image in the collection
                # use .first() to get one image, then format its date to 'YYYY'
                first_image = sar_collection.first()
                year_string = first_image.date().format('YYYY')
                
                # 1. Reduce the collection to a yearly mean image
                # This collapses the temporal dimension into one image
                yearly_median_image = sar_collection.median()

                # # 2. Perform reduceRegions on the yearly mean image
                # This calculates the mean pixel value within each polygon of the chunk
                s1_yearly_stats = yearly_median_image.reduceRegions(
                    collection=chunk_fc,
                    reducer=ee.Reducer.mean(),
                    scale=10,
                    crs='EPSG:3857',
                    tileScale=1)
                
                # 3. Add a property to indicate the year/period
                s1_yearly_stats = s1_yearly_stats.map(lambda f: f.set('system:1styear', year_string))
                
                # Create a descriptive task name for export
                task_desc = f'sentinel-1_{basename}_yearly_chunk_{chunk_idx}'
                
                # Set up and start an export task to Google Drive
                task = ee.batch.Export.table.toDrive(**{
                    'collection': s1_yearly_stats,  # The combined FeatureCollection with time series
                    'description': task_desc,  # Set file name
                    'folder': outdir,          # Folder on Google Drive
                    'fileFormat': 'CSV',       # Format (use uppercase for consistency)
                    # 'selectors': ['date'] + list(sar_collection.first().bandNames().getInfo())  # Include date and bands
                })
                
                task.start()
                
                print(f'\n    → Task: {task_desc} submitted '
                      f'({offset:,} – {min(offset+CHUNK_SIZE, total_cells)-1:,})')
    
                # Optional: sleep to avoid submitting too many tasks too quickly
                # Each project's queue supports a maximum of 3,000 tasks
                import time
                time.sleep(6) # sleep interval to aviod creating multi outdir in google drive
                # do a pre-check of time/chunk to set this sleep value


## sentinel-1 Yearly summary stats

In [ ]:
    
def RefinedLee(img):
    """
    Applies the Refined Lee Speckle Filter to each band of an image individually.
    The image must be in natural units (not in dB).
    The processed bands will be named as 'original_band_name_lee'.
    
    Important:
    - The input image must be in natural units (linear scale), not in dB.
    - The output image will have the same number of bands as the input, with each band renamed
      by appending '_lee' to the original band name.
    
    Parameters:
    img (ee.Image): Input Earth Engine Image with one or more bands.
    
    Returns:
    ee.Image: Filtered image with bands named as 'original_band_name_lee'.
    """

    # Get the list of band names.
    band_names = img.bandNames()

    # Define the function to process each band.
    def per_band(bandName):
        bandName = ee.String(bandName)
        band = img.select(bandName)
        
        # Set up 5x5 kernels.
        weights5 = ee.List.repeat(ee.List.repeat(1, 5), 5)
        kernel5 = ee.Kernel.fixed(5, 5, weights5, 2, 2, False)
    
        # Compute mean and variance using the 5x5 kernel.
        mean5 = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=kernel5
        )
        variance5 = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=kernel5
        )
    
        # Use a sample of the 5x5 windows inside a 9x9 window to determine gradients and directions.
        sample_weights = ee.List([
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0]
        ])
        sample_kernel = ee.Kernel.fixed(9, 9, sample_weights, 4, 4, False)
    
        # Calculate mean and variance for the sampled windows and store as bands.
        sample_mean = mean5.neighborhoodToBands(kernel=sample_kernel)
        sample_var = variance5.neighborhoodToBands(kernel=sample_kernel)
    
        # Determine the gradients for the sampled windows.
        gradients = sample_mean.select(1).subtract(sample_mean.select(7)).abs()
        gradients = gradients.addBands(
            sample_mean.select(6).subtract(sample_mean.select(2)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(3).subtract(sample_mean.select(5)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(0).subtract(sample_mean.select(8)).abs()
        )
    
        # Find the maximum gradient among gradient bands.
        max_gradient = gradients.reduce(ee.Reducer.max())
    
        # Create a mask for pixels that have the maximum gradient.
        gradmask = gradients.eq(max_gradient)
        # Duplicate gradmask bands: each gradient represents 2 directions.
        gradmask = gradmask.addBands(gradmask)
    
        # Determine the 8 directions.
        directions = sample_mean.select(1).subtract(sample_mean.select(4)) \
            .gt(sample_mean.select(4).subtract(sample_mean.select(7))).multiply(1)
        directions = directions.addBands(
            sample_mean.select(6).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(2))).multiply(2)
        )
        directions = directions.addBands(
            sample_mean.select(3).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(5))).multiply(3)
        )
        directions = directions.addBands(
            sample_mean.select(0).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(8))).multiply(4)
        )
    
        # The next 4 are the Not() of the previous 4.
        directions = directions.addBands(directions.select(0).Not().multiply(5))
        directions = directions.addBands(directions.select(1).Not().multiply(6))
        directions = directions.addBands(directions.select(2).Not().multiply(7))
        directions = directions.addBands(directions.select(3).Not().multiply(8))
    
        # Mask all values that are not 1-8.
        directions = directions.updateMask(gradmask)
    
        # "Collapse" the stack into a single band image.
        directions = directions.reduce(ee.Reducer.sum())
    
        # Calculate local noise variance.
        sample_stats = sample_var.divide(sample_mean.multiply(sample_mean))
        sigmaV = sample_stats.toArray() \
            .arraySort() \
            .arraySlice(0, 0, 5) \
            .arrayReduce(ee.Reducer.mean(), [0])
    
        # Set up the 9x9 kernels for directional statistics.
        rect_weights = ee.List.repeat(ee.List.repeat(0, 9), 4) \
            .cat(ee.List.repeat(ee.List.repeat(1, 9), 5))
        diag_weights = ee.List([
            [1, 0, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 1]
        ])
    
        rect_kernel = ee.Kernel.fixed(9, 9, rect_weights, 4, 4, False)
        diag_kernel = ee.Kernel.fixed(9, 9, diag_weights, 4, 4, False)
    
        # Create stacks for mean and variance using the directional kernels.
        dir_mean = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
        dir_var = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
    
        dir_mean = dir_mean.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.mean(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
        dir_var = dir_var.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.variance(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
    
        # Add bands for rotated kernels.
        for i in range(1, 4):
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
    
        # "Collapse" the stack into single band images.
        dir_mean = dir_mean.reduce(ee.Reducer.sum())
        dir_var = dir_var.reduce(ee.Reducer.sum())
    
        # Generate the filtered value.
        varX = dir_var.subtract(
            dir_mean.multiply(dir_mean).multiply(sigmaV)
        ).divide(sigmaV.add(1.0))
    
        b = varX.divide(dir_var)
        result = dir_mean.add(
            b.multiply(band.subtract(dir_mean))
        )
    
        # Return the result with the new band name.
        result = result.arrayFlatten([['sum']]).rename(bandName.cat('_lee'))
        return result

    # Map over the band names using the per_band function.
    filtered_images = band_names.map(per_band)

    # Convert the list of images into an ImageCollection.
    filtered_collection = ee.ImageCollection(filtered_images)

    # Convert the ImageCollection to a single image.
    filtered_image = filtered_collection.toBands()

    # Collect the new band names.
    new_band_names = band_names.map(lambda name: ee.String(name).cat('_lee'))

    # Rename the bands of the filtered image.
    filtered_image = filtered_image.rename(new_band_names)

    return filtered_image



def apply_RefinedLee(image):
    """Applies the Refined Lee filter to the 'HH' and 'HV' bands.

    Parameters:
    image (ee.Image): Input image containing radar bands.
    
    Returns:
    ee.Image: Image containing original 'HH' and 'HV' bands plus their filtered versions
              named 'HH_lee' and 'HV_lee'.
    """
    # Get the list of bands in the image
    band_names = image.bandNames()
    
    # Define the target bands
    target_bands = ee.List(['VV', 'VH'])
    
    # Filter the target bands to those that exist in the image
    #existing_bands = target_bands.filter(lambda b: band_names.contains(b))
    
    # Function to process each existing band
    def process_band(band_name):
        band_name = ee.String(band_name)
        band = image.select(band_name)
        #band_natural = toNatural(band)
        band_filtered = RefinedLee(band)
        #band_filtered_db = todb(band_filtered)
        return band_filtered
    
    # Map the processing function over the existing bands
    filtered_bands_list = target_bands.map(process_band)
    
    # Convert the list of images to a single image
    filtered_bands_image = ee.ImageCollection(filtered_bands_list).toBands()

    # Add the filtered bands to the original image
    return image.addBands(filtered_bands_image).rename(['VV','VH','VV_lee','VH_lee'])
    


def todb(image):
    """
    Converts filtered 'HH_lee' and 'HV_lee' bands from natural units back to decibel (dB) scale.
    
    The conversion formula used is: 10 * log10((DN)^2- 83.0 db
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee' and 'HV_lee' bands in natural units.
    
    Returns:
    ee.Image: Image with additional bands 'HH_lee_db' and 'HV_lee_db' in dB scale.
    """

    vv = image.select('VV_lee')
    vvdb = vv.log10().multiply(10).rename('VV_lee_db')
    vh = image.select('VH_lee')
    vhdb = vh.log10().multiply(10).rename('VH_lee_db')
    return image.addBands(vvdb).addBands(vhdb)



def toInt(image):
    """
    Converts 'HH_lee_db' and 'HV_lee_db' bands from float dB values to 16-bit integer scale.
    
    The conversion:
    - Scales values from the range [-50, 10] dB to [0, 65535] integer range.
    - Converts scaled values to 32-bit integers.
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with additional integer bands 'HH_lee_db_int' and 'HV_lee_db_int'.
    """

    hh = image.select('VV_lee_db')
    hhint = hh.unitScale(-50, 1).multiply(65535).toUint16().rename('VV_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    hv = image.select('VH_lee_db')
    hvint = hv.unitScale(-50, 1.multiply(65535).toUint16().rename('VH_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    return image.addBands(hhint).addBands(hvint)



def addTexture(image, glcmsize=5):
    """
    Adds GLCM texture measures (average) to specified bands of an image.
    Applies texture calculation only to bands listed in 'bands_to_analyze'.
    
    Parameters:
    image (ee.Image): Input image containing bands to analyze.
    glcmsize (int): Size of the window (kernel) for GLCM calculation (default 5).
    
    Returns:
    ee.Image: Image with added texture bands for specified bands.
    """
    bands_to_analyze = ee.List(['VV_lee_db_int', 'VH_lee_db_int'])  # Bands to apply GLCM to
    
    def applyTexture(band, img):
        band = ee.String(band)
        condition = bands_to_analyze.contains(band)
        return ee.Image(ee.Algorithms.If(
            condition,
            ee.Image(img).addBands(
                ee.Image(img).select([band])
                .glcmTexture(size=glcmsize, average=True)
            ),
            img
        ))

    # Apply GLCM texture to HH and HV bands
    band_names = image.bandNames()
    textured_image = ee.Image(band_names.iterate(applyTexture, image))
    
    return textured_image


def VHVV_sum(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'HH&HV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH&HV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    vh_add_vv = vh.add(vv).rename(['VH&VV'])   
    return image.addBands(vh_add_vv)

def VHVV_sub(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'HH&HV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH&HV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    vhvv_sub = vh.subtract(vv).rename(['VH-VV'])   
    return image.addBands(vhvv_sub)


def VHVV_ratio(image):
    """
    Calculates the ratio of HH to HV bands and appends it as a new band.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH_HV_ratio'.
    """
    # 1. Select the bands
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    
    # 2. Perform the division (HH / HV)
    # Using .divide() handles the pixel-wise math on the server side
    vhvv_ratio = vh.divide(vv).rename('VH/VV')
    
    # 3. Return the image with the new band added
    return image.addBands(vhvv_ratio)


def RVI(image):
    """
    Calculates the ratio of HH to HV bands and appends it as a new band.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH_HV_ratio'.
    """
    # 1. Select the bands
    vh = image.select('VH_lee_db')
    vv = image.select('VV_lee_db')
    
    # 2. Perform the division (HH / HV)
    # Using .divide() handles the pixel-wise math on the server side
    rvi = vh.divide(vh.add(vv)).rename('RVI')
    
    # 3. Return the image with the new band added
    return image.addBands(rvi)

    

# New function to prefix computed indices with 'S2_' while keeping raw bands (B*) unchanged
def add_s1_prefix_to_all_bands(image):
    """
    Renames EVERY band in the image by adding 'P2_' prefix.
    This applies to both raw Sentinel-2 bands (e.g., 'HH' → 'P2_HH')
    """
    band_names = image.bandNames()
    new_names = band_names.map(lambda name: ee.String('S1_').cat(name))
    return image.rename(new_names)


    
if __name__ == "__main__":

    # define chunk size
    CHUNK_SIZE = 1
    
    # Output directory name on Google Drive for exported images  
    outdir = '_sentinel1_yearly_summary_stats' 
    
    # gee assets path
    parent = 'projects/fates-mrv/assets'
    
    # Ask EE for the list of *direct* children of that folder
    resp = ee.data.listAssets({'parent': parent})  # dict with key 'assets'
    assets_meta = resp.get('assets', [])           # list of dictionaries
    
    # Keep only assets whose type == 'TABLE'   (vectors/shapefiles)
    table_ids = [a['name'] for a in assets_meta if a['type'] == 'TABLE']
    print('Found', len(table_ids), 'vector assets')

     # Print all found vector asset IDs with their index
    for i, aid in enumerate(table_ids):
        print(f'{i:2d}. {aid}')
    
    # 1. Define the semiannual
    year = 2019     
    START_DATE = f'{year}-01-01'
    END_DATE   = f'{year}-12-31'
   
    # Loop over all vector assets
    for asset_id in tqdm(table_ids):
        #print(table_ids)

        # Filter to process only assets "fishnet"
        if not asset_id.endswith(f'{year}'):
            continue  # Skip if it does not end with ''

        # Extract basename from asset ID for naming outputs
        basename = os.path.basename(asset_id)      # ← fishnet
        print('\n───────────────────────────────────────────────')
        print('Processing:', asset_id)

        # Load the vector asset as a FeatureCollection
        fishnet = ee.FeatureCollection(asset_id)
        total_cells = fishnet.size().getInfo()
        print('\nNumber of grid cells :', total_cells)
        print('Schema columns:', fishnet.first().propertyNames().getInfo()) 

        # spatial tiling : loop over the grid in chunks
        # for offset in range(0, total_cells, CHUNK_SIZE):
        for offset in range(5235, total_cells):
            chunk_idx = offset // CHUNK_SIZE
            chunk_fc  = ee.FeatureCollection(fishnet.toList(CHUNK_SIZE, offset))
            chunk_geom = chunk_fc.geometry()

            # Build the ImageCollection with processing steps
            sar_collection = (ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')
                              .filterBounds(chunk_geom)
                              .filterDate(START_DATE, END_DATE)
                              .filter(ee.Filter.And(ee.Filter.listContains('system:band_names', 'VH'), 
                                                    ee.Filter.listContains('system:band_names', 'VV')))
                              .select(['VH', 'VV'])
                              .map(apply_RefinedLee)
                              .map(todb)
                              .map(toInt)
                              .map(addTexture)
                              .map(VHVV_sum)
                              .map(VHVV_sub)
                              .map(VHVV_ratio)
                              .map(RVI)
                              .map(add_s1_prefix_to_all_bands)
                             )

            # Check bands before reduction
            collection_size = sar_collection.size().getInfo()
            
            if collection_size == 0:
                # No images found for this feature, skip processing
                print(f"No images found for chunk {chunk_idx}. Skipping...")
                continue  # skip to next chunk
                
            else:

                # 1. Extract the year from the first image in the collection
                # use .first() to get one image, then format its date to 'YYYY'
                first_image = sar_collection.first()
                year_string = first_image.date().format('YYYY')

                # Define a combined reducer that computes min, max, mean, median, and stdDev for every band
                stats_reducer = (ee.Reducer.minMax()
                                 .combine(reducer2=ee.Reducer.mean(), sharedInputs=True)
                                 .combine(reducer2=ee.Reducer.median(), sharedInputs=True)
                                 .combine(reducer2=ee.Reducer.stdDev(), sharedInputs=True))
               
                # Reduce the ImageCollection to single image with stats_reducer, with each band in Float consistently
                sar_stats = sar_collection.reduce(stats_reducer)
                bands = sar_collection.first().bandNames()
                
                # # check band before reduction
                # print("\nBands before ee.Reducer:", sar_collection.first().bandNames().getInfo())
                # # Check bands after reduction
                # print("Bands after ee.Reducer:", sar_stats.bandNames().getInfo())
                
                # # 2. Perform reduceRegions on the yearly mean image
                # This calculates the mean pixel value within each polygon of the chunk
                sar_yearly_stats = sar_stats.reduceRegions(
                    collection=chunk_fc,
                    reducer=ee.Reducer.mean(), 
                    scale=10,
                    crs='EPSG:3857',
                    tileScale=1)
                
                # Add range = max - min for every band
                sar_yearly_stats = sar_yearly_stats.map(lambda f: f.set(
                    ee.Dictionary.fromLists(
                        # Keys: e.g. "S2_B4_range"
                        bands.map(lambda b: ee.String(b).cat('_range')),
                        # Values: max - min (null if no valid pixels)
                        bands.map(lambda b: 
                            ee.Number(f.get(ee.String(b).cat('_max'))).subtract(
                                ee.Number(f.get(ee.String(b).cat('_min')))
                            )
                        )
                    )
                ))

                # Add a property with the year
                sar_yearly_stats = sar_yearly_stats.map(lambda f: f.set('system:1styear', year_string))
                
                # Create a descriptive task name for export
                task_desc = f'sentinel-1_{basename}_YearlyStats_chunk_{chunk_idx}'
                
                # Set up and start an export task to Google Drive
                task = ee.batch.Export.table.toDrive(**{
                    'collection': sar_yearly_stats,  # The combined FeatureCollection with time series
                    'description': task_desc,  # Set file name
                    'folder': outdir,          # Folder on Google Drive
                    'fileFormat': 'CSV',       # Format (use uppercase for consistency)
                    # 'selectors': ['date'] + list(s2_collection.first().bandNames().getInfo())  # Include date and bands
                })
                
                task.start()
                
                print(f'\n    → Task: {task_desc} submitted '
                      f'({offset:,} – {min(offset+CHUNK_SIZE, total_cells)-1:,})')
    
                # # Optional: sleep to avoid submitting too many tasks too quickly
                # # Each project's queue supports a maximum of 3,000 tasks
                # import time
                # time.sleep(1) # sleep interval to aviod creating multi outdir in google drive
                # # do a pre-check of time/chunk to set this sleep value

        

# PALSAR-2

## PALSAR-2 monthly median

In [ ]:

def resample_to_10m(image):
    """
    Resample image to 10m using bilinear interpolation.
    """
    return (image
            .resample('bilinear')   # interpolation method
            )
    
def RefinedLee(img):
    """
    Applies the Refined Lee Speckle Filter to each band of an image individually.
    The image must be in natural units (not in dB).
    The processed bands will be named as 'original_band_name_lee'.
    
    Important:
    - The input image must be in natural units (linear scale), not in dB.
    - The output image will have the same number of bands as the input, with each band renamed
      by appending '_lee' to the original band name.
    
    Parameters:
    img (ee.Image): Input Earth Engine Image with one or more bands.
    
    Returns:
    ee.Image: Filtered image with bands named as 'original_band_name_lee'.
    """

    # Get the list of band names.
    band_names = img.bandNames()

    # Define the function to process each band.
    def per_band(bandName):
        bandName = ee.String(bandName)
        band = img.select(bandName)
        
        # Set up 5x5 kernels.
        weights5 = ee.List.repeat(ee.List.repeat(1, 5), 5)
        kernel5 = ee.Kernel.fixed(5, 5, weights5, 2, 2, False)
    
        # Compute mean and variance using the 5x5 kernel.
        mean5 = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=kernel5
        )
        variance5 = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=kernel5
        )
    
        # Use a sample of the 5x5 windows inside a 9x9 window to determine gradients and directions.
        sample_weights = ee.List([
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0]
        ])
        sample_kernel = ee.Kernel.fixed(9, 9, sample_weights, 4, 4, False)
    
        # Calculate mean and variance for the sampled windows and store as bands.
        sample_mean = mean5.neighborhoodToBands(kernel=sample_kernel)
        sample_var = variance5.neighborhoodToBands(kernel=sample_kernel)
    
        # Determine the gradients for the sampled windows.
        gradients = sample_mean.select(1).subtract(sample_mean.select(7)).abs()
        gradients = gradients.addBands(
            sample_mean.select(6).subtract(sample_mean.select(2)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(3).subtract(sample_mean.select(5)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(0).subtract(sample_mean.select(8)).abs()
        )
    
        # Find the maximum gradient among gradient bands.
        max_gradient = gradients.reduce(ee.Reducer.max())
    
        # Create a mask for pixels that have the maximum gradient.
        gradmask = gradients.eq(max_gradient)
        # Duplicate gradmask bands: each gradient represents 2 directions.
        gradmask = gradmask.addBands(gradmask)
    
        # Determine the 8 directions.
        directions = sample_mean.select(1).subtract(sample_mean.select(4)) \
            .gt(sample_mean.select(4).subtract(sample_mean.select(7))).multiply(1)
        directions = directions.addBands(
            sample_mean.select(6).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(2))).multiply(2)
        )
        directions = directions.addBands(
            sample_mean.select(3).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(5))).multiply(3)
        )
        directions = directions.addBands(
            sample_mean.select(0).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(8))).multiply(4)
        )
    
        # The next 4 are the Not() of the previous 4.
        directions = directions.addBands(directions.select(0).Not().multiply(5))
        directions = directions.addBands(directions.select(1).Not().multiply(6))
        directions = directions.addBands(directions.select(2).Not().multiply(7))
        directions = directions.addBands(directions.select(3).Not().multiply(8))
    
        # Mask all values that are not 1-8.
        directions = directions.updateMask(gradmask)
    
        # "Collapse" the stack into a single band image.
        directions = directions.reduce(ee.Reducer.sum())
    
        # Calculate local noise variance.
        sample_stats = sample_var.divide(sample_mean.multiply(sample_mean))
        sigmaV = sample_stats.toArray() \
            .arraySort() \
            .arraySlice(0, 0, 5) \
            .arrayReduce(ee.Reducer.mean(), [0])
    
        # Set up the 9x9 kernels for directional statistics.
        rect_weights = ee.List.repeat(ee.List.repeat(0, 9), 4) \
            .cat(ee.List.repeat(ee.List.repeat(1, 9), 5))
        diag_weights = ee.List([
            [1, 0, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 1]
        ])
    
        rect_kernel = ee.Kernel.fixed(9, 9, rect_weights, 4, 4, False)
        diag_kernel = ee.Kernel.fixed(9, 9, diag_weights, 4, 4, False)
    
        # Create stacks for mean and variance using the directional kernels.
        dir_mean = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
        dir_var = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
    
        dir_mean = dir_mean.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.mean(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
        dir_var = dir_var.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.variance(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
    
        # Add bands for rotated kernels.
        for i in range(1, 4):
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
    
        # "Collapse" the stack into single band images.
        dir_mean = dir_mean.reduce(ee.Reducer.sum())
        dir_var = dir_var.reduce(ee.Reducer.sum())
    
        # Generate the filtered value.
        varX = dir_var.subtract(
            dir_mean.multiply(dir_mean).multiply(sigmaV)
        ).divide(sigmaV.add(1.0))
    
        b = varX.divide(dir_var)
        result = dir_mean.add(
            b.multiply(band.subtract(dir_mean))
        )
    
        # Return the result with the new band name.
        result = result.arrayFlatten([['sum']]).rename(bandName.cat('_lee'))
        return result

    # Map over the band names using the per_band function.
    filtered_images = band_names.map(per_band)

    # Convert the list of images into an ImageCollection.
    filtered_collection = ee.ImageCollection(filtered_images)

    # Convert the ImageCollection to a single image.
    filtered_image = filtered_collection.toBands()

    # Collect the new band names.
    new_band_names = band_names.map(lambda name: ee.String(name).cat('_lee'))

    # Rename the bands of the filtered image.
    filtered_image = filtered_image.rename(new_band_names)

    return filtered_image



def apply_RefinedLee(image):
    """Applies the Refined Lee filter to the 'HH' and 'HV' bands.

    Parameters:
    image (ee.Image): Input image containing radar bands.
    
    Returns:
    ee.Image: Image containing original 'HH' and 'HV' bands plus their filtered versions
              named 'HH_lee' and 'HV_lee'.
    """
    # Get the list of bands in the image
    band_names = image.bandNames()
    
    # Define the target bands
    target_bands = ee.List(['HH', 'HV'])
    
    # Filter the target bands to those that exist in the image
    #existing_bands = target_bands.filter(lambda b: band_names.contains(b))
    
    # Function to process each existing band
    def process_band(band_name):
        band_name = ee.String(band_name)
        band = image.select(band_name)
        #band_natural = toNatural(band)
        band_filtered = RefinedLee(band)
        #band_filtered_db = todb(band_filtered)
        return band_filtered
    
    # Map the processing function over the existing bands
    filtered_bands_list = target_bands.map(process_band)
    
    # Convert the list of images to a single image
    filtered_bands_image = ee.ImageCollection(filtered_bands_list).toBands()

    # Add the filtered bands to the original image
    return image.addBands(filtered_bands_image).rename(['HH','HV','HH_lee','HV_lee'])
    


def todb(image):
    """
    Converts filtered 'HH_lee' and 'HV_lee' bands from natural units back to decibel (dB) scale.
    
    The conversion formula used is: 10 * log10((DN)^2- 83.0 db
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee' and 'HV_lee' bands in natural units.
    
    Returns:
    ee.Image: Image with additional bands 'HH_lee_db' and 'HV_lee_db' in dB scale.
    """

    hh = image.select('HH_lee')
    hhdb = hh.pow(2).log10().multiply(10).subtract(83).rename('HH_lee_db')
    hv = image.select('HV_lee')
    hvdb = hv.pow(2).log10().multiply(10).subtract(83).rename('HV_lee_db')
    return image.addBands(hhdb).addBands(hvdb)



def toInt(image):
    """
    Converts 'HH_lee_db' and 'HV_lee_db' bands from float dB values to 16-bit integer scale.
    
    The conversion:
    - Scales values from the range [-50, 10] dB to [0, 65535] integer range.
    - Converts scaled values to 32-bit integers.
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with additional integer bands 'HH_lee_db_int' and 'HV_lee_db_int'.
    """

    hh = image.select('HH_lee_db')
    hhint = hh.unitScale(-50, 10).multiply(65535).toUint16().rename('HH_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    hv = image.select('HV_lee_db')
    hvint = hv.unitScale(-50, 10).multiply(65535).toUint16().rename('HV_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    return image.addBands(hhint).addBands(hvint)



def addTexture(image, glcmsize=5):
    """
    Adds GLCM texture measures (average) to specified bands of an image.
    Applies texture calculation only to bands listed in 'bands_to_analyze'.
    
    Parameters:
    image (ee.Image): Input image containing bands to analyze.
    glcmsize (int): Size of the window (kernel) for GLCM calculation (default 5).
    
    Returns:
    ee.Image: Image with added texture bands for specified bands.
    """
    bands_to_analyze = ee.List(['HH_lee_db_int', 'HV_lee_db_int'])  # Bands to apply GLCM to
    
    def applyTexture(band, img):
        band = ee.String(band)
        condition = bands_to_analyze.contains(band)
        return ee.Image(ee.Algorithms.If(
            condition,
            ee.Image(img).addBands(
                ee.Image(img).select([band])
                .glcmTexture(size=glcmsize, average=True)
            ),
            img
        ))

    # Apply GLCM texture to HH and HV bands
    band_names = image.bandNames()
    textured_image = ee.Image(band_names.iterate(applyTexture, image))
    
    return textured_image


def HHHV_sum(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'HH&HV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH&HV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    hh = image.select('HH_lee_db')
    hv = image.select('HV_lee_db')
    hh_add_hv = hh.add(hv).rename(['HH&HV'])   
    return image.addBands(hh_add_hv)

def HHHV_sub(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'HH-HV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH-HV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    hh = image.select('HH_lee_db')
    hv = image.select('HV_lee_db')
    hhhv_sub = hh.subtract(hv).rename(['HH-HV'])   
    return image.addBands(hhhv_sub)


def HVHH_ratio(image):
    """
    Calculates the ratio of HH to HV bands and appends it as a new band.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH_HV_ratio'.
    """
    # 1. Select the bands
    hh = image.select('HH_lee_db')
    hv = image.select('HV_lee_db')
    
    # 2. Perform the division (HH / HV)
    # Using .divide() handles the pixel-wise math on the server side
    hvhh_ratio = hv.divide(hh).rename('HV/HH')
    
    # 3. Return the image with the new band added
    return image.addBands(hvhh_ratio)

# New function to prefix computed indices with 'S2_' while keeping raw bands (B*) unchanged
def add_p2_prefix_to_all_bands(image):
    """
    Renames EVERY band in the image by adding 'P2_' prefix.
    This applies to both raw Sentinel-2 bands (e.g., 'HH' → 'P2_HH')
    """
    band_names = image.bandNames()
    new_names = band_names.map(lambda name: ee.String('P2_').cat(name))
    return image.rename(new_names)


    
if __name__ == "__main__":

    # define chunk size
    CHUNK_SIZE = 1
    
    # Output directory name on Google Drive for exported images  
    outdir = '_palsar2_monthly_median' 
    
    # gee assets path
    parent = 'projects/fates-mrv/assets'
    
    # Ask EE for the list of *direct* children of that folder
    resp = ee.data.listAssets({'parent': parent})  # dict with key 'assets'
    assets_meta = resp.get('assets', [])           # list of dictionaries
    
    # Keep only assets whose type == 'TABLE'   (vectors/shapefiles)
    table_ids = [a['name'] for a in assets_meta if a['type'] == 'TABLE']
    print('Found', len(table_ids), 'vector assets')

     # Print all found vector asset IDs with their index
    for i, aid in enumerate(table_ids):
        print(f'{i:2d}. {aid}')
    
    # 1. Define the Quarters
    year = 2019     
    START_DATE = f'{year}-01-01'
    END_DATE   = f'{year}-12-31'
      
    # Dynamic monthly
    months = [
        ('Jan', f'{year}-01-01', f'{year}-01-31'),
        ('Feb', f'{year}-02-01', f'{year}-02-28'),
        ('Mar', f'{year}-03-01', f'{year}-03-31'),
        ('Apr', f'{year}-04-01', f'{year}-04-30'),
        ('May', f'{year}-05-01', f'{year}-05-31'),
        ('Jun', f'{year}-06-01', f'{year}-06-30'),
        ('Jul', f'{year}-07-01', f'{year}-07-31'),
        ('Aug', f'{year}-08-01', f'{year}-08-31'),
        ('Sep', f'{year}-09-01', f'{year}-09-30'),
        ('Oct', f'{year}-10-01', f'{year}-10-31'),
        ('Nov', f'{year}-11-01', f'{year}-11-30'),
        ('Dec', f'{year}-12-01', f'{year}-12-31'),
    ]
    
    # Loop over all vector assets
    for asset_id in tqdm(table_ids):
        #print(table_ids)

        # Filter to process only assets "fishnet"
        if not asset_id.endswith(f'{year}'):
            continue  # Skip if it does not end with ''

        # Extract basename from asset ID for naming outputs
        basename = os.path.basename(asset_id)      # ← fishnet
        print('\n───────────────────────────────────────────────')
        print('Processing:', asset_id)

        # Load the vector asset as a FeatureCollection
        fishnet = ee.FeatureCollection(asset_id)
        total_cells = fishnet.size().getInfo()
        print('\nNumber of grid cells :', total_cells)
        print('Schema columns:', fishnet.first().propertyNames().getInfo()) 

        # spatial tiling : loop over the grid in chunks
        for offset in range(0, total_cells, CHUNK_SIZE):
        # for offset in range(568, total_cells):
            chunk_idx = offset // CHUNK_SIZE
            chunk_fc  = ee.FeatureCollection(fishnet.toList(CHUNK_SIZE, offset))
            chunk_geom = chunk_fc.geometry()

            # Build the ImageCollection with processing steps
            sar_collection = (ee.ImageCollection('JAXA/ALOS/PALSAR-2/Level2_2/ScanSAR')
                              .filterBounds(chunk_geom)
                              .filterDate(START_DATE, END_DATE)
                              .filter(ee.Filter.And(ee.Filter.listContains('system:band_names', 'HH'), 
                                                    ee.Filter.listContains('system:band_names', 'HV')))
                              .select(['HH', 'HV'])
                              .map(apply_RefinedLee)
                              .map(todb)
                              .map(toInt)
                              .map(addTexture)
                              .map(HHHV_sum)
                              .map(HHHV_sub)
                              .map(HVHH_ratio)
                              # .map(resample_to_10m)
                              .map(add_p2_prefix_to_all_bands)
                             )
            
            # Check bands before reduction
            collection_size = sar_collection.size().getInfo()
            if collection_size == 0:
                # No images found for this feature, skip processing
                print(f"No images found for chunk {chunk_idx} in {year}. Skipping...")
                continue  # skip to next chunk
            
            # monthly processing
            all_monthly_fc = None
            has_month_data = False

            for m_name, m_start, m_end in months:
                m_col = sar_collection.filterDate(m_start, m_end)
                if m_col.size().getInfo() == 0:
                    print(f"  No images in {m_name} for chunk {chunk_idx}")
                    continue

                # composite for the month
                composite = m_col.median()

                # Zonal mean extraction
                reduced = composite.reduceRegions(
                    collection=chunk_fc,
                    reducer=ee.Reducer.mean(),
                    scale=10,
                    crs='EPSG:3857',
                    tileScale=1
                )

                reduced_q = reduced.map(lambda f: f.set({
                    'quarter': m_name,
                    'year': year,
                    'start_date': m_start,
                    'end_date': m_end
                }))

                if all_monthly_fc is None:
                    all_monthly_fc = reduced_q
                else:
                    all_monthly_fc = all_monthly_fc.merge(reduced_q)
                
                has_month_data = True

            if not has_month_data:
                print(f"No valid monthly data for chunk {chunk_idx}. Skipping export.")
                continue        
              
            # Create a descriptive task name for export
            task_desc = f'palsar-2_{basename}_monthly_chunk_{chunk_idx}'
            
            # Set up and start an export task to Google Drive
            task = ee.batch.Export.table.toDrive(**{
                'collection': all_monthly_fc,  # The combined FeatureCollection with time series
                'description': task_desc,  # Set file name
                'folder': outdir,          # Folder on Google Drive
                'fileFormat': 'CSV',       # Format (use uppercase for consistency)
                # 'selectors': ['date'] + list(sar_collection.first().bandNames().getInfo())  # Include date and bands
            })
            
            task.start()
            
            print(f'\n    → Task: {task_desc} submitted '
                  f'({offset:,} – {min(offset+CHUNK_SIZE, total_cells)-1:,})')

            # # Optional: sleep to avoid submitting too many tasks too quickly
            # # Each project's queue supports a maximum of 3,000 tasks
            # import time
            # time.sleep(3) # sleep interval to aviod creating multi outdir in google drive
            # # do a pre-check of time/chunk to set this sleep value


## PALSAR-2 quarterly median

In [ ]:
 
def resample_to_10m(image):
    """
    Resample image to 10m using bilinear interpolation.
    """
    return (image
            .resample('bilinear')   # interpolation method
            )
   
def RefinedLee(img):
    """
    Applies the Refined Lee Speckle Filter to each band of an image individually.
    The image must be in natural units (not in dB).
    The processed bands will be named as 'original_band_name_lee'.
    
    Important:
    - The input image must be in natural units (linear scale), not in dB.
    - The output image will have the same number of bands as the input, with each band renamed
      by appending '_lee' to the original band name.
    
    Parameters:
    img (ee.Image): Input Earth Engine Image with one or more bands.
    
    Returns:
    ee.Image: Filtered image with bands named as 'original_band_name_lee'.
    """

    # Get the list of band names.
    band_names = img.bandNames()

    # Define the function to process each band.
    def per_band(bandName):
        bandName = ee.String(bandName)
        band = img.select(bandName)
        
        # Set up 5x5 kernels.
        weights5 = ee.List.repeat(ee.List.repeat(1, 5), 5)
        kernel5 = ee.Kernel.fixed(5, 5, weights5, 2, 2, False)
    
        # Compute mean and variance using the 5x5 kernel.
        mean5 = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=kernel5
        )
        variance5 = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=kernel5
        )
    
        # Use a sample of the 5x5 windows inside a 9x9 window to determine gradients and directions.
        sample_weights = ee.List([
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0]
        ])
        sample_kernel = ee.Kernel.fixed(9, 9, sample_weights, 4, 4, False)
    
        # Calculate mean and variance for the sampled windows and store as bands.
        sample_mean = mean5.neighborhoodToBands(kernel=sample_kernel)
        sample_var = variance5.neighborhoodToBands(kernel=sample_kernel)
    
        # Determine the gradients for the sampled windows.
        gradients = sample_mean.select(1).subtract(sample_mean.select(7)).abs()
        gradients = gradients.addBands(
            sample_mean.select(6).subtract(sample_mean.select(2)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(3).subtract(sample_mean.select(5)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(0).subtract(sample_mean.select(8)).abs()
        )
    
        # Find the maximum gradient among gradient bands.
        max_gradient = gradients.reduce(ee.Reducer.max())
    
        # Create a mask for pixels that have the maximum gradient.
        gradmask = gradients.eq(max_gradient)
        # Duplicate gradmask bands: each gradient represents 2 directions.
        gradmask = gradmask.addBands(gradmask)
    
        # Determine the 8 directions.
        directions = sample_mean.select(1).subtract(sample_mean.select(4)) \
            .gt(sample_mean.select(4).subtract(sample_mean.select(7))).multiply(1)
        directions = directions.addBands(
            sample_mean.select(6).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(2))).multiply(2)
        )
        directions = directions.addBands(
            sample_mean.select(3).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(5))).multiply(3)
        )
        directions = directions.addBands(
            sample_mean.select(0).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(8))).multiply(4)
        )
    
        # The next 4 are the Not() of the previous 4.
        directions = directions.addBands(directions.select(0).Not().multiply(5))
        directions = directions.addBands(directions.select(1).Not().multiply(6))
        directions = directions.addBands(directions.select(2).Not().multiply(7))
        directions = directions.addBands(directions.select(3).Not().multiply(8))
    
        # Mask all values that are not 1-8.
        directions = directions.updateMask(gradmask)
    
        # "Collapse" the stack into a single band image.
        directions = directions.reduce(ee.Reducer.sum())
    
        # Calculate local noise variance.
        sample_stats = sample_var.divide(sample_mean.multiply(sample_mean))
        sigmaV = sample_stats.toArray() \
            .arraySort() \
            .arraySlice(0, 0, 5) \
            .arrayReduce(ee.Reducer.mean(), [0])
    
        # Set up the 9x9 kernels for directional statistics.
        rect_weights = ee.List.repeat(ee.List.repeat(0, 9), 4) \
            .cat(ee.List.repeat(ee.List.repeat(1, 9), 5))
        diag_weights = ee.List([
            [1, 0, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 1]
        ])
    
        rect_kernel = ee.Kernel.fixed(9, 9, rect_weights, 4, 4, False)
        diag_kernel = ee.Kernel.fixed(9, 9, diag_weights, 4, 4, False)
    
        # Create stacks for mean and variance using the directional kernels.
        dir_mean = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
        dir_var = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
    
        dir_mean = dir_mean.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.mean(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
        dir_var = dir_var.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.variance(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
    
        # Add bands for rotated kernels.
        for i in range(1, 4):
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
    
        # "Collapse" the stack into single band images.
        dir_mean = dir_mean.reduce(ee.Reducer.sum())
        dir_var = dir_var.reduce(ee.Reducer.sum())
    
        # Generate the filtered value.
        varX = dir_var.subtract(
            dir_mean.multiply(dir_mean).multiply(sigmaV)
        ).divide(sigmaV.add(1.0))
    
        b = varX.divide(dir_var)
        result = dir_mean.add(
            b.multiply(band.subtract(dir_mean))
        )
    
        # Return the result with the new band name.
        result = result.arrayFlatten([['sum']]).rename(bandName.cat('_lee'))
        return result

    # Map over the band names using the per_band function.
    filtered_images = band_names.map(per_band)

    # Convert the list of images into an ImageCollection.
    filtered_collection = ee.ImageCollection(filtered_images)

    # Convert the ImageCollection to a single image.
    filtered_image = filtered_collection.toBands()

    # Collect the new band names.
    new_band_names = band_names.map(lambda name: ee.String(name).cat('_lee'))

    # Rename the bands of the filtered image.
    filtered_image = filtered_image.rename(new_band_names)

    return filtered_image



def apply_RefinedLee(image):
    """Applies the Refined Lee filter to the 'HH' and 'HV' bands.

    Parameters:
    image (ee.Image): Input image containing radar bands.
    
    Returns:
    ee.Image: Image containing original 'HH' and 'HV' bands plus their filtered versions
              named 'HH_lee' and 'HV_lee'.
    """
    # Get the list of bands in the image
    band_names = image.bandNames()
    
    # Define the target bands
    target_bands = ee.List(['HH', 'HV'])
    
    # Filter the target bands to those that exist in the image
    #existing_bands = target_bands.filter(lambda b: band_names.contains(b))
    
    # Function to process each existing band
    def process_band(band_name):
        band_name = ee.String(band_name)
        band = image.select(band_name)
        #band_natural = toNatural(band)
        band_filtered = RefinedLee(band)
        #band_filtered_db = todb(band_filtered)
        return band_filtered
    
    # Map the processing function over the existing bands
    filtered_bands_list = target_bands.map(process_band)
    
    # Convert the list of images to a single image
    filtered_bands_image = ee.ImageCollection(filtered_bands_list).toBands()

    # Add the filtered bands to the original image
    return image.addBands(filtered_bands_image).rename(['HH','HV','HH_lee','HV_lee'])
    


def todb(image):
    """
    Converts filtered 'HH_lee' and 'HV_lee' bands from natural units back to decibel (dB) scale.
    
    The conversion formula used is: 10 * log10((DN)^2- 83.0 db
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee' and 'HV_lee' bands in natural units.
    
    Returns:
    ee.Image: Image with additional bands 'HH_lee_db' and 'HV_lee_db' in dB scale.
    """

    hh = image.select('HH_lee')
    hhdb = hh.pow(2).log10().multiply(10).subtract(83).rename('HH_lee_db')
    hv = image.select('HV_lee')
    hvdb = hv.pow(2).log10().multiply(10).subtract(83).rename('HV_lee_db')
    return image.addBands(hhdb).addBands(hvdb)



def toInt(image):
    """
    Converts 'HH_lee_db' and 'HV_lee_db' bands from float dB values to 16-bit integer scale.
    
    The conversion:
    - Scales values from the range [-50, 10] dB to [0, 65535] integer range.
    - Converts scaled values to 32-bit integers.
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with additional integer bands 'HH_lee_db_int' and 'HV_lee_db_int'.
    """

    hh = image.select('HH_lee_db')
    hhint = hh.unitScale(-50, 10).multiply(65535).toUint16().rename('HH_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    hv = image.select('HV_lee_db')
    hvint = hv.unitScale(-50, 10).multiply(65535).toUint16().rename('HV_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    return image.addBands(hhint).addBands(hvint)



def addTexture(image, glcmsize=5):
    """
    Adds GLCM texture measures (average) to specified bands of an image.
    Applies texture calculation only to bands listed in 'bands_to_analyze'.
    
    Parameters:
    image (ee.Image): Input image containing bands to analyze.
    glcmsize (int): Size of the window (kernel) for GLCM calculation (default 5).
    
    Returns:
    ee.Image: Image with added texture bands for specified bands.
    """
    bands_to_analyze = ee.List(['HH_lee_db_int', 'HV_lee_db_int'])  # Bands to apply GLCM to
    
    def applyTexture(band, img):
        band = ee.String(band)
        condition = bands_to_analyze.contains(band)
        return ee.Image(ee.Algorithms.If(
            condition,
            ee.Image(img).addBands(
                ee.Image(img).select([band])
                .glcmTexture(size=glcmsize, average=True)
            ),
            img
        ))

    # Apply GLCM texture to HH and HV bands
    band_names = image.bandNames()
    textured_image = ee.Image(band_names.iterate(applyTexture, image))
    
    return textured_image


def HHHV_sum(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'HH&HV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH&HV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    hh = image.select('HH_lee_db')
    hv = image.select('HV_lee_db')
    hh_add_hv = hh.add(hv).rename(['HH&HV'])   
    return image.addBands(hh_add_hv)

def HHHV_sub(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'HH-HV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH-HV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    hh = image.select('HH_lee_db')
    hv = image.select('HV_lee_db')
    hhhv_sub = hh.subtract(hv).rename(['HH-HV'])   
    return image.addBands(hhhv_sub)


def HVHH_ratio(image):
    """
    Calculates the ratio of HH to HV bands and appends it as a new band.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH_HV_ratio'.
    """
    # 1. Select the bands
    hh = image.select('HH_lee_db')
    hv = image.select('HV_lee_db')
    
    # 2. Perform the division (HH / HV)
    # Using .divide() handles the pixel-wise math on the server side
    hvhh_ratio = hv.divide(hh).rename('HV/HH')
    
    # 3. Return the image with the new band added
    return image.addBands(hvhh_ratio)

# New function to prefix computed indices with 'S2_' while keeping raw bands (B*) unchanged
def add_p2_prefix_to_all_bands(image):
    """
    Renames EVERY band in the image by adding 'P2_' prefix.
    This applies to both raw Sentinel-2 bands (e.g., 'HH' → 'P2_HH')
    """
    band_names = image.bandNames()
    new_names = band_names.map(lambda name: ee.String('P2_').cat(name))
    return image.rename(new_names)


    
if __name__ == "__main__":

    # define chunk size
    CHUNK_SIZE = 1
    
    # Output directory name on Google Drive for exported images  
    outdir = '_palsar2_quarterly_median' 
    
    # gee assets path
    parent = 'projects/fates-mrv/assets'
    
    # Ask EE for the list of *direct* children of that folder
    resp = ee.data.listAssets({'parent': parent})  # dict with key 'assets'
    assets_meta = resp.get('assets', [])           # list of dictionaries
    
    # Keep only assets whose type == 'TABLE'   (vectors/shapefiles)
    table_ids = [a['name'] for a in assets_meta if a['type'] == 'TABLE']
    print('Found', len(table_ids), 'vector assets')

     # Print all found vector asset IDs with their index
    for i, aid in enumerate(table_ids):
        print(f'{i:2d}. {aid}')
    
    # 1. Define the Quarters
    year = 2018     
    START_DATE = f'{year}-01-01'
    END_DATE   = f'{year}-12-31'
      
    # Dynamic quarters
    quarters = [
            ('Q1', f'{year}-01-01', f'{year}-03-31'),
            ('Q2', f'{year}-04-01', f'{year}-06-30'),
            ('Q3', f'{year}-07-01', f'{year}-09-30'),
            ('Q4', f'{year}-10-01', f'{year}-12-31'),
        ]
    
    # Loop over all vector assets
    for asset_id in tqdm(table_ids):
        #print(table_ids)

        # Filter to process only assets "fishnet"
        if not asset_id.endswith(f'{year}'):
            continue  # Skip if it does not end with ''

        # Extract basename from asset ID for naming outputs
        basename = os.path.basename(asset_id)      # ← fishnet
        print('\n───────────────────────────────────────────────')
        print('Processing:', asset_id)

        # Load the vector asset as a FeatureCollection
        fishnet = ee.FeatureCollection(asset_id)
        total_cells = fishnet.size().getInfo()
        print('\nNumber of grid cells :', total_cells)
        print('Schema columns:', fishnet.first().propertyNames().getInfo()) 

        # spatial tiling : loop over the grid in chunks
        for offset in range(0, total_cells, CHUNK_SIZE):
        # for offset in range(719, 722):
        
            chunk_idx = offset // CHUNK_SIZE
            chunk_fc  = ee.FeatureCollection(fishnet.toList(CHUNK_SIZE, offset))
            chunk_geom = chunk_fc.geometry()

            # Build the ImageCollection with processing steps
            sar_collection = (ee.ImageCollection('JAXA/ALOS/PALSAR-2/Level2_2/ScanSAR')
                              .filterBounds(chunk_geom)
                              .filterDate(START_DATE, END_DATE)
                              .filter(ee.Filter.And(ee.Filter.listContains('system:band_names', 'HH'), 
                                                    ee.Filter.listContains('system:band_names', 'HV')))
                              .select(['HH', 'HV'])
                              .map(apply_RefinedLee)
                              .map(todb)
                              .map(toInt)
                              .map(addTexture)
                              .map(HHHV_sum)
                              .map(HHHV_sub)
                              .map(HVHH_ratio)
                              # .map(resample_to_10m)
                              .map(add_p2_prefix_to_all_bands)
                             )
            
            # Check bands before reduction
            collection_size = sar_collection.size().getInfo()
            if collection_size == 0:
                # No images found for this feature, skip processing
                print(f"No images found for chunk {chunk_idx} in {year}. Skipping...")
                continue  # skip to next chunk

            # Quarterly processing
            all_quarterly_fc = None
            has_quarter_data = False

            for q_name, q_start, q_end in quarters:
                q_col = sar_collection.filterDate(q_start, q_end)
                if q_col.size().getInfo() == 0:
                    print(f"  No images in {q_name} for chunk {chunk_idx}")
                    continue

                # composite for the quarter
                composite = q_col.median()

                # Zonal mean extraction
                reduced = composite.reduceRegions(
                    collection=chunk_fc,
                    reducer=ee.Reducer.mean(),
                    scale=10,
                    crs='EPSG:3857',
                    tileScale=1
                )

                reduced_q = reduced.map(lambda f: f.set({
                    'quarter': q_name,
                    'year': year,
                    'start_date': q_start,
                    'end_date': q_end
                }))

                if all_quarterly_fc is None:
                    all_quarterly_fc = reduced_q
                else:
                    all_quarterly_fc = all_quarterly_fc.merge(reduced_q)
                
                has_quarter_data = True

            if not has_quarter_data:
                print(f"No valid quarterly data for chunk {chunk_idx}. Skipping export.")
                continue        

                
            # Create a descriptive task name for export
            task_desc = f'palsar-2_{basename}_quarterly_chunk_{chunk_idx}'
            
            # Set up and start an export task to Google Drive
            task = ee.batch.Export.table.toDrive(**{
                'collection': all_quarterly_fc,  # The combined FeatureCollection with time series
                'description': task_desc,  # Set file name
                # 'folder': outdir,          # Folder on Google Drive
                'fileFormat': 'CSV',       # Format (use uppercase for consistency)
                # 'selectors': ['date'] + list(sar_collection.first().bandNames().getInfo())  # Include date and bands
            })
            
            task.start()
            
            print(f'\n    → Task: {task_desc} submitted '
                  f'({offset:,} – {min(offset+CHUNK_SIZE, total_cells)-1:,})')

            # Optional: sleep to avoid submitting too many tasks too quickly
            # Each project's queue supports a maximum of 3,000 tasks
            import time
            time.sleep(3) # sleep interval to aviod creating multi outdir in google drive
            # do a pre-check of time/chunk to set this sleep value


## PALSAR-2 semiannual median 

In [ ]:
 
def resample_to_10m(image):
    """
    Resample image to 10m using bilinear interpolation.
    """
    return (image
            .resample('bilinear')   # interpolation method
            )
   
def RefinedLee(img):
    """
    Applies the Refined Lee Speckle Filter to each band of an image individually.
    The image must be in natural units (not in dB).
    The processed bands will be named as 'original_band_name_lee'.
    
    Important:
    - The input image must be in natural units (linear scale), not in dB.
    - The output image will have the same number of bands as the input, with each band renamed
      by appending '_lee' to the original band name.
    
    Parameters:
    img (ee.Image): Input Earth Engine Image with one or more bands.
    
    Returns:
    ee.Image: Filtered image with bands named as 'original_band_name_lee'.
    """

    # Get the list of band names.
    band_names = img.bandNames()

    # Define the function to process each band.
    def per_band(bandName):
        bandName = ee.String(bandName)
        band = img.select(bandName)
        
        # Set up 5x5 kernels.
        weights5 = ee.List.repeat(ee.List.repeat(1, 5), 5)
        kernel5 = ee.Kernel.fixed(5, 5, weights5, 2, 2, False)
    
        # Compute mean and variance using the 5x5 kernel.
        mean5 = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=kernel5
        )
        variance5 = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=kernel5
        )
    
        # Use a sample of the 5x5 windows inside a 9x9 window to determine gradients and directions.
        sample_weights = ee.List([
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0]
        ])
        sample_kernel = ee.Kernel.fixed(9, 9, sample_weights, 4, 4, False)
    
        # Calculate mean and variance for the sampled windows and store as bands.
        sample_mean = mean5.neighborhoodToBands(kernel=sample_kernel)
        sample_var = variance5.neighborhoodToBands(kernel=sample_kernel)
    
        # Determine the gradients for the sampled windows.
        gradients = sample_mean.select(1).subtract(sample_mean.select(7)).abs()
        gradients = gradients.addBands(
            sample_mean.select(6).subtract(sample_mean.select(2)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(3).subtract(sample_mean.select(5)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(0).subtract(sample_mean.select(8)).abs()
        )
    
        # Find the maximum gradient among gradient bands.
        max_gradient = gradients.reduce(ee.Reducer.max())
    
        # Create a mask for pixels that have the maximum gradient.
        gradmask = gradients.eq(max_gradient)
        # Duplicate gradmask bands: each gradient represents 2 directions.
        gradmask = gradmask.addBands(gradmask)
    
        # Determine the 8 directions.
        directions = sample_mean.select(1).subtract(sample_mean.select(4)) \
            .gt(sample_mean.select(4).subtract(sample_mean.select(7))).multiply(1)
        directions = directions.addBands(
            sample_mean.select(6).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(2))).multiply(2)
        )
        directions = directions.addBands(
            sample_mean.select(3).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(5))).multiply(3)
        )
        directions = directions.addBands(
            sample_mean.select(0).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(8))).multiply(4)
        )
    
        # The next 4 are the Not() of the previous 4.
        directions = directions.addBands(directions.select(0).Not().multiply(5))
        directions = directions.addBands(directions.select(1).Not().multiply(6))
        directions = directions.addBands(directions.select(2).Not().multiply(7))
        directions = directions.addBands(directions.select(3).Not().multiply(8))
    
        # Mask all values that are not 1-8.
        directions = directions.updateMask(gradmask)
    
        # "Collapse" the stack into a single band image.
        directions = directions.reduce(ee.Reducer.sum())
    
        # Calculate local noise variance.
        sample_stats = sample_var.divide(sample_mean.multiply(sample_mean))
        sigmaV = sample_stats.toArray() \
            .arraySort() \
            .arraySlice(0, 0, 5) \
            .arrayReduce(ee.Reducer.mean(), [0])
    
        # Set up the 9x9 kernels for directional statistics.
        rect_weights = ee.List.repeat(ee.List.repeat(0, 9), 4) \
            .cat(ee.List.repeat(ee.List.repeat(1, 9), 5))
        diag_weights = ee.List([
            [1, 0, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 1]
        ])
    
        rect_kernel = ee.Kernel.fixed(9, 9, rect_weights, 4, 4, False)
        diag_kernel = ee.Kernel.fixed(9, 9, diag_weights, 4, 4, False)
    
        # Create stacks for mean and variance using the directional kernels.
        dir_mean = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
        dir_var = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
    
        dir_mean = dir_mean.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.mean(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
        dir_var = dir_var.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.variance(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
    
        # Add bands for rotated kernels.
        for i in range(1, 4):
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
    
        # "Collapse" the stack into single band images.
        dir_mean = dir_mean.reduce(ee.Reducer.sum())
        dir_var = dir_var.reduce(ee.Reducer.sum())
    
        # Generate the filtered value.
        varX = dir_var.subtract(
            dir_mean.multiply(dir_mean).multiply(sigmaV)
        ).divide(sigmaV.add(1.0))
    
        b = varX.divide(dir_var)
        result = dir_mean.add(
            b.multiply(band.subtract(dir_mean))
        )
    
        # Return the result with the new band name.
        result = result.arrayFlatten([['sum']]).rename(bandName.cat('_lee'))
        return result

    # Map over the band names using the per_band function.
    filtered_images = band_names.map(per_band)

    # Convert the list of images into an ImageCollection.
    filtered_collection = ee.ImageCollection(filtered_images)

    # Convert the ImageCollection to a single image.
    filtered_image = filtered_collection.toBands()

    # Collect the new band names.
    new_band_names = band_names.map(lambda name: ee.String(name).cat('_lee'))

    # Rename the bands of the filtered image.
    filtered_image = filtered_image.rename(new_band_names)

    return filtered_image



def apply_RefinedLee(image):
    """Applies the Refined Lee filter to the 'HH' and 'HV' bands.

    Parameters:
    image (ee.Image): Input image containing radar bands.
    
    Returns:
    ee.Image: Image containing original 'HH' and 'HV' bands plus their filtered versions
              named 'HH_lee' and 'HV_lee'.
    """
    # Get the list of bands in the image
    band_names = image.bandNames()
    
    # Define the target bands
    target_bands = ee.List(['HH', 'HV'])
    
    # Filter the target bands to those that exist in the image
    #existing_bands = target_bands.filter(lambda b: band_names.contains(b))
    
    # Function to process each existing band
    def process_band(band_name):
        band_name = ee.String(band_name)
        band = image.select(band_name)
        #band_natural = toNatural(band)
        band_filtered = RefinedLee(band)
        #band_filtered_db = todb(band_filtered)
        return band_filtered
    
    # Map the processing function over the existing bands
    filtered_bands_list = target_bands.map(process_band)
    
    # Convert the list of images to a single image
    filtered_bands_image = ee.ImageCollection(filtered_bands_list).toBands()

    # Add the filtered bands to the original image
    return image.addBands(filtered_bands_image).rename(['HH','HV','HH_lee','HV_lee'])
    


def todb(image):
    """
    Converts filtered 'HH_lee' and 'HV_lee' bands from natural units back to decibel (dB) scale.
    
    The conversion formula used is: 10 * log10((DN)^2- 83.0 db
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee' and 'HV_lee' bands in natural units.
    
    Returns:
    ee.Image: Image with additional bands 'HH_lee_db' and 'HV_lee_db' in dB scale.
    """

    hh = image.select('HH_lee')
    hhdb = hh.pow(2).log10().multiply(10).subtract(83).rename('HH_lee_db')
    hv = image.select('HV_lee')
    hvdb = hv.pow(2).log10().multiply(10).subtract(83).rename('HV_lee_db')
    return image.addBands(hhdb).addBands(hvdb)



def toInt(image):
    """
    Converts 'HH_lee_db' and 'HV_lee_db' bands from float dB values to 16-bit integer scale.
    
    The conversion:
    - Scales values from the range [-50, 10] dB to [0, 65535] integer range.
    - Converts scaled values to 32-bit integers.
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with additional integer bands 'HH_lee_db_int' and 'HV_lee_db_int'.
    """

    hh = image.select('HH_lee_db')
    hhint = hh.unitScale(-50, 10).multiply(65535).toUint16().rename('HH_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    hv = image.select('HV_lee_db')
    hvint = hv.unitScale(-50, 10).multiply(65535).toUint16().rename('HV_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    return image.addBands(hhint).addBands(hvint)



def addTexture(image, glcmsize=5):
    """
    Adds GLCM texture measures (average) to specified bands of an image.
    Applies texture calculation only to bands listed in 'bands_to_analyze'.
    
    Parameters:
    image (ee.Image): Input image containing bands to analyze.
    glcmsize (int): Size of the window (kernel) for GLCM calculation (default 5).
    
    Returns:
    ee.Image: Image with added texture bands for specified bands.
    """
    bands_to_analyze = ee.List(['HH_lee_db_int', 'HV_lee_db_int'])  # Bands to apply GLCM to
    
    def applyTexture(band, img):
        band = ee.String(band)
        condition = bands_to_analyze.contains(band)
        return ee.Image(ee.Algorithms.If(
            condition,
            ee.Image(img).addBands(
                ee.Image(img).select([band])
                .glcmTexture(size=glcmsize, average=True)
            ),
            img
        ))

    # Apply GLCM texture to HH and HV bands
    band_names = image.bandNames()
    textured_image = ee.Image(band_names.iterate(applyTexture, image))
    
    return textured_image


def HHHV_sum(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'HH&HV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH&HV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    hh = image.select('HH_lee_db')
    hv = image.select('HV_lee_db')
    hh_add_hv = hh.add(hv).rename(['HH&HV'])   
    return image.addBands(hh_add_hv)

def HHHV_sub(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'HH-HV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH-HV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    hh = image.select('HH_lee_db')
    hv = image.select('HV_lee_db')
    hhhv_sub = hh.subtract(hv).rename(['HH-HV'])   
    return image.addBands(hhhv_sub)


def HVHH_ratio(image):
    """
    Calculates the ratio of HH to HV bands and appends it as a new band.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH_HV_ratio'.
    """
    # 1. Select the bands
    hh = image.select('HH_lee_db')
    hv = image.select('HV_lee_db')
    
    # 2. Perform the division (HH / HV)
    # Using .divide() handles the pixel-wise math on the server side
    hvhh_ratio = hv.divide(hh).rename('HV/HH')
    
    # 3. Return the image with the new band added
    return image.addBands(hvhh_ratio)

# New function to prefix computed indices with 'S2_' while keeping raw bands (B*) unchanged
def add_p2_prefix_to_all_bands(image):
    """
    Renames EVERY band in the image by adding 'P2_' prefix.
    This applies to both raw Sentinel-2 bands (e.g., 'HH' → 'P2_HH')
    """
    band_names = image.bandNames()
    new_names = band_names.map(lambda name: ee.String('P2_').cat(name))
    return image.rename(new_names)


    
if __name__ == "__main__":

    # define chunk size
    CHUNK_SIZE = 1
    
    # Output directory name on Google Drive for exported images  
    outdir = '_palsar2_semiannually_median' 
    
    # gee assets path
    parent = 'projects/fates-mrv/assets'
    
    # Ask EE for the list of *direct* children of that folder
    resp = ee.data.listAssets({'parent': parent})  # dict with key 'assets'
    assets_meta = resp.get('assets', [])           # list of dictionaries
    
    # Keep only assets whose type == 'TABLE'   (vectors/shapefiles)
    table_ids = [a['name'] for a in assets_meta if a['type'] == 'TABLE']
    print('Found', len(table_ids), 'vector assets')

     # Print all found vector asset IDs with their index
    for i, aid in enumerate(table_ids):
        print(f'{i:2d}. {aid}')
    
    # 1. Define the semiannual
    year = 2019     
    START_DATE = f'{year}-01-01'
    END_DATE   = f'{year}-12-31'
      
    # Dynamic semiannual
    semiannual = [
            ('Semiannual1', f'{year}-01-01', f'{year}-06-30'),
            ('Semiannual2', f'{year}-07-01', f'{year}-12-31')
        ]
    
    
    # Loop over all vector assets
    for asset_id in tqdm(table_ids):
        #print(table_ids)

        # Filter to process only assets "fishnet"
        if not asset_id.endswith(f'{year}'):
            continue  # Skip if it does not end with ''

        # Extract basename from asset ID for naming outputs
        basename = os.path.basename(asset_id)      # ← fishnet
        print('\n───────────────────────────────────────────────')
        print('Processing:', asset_id)

        # Load the vector asset as a FeatureCollection
        fishnet = ee.FeatureCollection(asset_id)
        total_cells = fishnet.size().getInfo()
        print('\nNumber of grid cells :', total_cells)
        print('Schema columns:', fishnet.first().propertyNames().getInfo()) 

        # spatial tiling : loop over the grid in chunks
        for offset in range(0, total_cells, CHUNK_SIZE):
        # for offset in range(4789, total_cells):
            chunk_idx = offset // CHUNK_SIZE
            chunk_fc  = ee.FeatureCollection(fishnet.toList(CHUNK_SIZE, offset))
            chunk_geom = chunk_fc.geometry()

            # Build the ImageCollection with processing steps
            sar_collection = (ee.ImageCollection('JAXA/ALOS/PALSAR-2/Level2_2/ScanSAR')
                              .filterBounds(chunk_geom)
                              .filterDate(START_DATE, END_DATE)
                              .filter(ee.Filter.And(ee.Filter.listContains('system:band_names', 'HH'), 
                                                    ee.Filter.listContains('system:band_names', 'HV')))
                              .select(['HH', 'HV'])
                              .map(apply_RefinedLee)
                              .map(todb)
                              .map(toInt)
                              .map(addTexture)
                              .map(HHHV_sum)
                              .map(HHHV_sub)
                              .map(HVHH_ratio)
                              # .map(resample_to_10m)
                              .map(add_p2_prefix_to_all_bands)
                             )
            
            # Check bands before reduction
            collection_size = sar_collection.size().getInfo()
            if collection_size == 0:
                # No images found for this feature, skip processing
                print(f"No images found for chunk {chunk_idx} in {year}. Skipping...")
                continue  # skip to next chunk

            # semiannual processing
            all_semiannual_fc = None
            has_semiannual_data = False

            for name, start, end in semiannual:
                col = sar_collection.filterDate(start, end)
                if col.size().getInfo() == 0:
                    print(f"  No images in {name} for chunk {chunk_idx}")
                    continue

                # composite for the semiannual
                composite = col.median()

                # Zonal mean extraction
                reduced = composite.reduceRegions(
                    collection=chunk_fc,
                    reducer=ee.Reducer.mean(),
                    scale=10,
                    crs='EPSG:3857',
                    tileScale=1
                )

                reduced_q = reduced.map(lambda f: f.set({
                    'quarter': name,
                    'year': year,
                    'start_date': start,
                    'end_date': end
                }))

                if all_semiannual_fc is None:
                    all_semiannual_fc = reduced_q
                else:
                    all_semiannual_fc = all_semiannual_fc.merge(reduced_q)
                
                has_semiannual_data = True

            if not has_semiannual_data:
                print(f"No valid semiannually data for chunk {chunk_idx}. Skipping export.")
                continue        

                
            # Create a descriptive task name for export
            task_desc = f'palsar-2_{basename}_semiannually_chunk_{chunk_idx}'
            
            # Set up and start an export task to Google Drive
            task = ee.batch.Export.table.toDrive(**{
                'collection': all_semiannual_fc,  # The combined FeatureCollection with time series
                'description': task_desc,  # Set file name
                'folder': outdir,          # Folder on Google Drive
                'fileFormat': 'CSV',       # Format (use uppercase for consistency)
                # 'selectors': ['date'] + list(sar_collection.first().bandNames().getInfo())  # Include date and bands
            })
            
            task.start()
            
            print(f'\n    → Task: {task_desc} submitted '
                  f'({offset:,} – {min(offset+CHUNK_SIZE, total_cells)-1:,})')

            # Optional: sleep to avoid submitting too many tasks too quickly
            # Each project's queue supports a maximum of 3,000 tasks
            import time
            time.sleep(1) # sleep interval to aviod creating multi outdir in google drive
            # do a pre-check of time/chunk to set this sleep value


## PALSAR-2 Yearly Mosaic V2.5

In [ ]:

def resample_to_10m(image):
    """
    Resample image to 10m using bilinear interpolation.
    """
    return (image
            .resample('bilinear')   # interpolation method
            )
    
def RefinedLee(img):
    """
    Applies the Refined Lee Speckle Filter to each band of an image individually.
    The image must be in natural units (not in dB).
    The processed bands will be named as 'original_band_name_lee'.
    
    Important:
    - The input image must be in natural units (linear scale), not in dB.
    - The output image will have the same number of bands as the input, with each band renamed
      by appending '_lee' to the original band name.
    
    Parameters:
    img (ee.Image): Input Earth Engine Image with one or more bands.
    
    Returns:
    ee.Image: Filtered image with bands named as 'original_band_name_lee'.
    """

    # Get the list of band names.
    band_names = img.bandNames()

    # Define the function to process each band.
    def per_band(bandName):
        bandName = ee.String(bandName)
        band = img.select(bandName)
        
        # Set up 5x5 kernels.
        weights5 = ee.List.repeat(ee.List.repeat(1, 5), 5)
        kernel5 = ee.Kernel.fixed(5, 5, weights5, 2, 2, False)
    
        # Compute mean and variance using the 5x5 kernel.
        mean5 = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=kernel5
        )
        variance5 = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=kernel5
        )
    
        # Use a sample of the 5x5 windows inside a 9x9 window to determine gradients and directions.
        sample_weights = ee.List([
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0]
        ])
        sample_kernel = ee.Kernel.fixed(9, 9, sample_weights, 4, 4, False)
    
        # Calculate mean and variance for the sampled windows and store as bands.
        sample_mean = mean5.neighborhoodToBands(kernel=sample_kernel)
        sample_var = variance5.neighborhoodToBands(kernel=sample_kernel)
    
        # Determine the gradients for the sampled windows.
        gradients = sample_mean.select(1).subtract(sample_mean.select(7)).abs()
        gradients = gradients.addBands(
            sample_mean.select(6).subtract(sample_mean.select(2)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(3).subtract(sample_mean.select(5)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(0).subtract(sample_mean.select(8)).abs()
        )
    
        # Find the maximum gradient among gradient bands.
        max_gradient = gradients.reduce(ee.Reducer.max())
    
        # Create a mask for pixels that have the maximum gradient.
        gradmask = gradients.eq(max_gradient)
        # Duplicate gradmask bands: each gradient represents 2 directions.
        gradmask = gradmask.addBands(gradmask)
    
        # Determine the 8 directions.
        directions = sample_mean.select(1).subtract(sample_mean.select(4)) \
            .gt(sample_mean.select(4).subtract(sample_mean.select(7))).multiply(1)
        directions = directions.addBands(
            sample_mean.select(6).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(2))).multiply(2)
        )
        directions = directions.addBands(
            sample_mean.select(3).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(5))).multiply(3)
        )
        directions = directions.addBands(
            sample_mean.select(0).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(8))).multiply(4)
        )
    
        # The next 4 are the Not() of the previous 4.
        directions = directions.addBands(directions.select(0).Not().multiply(5))
        directions = directions.addBands(directions.select(1).Not().multiply(6))
        directions = directions.addBands(directions.select(2).Not().multiply(7))
        directions = directions.addBands(directions.select(3).Not().multiply(8))
    
        # Mask all values that are not 1-8.
        directions = directions.updateMask(gradmask)
    
        # "Collapse" the stack into a single band image.
        directions = directions.reduce(ee.Reducer.sum())
    
        # Calculate local noise variance.
        sample_stats = sample_var.divide(sample_mean.multiply(sample_mean))
        sigmaV = sample_stats.toArray() \
            .arraySort() \
            .arraySlice(0, 0, 5) \
            .arrayReduce(ee.Reducer.mean(), [0])
    
        # Set up the 9x9 kernels for directional statistics.
        rect_weights = ee.List.repeat(ee.List.repeat(0, 9), 4) \
            .cat(ee.List.repeat(ee.List.repeat(1, 9), 5))
        diag_weights = ee.List([
            [1, 0, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 1]
        ])
    
        rect_kernel = ee.Kernel.fixed(9, 9, rect_weights, 4, 4, False)
        diag_kernel = ee.Kernel.fixed(9, 9, diag_weights, 4, 4, False)
    
        # Create stacks for mean and variance using the directional kernels.
        dir_mean = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
        dir_var = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
    
        dir_mean = dir_mean.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.mean(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
        dir_var = dir_var.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.variance(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
    
        # Add bands for rotated kernels.
        for i in range(1, 4):
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
    
        # "Collapse" the stack into single band images.
        dir_mean = dir_mean.reduce(ee.Reducer.sum())
        dir_var = dir_var.reduce(ee.Reducer.sum())
    
        # Generate the filtered value.
        varX = dir_var.subtract(
            dir_mean.multiply(dir_mean).multiply(sigmaV)
        ).divide(sigmaV.add(1.0))
    
        b = varX.divide(dir_var)
        result = dir_mean.add(
            b.multiply(band.subtract(dir_mean))
        )
    
        # Return the result with the new band name.
        result = result.arrayFlatten([['sum']]).rename(bandName.cat('_lee'))
        return result

    # Map over the band names using the per_band function.
    filtered_images = band_names.map(per_band)

    # Convert the list of images into an ImageCollection.
    filtered_collection = ee.ImageCollection(filtered_images)

    # Convert the ImageCollection to a single image.
    filtered_image = filtered_collection.toBands()

    # Collect the new band names.
    new_band_names = band_names.map(lambda name: ee.String(name).cat('_lee'))

    # Rename the bands of the filtered image.
    filtered_image = filtered_image.rename(new_band_names)

    return filtered_image



def apply_RefinedLee(image):
    """Applies the Refined Lee filter to the 'HH' and 'HV' bands.

    Parameters:
    image (ee.Image): Input image containing radar bands.
    
    Returns:
    ee.Image: Image containing original 'HH' and 'HV' bands plus their filtered versions
              named 'HH_lee' and 'HV_lee'.
    """
    # Get the list of bands in the image
    band_names = image.bandNames()
    
    # Define the target bands
    target_bands = ee.List(['HH', 'HV'])
    
    # Filter the target bands to those that exist in the image
    #existing_bands = target_bands.filter(lambda b: band_names.contains(b))
    
    # Function to process each existing band
    def process_band(band_name):
        band_name = ee.String(band_name)
        band = image.select(band_name)
        #band_natural = toNatural(band)
        band_filtered = RefinedLee(band)
        #band_filtered_db = todb(band_filtered)
        return band_filtered
    
    # Map the processing function over the existing bands
    filtered_bands_list = target_bands.map(process_band)
    
    # Convert the list of images to a single image
    filtered_bands_image = ee.ImageCollection(filtered_bands_list).toBands()

    # Add the filtered bands to the original image
    return image.addBands(filtered_bands_image).rename(['HH','HV','HH_lee','HV_lee'])
    


def todb(image):
    """
    Converts filtered 'HH_lee' and 'HV_lee' bands from natural units back to decibel (dB) scale.
    
    The conversion formula used is: 10 * log10((DN)^2- 83.0 db
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee' and 'HV_lee' bands in natural units.
    
    Returns:
    ee.Image: Image with additional bands 'HH_lee_db' and 'HV_lee_db' in dB scale.
    """

    hh = image.select('HH_lee')
    hhdb = hh.pow(2).log10().multiply(10).subtract(83).rename('HH_lee_db')
    hv = image.select('HV_lee')
    hvdb = hv.pow(2).log10().multiply(10).subtract(83).rename('HV_lee_db')
    return image.addBands(hhdb).addBands(hvdb)



def toInt(image):
    """
    Converts 'HH_lee_db' and 'HV_lee_db' bands from float dB values to 16-bit integer scale.
    
    The conversion:
    - Scales values from the range [-50, 10] dB to [0, 65535] integer range.
    - Converts scaled values to 32-bit integers.
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with additional integer bands 'HH_lee_db_int' and 'HV_lee_db_int'.
    """

    hh = image.select('HH_lee_db')
    hhint = hh.unitScale(-50, 10).multiply(65535).toUint16().rename('HH_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    hv = image.select('HV_lee_db')
    hvint = hv.unitScale(-50, 10).multiply(65535).toUint16().rename('HV_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    return image.addBands(hhint).addBands(hvint)



def addTexture(image, glcmsize=5):
    """
    Adds GLCM texture measures (average) to specified bands of an image.
    Applies texture calculation only to bands listed in 'bands_to_analyze'.
    
    Parameters:
    image (ee.Image): Input image containing bands to analyze.
    glcmsize (int): Size of the window (kernel) for GLCM calculation (default 5).
    
    Returns:
    ee.Image: Image with added texture bands for specified bands.
    """
    bands_to_analyze = ee.List(['HH_lee_db_int', 'HV_lee_db_int'])  # Bands to apply GLCM to
    
    def applyTexture(band, img):
        band = ee.String(band)
        condition = bands_to_analyze.contains(band)
        return ee.Image(ee.Algorithms.If(
            condition,
            ee.Image(img).addBands(
                ee.Image(img).select([band])
                .glcmTexture(size=glcmsize, average=True)
            ),
            img
        ))

    # Apply GLCM texture to HH and HV bands
    band_names = image.bandNames()
    textured_image = ee.Image(band_names.iterate(applyTexture, image))
    
    return textured_image


def HHHV_sum(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'HH&HV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH&HV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    hh = image.select('HH_lee_db')
    hv = image.select('HV_lee_db')
    hh_add_hv = hh.add(hv).rename(['HH&HV'])   
    return image.addBands(hh_add_hv)

def HHHV_sub(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'HH-HV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH-HV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    hh = image.select('HH_lee_db')
    hv = image.select('HV_lee_db')
    hhhv_sub = hh.subtract(hv).rename(['HH-HV'])   
    return image.addBands(hhhv_sub)


def HVHH_ratio(image):
    """
    Calculates the ratio of HH to HV bands and appends it as a new band.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH_HV_ratio'.
    """
    # 1. Select the bands
    hh = image.select('HH_lee_db')
    hv = image.select('HV_lee_db')
    
    # 2. Perform the division (HH / HV)
    # Using .divide() handles the pixel-wise math on the server side
    hvhh_ratio = hv.divide(hh).rename('HV/HH')
    
    # 3. Return the image with the new band added
    return image.addBands(hvhh_ratio)

# New function to prefix computed indices with 'S2_' while keeping raw bands (B*) unchanged
def add_p2_prefix_to_all_bands(image):
    """
    Renames EVERY band in the image by adding 'P2_' prefix.
    This applies to both raw Sentinel-2 bands (e.g., 'HH' → 'P2_HH')
    """
    band_names = image.bandNames()
    new_names = band_names.map(lambda name: ee.String('P2_').cat(name))
    return image.rename(new_names)


if __name__ == "__main__":

    # define chunk size
    CHUNK_SIZE = 1
    
    # Output directory name on Google Drive for exported images  
    outdir = '_palsar2_yearly_mosaic_P2' 
    
    # gee assets path
    parent = 'projects/fates-mrv/assets'
    
    # Ask EE for the list of *direct* children of that folder
    resp = ee.data.listAssets({'parent': parent})  # dict with key 'assets'
    assets_meta = resp.get('assets', [])           # list of dictionaries
    
    # Keep only assets whose type == 'TABLE'   (vectors/shapefiles)
    table_ids = [a['name'] for a in assets_meta if a['type'] == 'TABLE']
    print('Found', len(table_ids), 'vector assets')

     # Print all found vector asset IDs with their index
    for i, aid in enumerate(table_ids):
        print(f'{i:2d}. {aid}')
   
    # 1. Define the Quarters
    year = 2019     
    START_DATE = f'{year}-01-01'
    END_DATE   = f'{year}-12-31'
    
    # Loop over all vector assets
    for asset_id in tqdm(table_ids):
        #print(table_ids)

        # Filter to process only assets "fishnet"
        if not asset_id.endswith(f'{year}'):
            continue  # Skip if it does not end with ''

        # Extract basename from asset ID for naming outputs
        basename = os.path.basename(asset_id)      # ← fishnet
        print('\n───────────────────────────────────────────────')
        print('Processing:', asset_id)

        # Load the vector asset as a FeatureCollection
        fishnet = ee.FeatureCollection(asset_id)
        total_cells = fishnet.size().getInfo()
        print('\nNumber of grid cells :', total_cells)
        print('Schema columns:', fishnet.first().propertyNames().getInfo()) 

        # spatial tiling : loop over the grid in chunks
        # for offset in range(0, total_cells, CHUNK_SIZE):
        # for offset in [2285, 2286, 2287, 2288, 2289]:
        for offset in range(4410, total_cells):
            chunk_idx = offset // CHUNK_SIZE
            chunk_fc  = ee.FeatureCollection(fishnet.toList(CHUNK_SIZE, offset))
            chunk_geom = chunk_fc.geometry()

            # Build the ImageCollection with processing steps
            sar_collection = (ee.ImageCollection('JAXA/ALOS/PALSAR/YEARLY/SAR_EPOCH')
                              .filterBounds(chunk_geom)
                              .filterDate(START_DATE, END_DATE)
                              .filter(ee.Filter.And(ee.Filter.listContains('system:band_names', 'HH'), 
                                                    ee.Filter.listContains('system:band_names', 'HV')))
                              .select(['HH', 'HV'])
                              .map(apply_RefinedLee)
                              .map(todb)
                              .map(toInt)
                              .map(addTexture)
                              .map(HHHV_sum)
                              .map(HHHV_sub)
                              .map(HVHH_ratio)
                              # .map(resample_to_10m)
                              .map(add_p2_prefix_to_all_bands)
                             )
            
            # Check bands before reduction
            collection_size = sar_collection.size().getInfo()
            
            if collection_size == 0:
                # No images found for this feature, skip processing
                print(f"No images found for chunk {chunk_idx}. Skipping...")
                continue  # skip to next chunk
                
            else:


                # Map over the ImageCollection to perform reduceRegions on each image
                # and add a 'date' property to each resulting feature
                def addDateAndReduce(image):
                    # Perform reduceRegions on the current image
                    reduced_features = image.reduceRegions(
                        collection=chunk_fc,  # The chunk of features
                        reducer=ee.Reducer.mean(),  # Mean per feature (polygon)
                        scale=10,  # Resolution
                        crs='EPSG:3857',
                        tileScale=1
                    )
                    # Add the date as a property to each feature
                    return reduced_features.map(lambda feature: feature.set('date', image.date().format('YYYY-MM-dd')))
                
                # Apply the function to each image in the collection and flatten the result
                yearly_palsar2_samples = sar_collection.map(addDateAndReduce).flatten()
                
                # Create a descriptive task name for export
                task_desc = f'palsar-2_{basename}_yearly_palsar2_chunk_{chunk_idx}'
                
                # Set up and start an export task to Google Drive
                task = ee.batch.Export.table.toDrive(**{
                    'collection': yearly_palsar2_samples,  # The combined FeatureCollection with time series
                    'description': task_desc,  # Set file name
                    'folder': outdir,          # Folder on Google Drive
                    'fileFormat': 'CSV',       # Format (use uppercase for consistency)
                    # 'selectors': ['date'] + list(sar_collection.first().bandNames().getInfo())  # Include date and bands
                })
                
                task.start()
                
                print(f'\n    → Task: {task_desc} submitted '
                      f'({offset:,} – {min(offset+CHUNK_SIZE, total_cells)-1:,})')
    
                # # Optional: sleep to avoid submitting too many tasks too quickly
                # # Each project's queue supports a maximum of 3,000 tasks
                # import time
                # time.sleep(1) # sleep interval to aviod creating multi outdir in google drive
                # # do a pre-check of time/chunk to set this sleep value


## PALSAR-2 Yearly medium

In [ ]:
def resample_to_10m(image):
    """
    Resample image to 10m using bilinear interpolation.
    """
    return (image
            .resample('bilinear')   # interpolation method
            )
   
def RefinedLee(img):
    """
    Applies the Refined Lee Speckle Filter to each band of an image individually.
    The image must be in natural units (not in dB).
    The processed bands will be named as 'original_band_name_lee'.
    
    Important:
    - The input image must be in natural units (linear scale), not in dB.
    - The output image will have the same number of bands as the input, with each band renamed
      by appending '_lee' to the original band name.
    
    Parameters:
    img (ee.Image): Input Earth Engine Image with one or more bands.
    
    Returns:
    ee.Image: Filtered image with bands named as 'original_band_name_lee'.
    """

    # Get the list of band names.
    band_names = img.bandNames()

    # Define the function to process each band.
    def per_band(bandName):
        bandName = ee.String(bandName)
        band = img.select(bandName)
        
        # Set up 5x5 kernels.
        weights5 = ee.List.repeat(ee.List.repeat(1, 5), 5)
        kernel5 = ee.Kernel.fixed(5, 5, weights5, 2, 2, False)
    
        # Compute mean and variance using the 5x5 kernel.
        mean5 = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=kernel5
        )
        variance5 = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=kernel5
        )
    
        # Use a sample of the 5x5 windows inside a 9x9 window to determine gradients and directions.
        sample_weights = ee.List([
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0]
        ])
        sample_kernel = ee.Kernel.fixed(9, 9, sample_weights, 4, 4, False)
    
        # Calculate mean and variance for the sampled windows and store as bands.
        sample_mean = mean5.neighborhoodToBands(kernel=sample_kernel)
        sample_var = variance5.neighborhoodToBands(kernel=sample_kernel)
    
        # Determine the gradients for the sampled windows.
        gradients = sample_mean.select(1).subtract(sample_mean.select(7)).abs()
        gradients = gradients.addBands(
            sample_mean.select(6).subtract(sample_mean.select(2)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(3).subtract(sample_mean.select(5)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(0).subtract(sample_mean.select(8)).abs()
        )
    
        # Find the maximum gradient among gradient bands.
        max_gradient = gradients.reduce(ee.Reducer.max())
    
        # Create a mask for pixels that have the maximum gradient.
        gradmask = gradients.eq(max_gradient)
        # Duplicate gradmask bands: each gradient represents 2 directions.
        gradmask = gradmask.addBands(gradmask)
    
        # Determine the 8 directions.
        directions = sample_mean.select(1).subtract(sample_mean.select(4)) \
            .gt(sample_mean.select(4).subtract(sample_mean.select(7))).multiply(1)
        directions = directions.addBands(
            sample_mean.select(6).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(2))).multiply(2)
        )
        directions = directions.addBands(
            sample_mean.select(3).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(5))).multiply(3)
        )
        directions = directions.addBands(
            sample_mean.select(0).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(8))).multiply(4)
        )
    
        # The next 4 are the Not() of the previous 4.
        directions = directions.addBands(directions.select(0).Not().multiply(5))
        directions = directions.addBands(directions.select(1).Not().multiply(6))
        directions = directions.addBands(directions.select(2).Not().multiply(7))
        directions = directions.addBands(directions.select(3).Not().multiply(8))
    
        # Mask all values that are not 1-8.
        directions = directions.updateMask(gradmask)
    
        # "Collapse" the stack into a single band image.
        directions = directions.reduce(ee.Reducer.sum())
    
        # Calculate local noise variance.
        sample_stats = sample_var.divide(sample_mean.multiply(sample_mean))
        sigmaV = sample_stats.toArray() \
            .arraySort() \
            .arraySlice(0, 0, 5) \
            .arrayReduce(ee.Reducer.mean(), [0])
    
        # Set up the 9x9 kernels for directional statistics.
        rect_weights = ee.List.repeat(ee.List.repeat(0, 9), 4) \
            .cat(ee.List.repeat(ee.List.repeat(1, 9), 5))
        diag_weights = ee.List([
            [1, 0, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 1]
        ])
    
        rect_kernel = ee.Kernel.fixed(9, 9, rect_weights, 4, 4, False)
        diag_kernel = ee.Kernel.fixed(9, 9, diag_weights, 4, 4, False)
    
        # Create stacks for mean and variance using the directional kernels.
        dir_mean = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
        dir_var = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
    
        dir_mean = dir_mean.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.mean(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
        dir_var = dir_var.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.variance(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
    
        # Add bands for rotated kernels.
        for i in range(1, 4):
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
    
        # "Collapse" the stack into single band images.
        dir_mean = dir_mean.reduce(ee.Reducer.sum())
        dir_var = dir_var.reduce(ee.Reducer.sum())
    
        # Generate the filtered value.
        varX = dir_var.subtract(
            dir_mean.multiply(dir_mean).multiply(sigmaV)
        ).divide(sigmaV.add(1.0))
    
        b = varX.divide(dir_var)
        result = dir_mean.add(
            b.multiply(band.subtract(dir_mean))
        )
    
        # Return the result with the new band name.
        result = result.arrayFlatten([['sum']]).rename(bandName.cat('_lee'))
        return result

    # Map over the band names using the per_band function.
    filtered_images = band_names.map(per_band)

    # Convert the list of images into an ImageCollection.
    filtered_collection = ee.ImageCollection(filtered_images)

    # Convert the ImageCollection to a single image.
    filtered_image = filtered_collection.toBands()

    # Collect the new band names.
    new_band_names = band_names.map(lambda name: ee.String(name).cat('_lee'))

    # Rename the bands of the filtered image.
    filtered_image = filtered_image.rename(new_band_names)

    return filtered_image



def apply_RefinedLee(image):
    """Applies the Refined Lee filter to the 'HH' and 'HV' bands.

    Parameters:
    image (ee.Image): Input image containing radar bands.
    
    Returns:
    ee.Image: Image containing original 'HH' and 'HV' bands plus their filtered versions
              named 'HH_lee' and 'HV_lee'.
    """
    # Get the list of bands in the image
    band_names = image.bandNames()
    
    # Define the target bands
    target_bands = ee.List(['HH', 'HV'])
    
    # Filter the target bands to those that exist in the image
    #existing_bands = target_bands.filter(lambda b: band_names.contains(b))
    
    # Function to process each existing band
    def process_band(band_name):
        band_name = ee.String(band_name)
        band = image.select(band_name)
        #band_natural = toNatural(band)
        band_filtered = RefinedLee(band)
        #band_filtered_db = todb(band_filtered)
        return band_filtered
    
    # Map the processing function over the existing bands
    filtered_bands_list = target_bands.map(process_band)
    
    # Convert the list of images to a single image
    filtered_bands_image = ee.ImageCollection(filtered_bands_list).toBands()

    # Add the filtered bands to the original image
    return image.addBands(filtered_bands_image).rename(['HH','HV','HH_lee','HV_lee'])
    


def todb(image):
    """
    Converts filtered 'HH_lee' and 'HV_lee' bands from natural units back to decibel (dB) scale.
    
    The conversion formula used is: 10 * log10((DN)^2- 83.0 db
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee' and 'HV_lee' bands in natural units.
    
    Returns:
    ee.Image: Image with additional bands 'HH_lee_db' and 'HV_lee_db' in dB scale.
    """

    hh = image.select('HH_lee')
    hhdb = hh.pow(2).log10().multiply(10).subtract(83).rename('HH_lee_db')
    hv = image.select('HV_lee')
    hvdb = hv.pow(2).log10().multiply(10).subtract(83).rename('HV_lee_db')
    return image.addBands(hhdb).addBands(hvdb)



def toInt(image):
    """
    Converts 'HH_lee_db' and 'HV_lee_db' bands from float dB values to 16-bit integer scale.
    
    The conversion:
    - Scales values from the range [-50, 10] dB to [0, 65535] integer range.
    - Converts scaled values to 32-bit integers.
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with additional integer bands 'HH_lee_db_int' and 'HV_lee_db_int'.
    """

    hh = image.select('HH_lee_db')
    hhint = hh.unitScale(-50, 10).multiply(65535).toUint16().rename('HH_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    hv = image.select('HV_lee_db')
    hvint = hv.unitScale(-50, 10).multiply(65535).toUint16().rename('HV_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    return image.addBands(hhint).addBands(hvint)



def addTexture(image, glcmsize=5):
    """
    Adds GLCM texture measures (average) to specified bands of an image.
    Applies texture calculation only to bands listed in 'bands_to_analyze'.
    
    Parameters:
    image (ee.Image): Input image containing bands to analyze.
    glcmsize (int): Size of the window (kernel) for GLCM calculation (default 5).
    
    Returns:
    ee.Image: Image with added texture bands for specified bands.
    """
    bands_to_analyze = ee.List(['HH_lee_db_int', 'HV_lee_db_int'])  # Bands to apply GLCM to
    
    def applyTexture(band, img):
        band = ee.String(band)
        condition = bands_to_analyze.contains(band)
        return ee.Image(ee.Algorithms.If(
            condition,
            ee.Image(img).addBands(
                ee.Image(img).select([band])
                .glcmTexture(size=glcmsize, average=True)
            ),
            img
        ))

    # Apply GLCM texture to HH and HV bands
    band_names = image.bandNames()
    textured_image = ee.Image(band_names.iterate(applyTexture, image))
    
    return textured_image


def HHHV_sum(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'HH&HV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH&HV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    hh = image.select('HH_lee_db')
    hv = image.select('HV_lee_db')
    hh_add_hv = hh.add(hv).rename(['HH&HV'])   
    return image.addBands(hh_add_hv)

def HHHV_sub(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'HH-HV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH-HV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    hh = image.select('HH_lee_db')
    hv = image.select('HV_lee_db')
    hhhv_sub = hh.subtract(hv).rename(['HH-HV'])   
    return image.addBands(hhhv_sub)


def HVHH_ratio(image):
    """
    Calculates the ratio of HH to HV bands and appends it as a new band.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH_HV_ratio'.
    """
    # 1. Select the bands
    hh = image.select('HH_lee_db')
    hv = image.select('HV_lee_db')
    
    # 2. Perform the division (HH / HV)
    # Using .divide() handles the pixel-wise math on the server side
    hvhh_ratio = hv.divide(hh).rename('HV/HH')
    
    # 3. Return the image with the new band added
    return image.addBands(hvhh_ratio)

# New function to prefix computed indices with 'S2_' while keeping raw bands (B*) unchanged
def add_p2_prefix_to_all_bands(image):
    """
    Renames EVERY band in the image by adding 'P2_' prefix.
    This applies to both raw Sentinel-2 bands (e.g., 'HH' → 'P2_HH')
    """
    band_names = image.bandNames()
    new_names = band_names.map(lambda name: ee.String('P2_').cat(name))
    return image.rename(new_names)


    
if __name__ == "__main__":

    # define chunk size
    CHUNK_SIZE = 1
    
    # Output directory name on Google Drive for exported images  
    outdir = '_palsar2_yearly_median' 
    
    # gee assets path
    parent = 'projects/fates-mrv/assets'
    
    # Ask EE for the list of *direct* children of that folder
    resp = ee.data.listAssets({'parent': parent})  # dict with key 'assets'
    assets_meta = resp.get('assets', [])           # list of dictionaries
    
    # Keep only assets whose type == 'TABLE'   (vectors/shapefiles)
    table_ids = [a['name'] for a in assets_meta if a['type'] == 'TABLE']
    print('Found', len(table_ids), 'vector assets')

     # Print all found vector asset IDs with their index
    for i, aid in enumerate(table_ids):
        print(f'{i:2d}. {aid}')
    
    # 1. Define date
    year = 2019     
    START_DATE = f'{year}-01-01'
    END_DATE   = f'{year}-12-31'
    
    # Loop over all vector assets
    for asset_id in tqdm(table_ids):
        #print(table_ids)

        # Filter to process only assets "fishnet"
        if not asset_id.endswith(f'{year}'):
            continue  # Skip if it does not end with ''

        # Extract basename from asset ID for naming outputs
        basename = os.path.basename(asset_id)      # ← fishnet
        print('\n───────────────────────────────────────────────')
        print('Processing:', asset_id)

        # Load the vector asset as a FeatureCollection
        fishnet = ee.FeatureCollection(asset_id)
        total_cells = fishnet.size().getInfo()
        print('\nNumber of grid cells :', total_cells)
        print('Schema columns:', fishnet.first().propertyNames().getInfo()) 

        # spatial tiling : loop over the grid in chunks
        for offset in range(0, total_cells, CHUNK_SIZE):
        # for offset in range(4789, total_cells):
            chunk_idx = offset // CHUNK_SIZE
            chunk_fc  = ee.FeatureCollection(fishnet.toList(CHUNK_SIZE, offset))
            chunk_geom = chunk_fc.geometry()

            # Build the ImageCollection with processing steps
            sar_collection = (ee.ImageCollection('JAXA/ALOS/PALSAR-2/Level2_2/ScanSAR')
                              .filterBounds(chunk_geom)
                              .filterDate(START_DATE, END_DATE)
                              .filter(ee.Filter.And(ee.Filter.listContains('system:band_names', 'HH'), 
                                                    ee.Filter.listContains('system:band_names', 'HV')))
                              .select(['HH', 'HV'])
                              .map(apply_RefinedLee)
                              .map(todb)
                              .map(toInt)
                              .map(addTexture)
                              .map(HHHV_sum)
                              .map(HHHV_sub)
                              .map(HVHH_ratio)
                              # .map(resample_to_10m)
                              .map(add_p2_prefix_to_all_bands)
                             )
            
            # Check bands before reduction
            collection_size = sar_collection.size().getInfo()
            if collection_size == 0:
                # No images found for this feature, skip processing
                print(f"No images found for chunk {chunk_idx} in {year}. Skipping...")
                continue  # skip to next chunk

            else:

                # 1. Extract the year from the first image in the collection
                # use .first() to get one image, then format its date to 'YYYY'
                first_image = sar_collection.first()
                year_string = first_image.date().format('YYYY')
                
                # 1. Reduce the collection to a yearly mean image
                # This collapses the temporal dimension into one image
                yearly_median_image = sar_collection.median()

                # # 2. Perform reduceRegions on the yearly mean image
                # This calculates the mean pixel value within each polygon of the chunk
                p2_yearly_stats = yearly_median_image.reduceRegions(
                    collection=chunk_fc,
                    reducer=ee.Reducer.mean(),
                    scale=10,
                    crs='EPSG:3857',
                    tileScale=1)
                
                # 3. Add a property to indicate the year/period
                p2_yearly_stats = p2_yearly_stats.map(lambda f: f.set('system:1styear', year_string))
                
                # Create a descriptive task name for export
                task_desc = f'palsar-2_{basename}_yearly_chunk_{chunk_idx}'
                
                # Set up and start an export task to Google Drive
                task = ee.batch.Export.table.toDrive(**{
                    'collection': p2_yearly_stats,  # The combined FeatureCollection with time series
                    'description': task_desc,  # Set file name
                    'folder': outdir,          # Folder on Google Drive
                    'fileFormat': 'CSV',       # Format (use uppercase for consistency)
                    # 'selectors': ['date'] + list(sar_collection.first().bandNames().getInfo())  # Include date and bands
                })
                
                task.start()
                
                print(f'\n    → Task: {task_desc} submitted '
                      f'({offset:,} – {min(offset+CHUNK_SIZE, total_cells)-1:,})')
    
                # Optional: sleep to avoid submitting too many tasks too quickly
                # Each project's queue supports a maximum of 3,000 tasks
                import time
                time.sleep(12) # sleep interval to aviod creating multi outdir in google drive
                # do a pre-check of time/chunk to set this sleep value


## PALSAR-2 Yearly summary stats

In [ ]:
 
def resample_to_10m(image):
    """
    Resample image to 10m using bilinear interpolation.
    """
    return (image
            .resample('bilinear')   # interpolation method
            )
   
def RefinedLee(img):
    """
    Applies the Refined Lee Speckle Filter to each band of an image individually.
    The image must be in natural units (not in dB).
    The processed bands will be named as 'original_band_name_lee'.
    
    Important:
    - The input image must be in natural units (linear scale), not in dB.
    - The output image will have the same number of bands as the input, with each band renamed
      by appending '_lee' to the original band name.
    
    Parameters:
    img (ee.Image): Input Earth Engine Image with one or more bands.
    
    Returns:
    ee.Image: Filtered image with bands named as 'original_band_name_lee'.
    """

    # Get the list of band names.
    band_names = img.bandNames()

    # Define the function to process each band.
    def per_band(bandName):
        bandName = ee.String(bandName)
        band = img.select(bandName)
        
        # Set up 5x5 kernels.
        weights5 = ee.List.repeat(ee.List.repeat(1, 5), 5)
        kernel5 = ee.Kernel.fixed(5, 5, weights5, 2, 2, False)
    
        # Compute mean and variance using the 5x5 kernel.
        mean5 = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=kernel5
        )
        variance5 = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=kernel5
        )
    
        # Use a sample of the 5x5 windows inside a 9x9 window to determine gradients and directions.
        sample_weights = ee.List([
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0]
        ])
        sample_kernel = ee.Kernel.fixed(9, 9, sample_weights, 4, 4, False)
    
        # Calculate mean and variance for the sampled windows and store as bands.
        sample_mean = mean5.neighborhoodToBands(kernel=sample_kernel)
        sample_var = variance5.neighborhoodToBands(kernel=sample_kernel)
    
        # Determine the gradients for the sampled windows.
        gradients = sample_mean.select(1).subtract(sample_mean.select(7)).abs()
        gradients = gradients.addBands(
            sample_mean.select(6).subtract(sample_mean.select(2)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(3).subtract(sample_mean.select(5)).abs()
        )
        gradients = gradients.addBands(
            sample_mean.select(0).subtract(sample_mean.select(8)).abs()
        )
    
        # Find the maximum gradient among gradient bands.
        max_gradient = gradients.reduce(ee.Reducer.max())
    
        # Create a mask for pixels that have the maximum gradient.
        gradmask = gradients.eq(max_gradient)
        # Duplicate gradmask bands: each gradient represents 2 directions.
        gradmask = gradmask.addBands(gradmask)
    
        # Determine the 8 directions.
        directions = sample_mean.select(1).subtract(sample_mean.select(4)) \
            .gt(sample_mean.select(4).subtract(sample_mean.select(7))).multiply(1)
        directions = directions.addBands(
            sample_mean.select(6).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(2))).multiply(2)
        )
        directions = directions.addBands(
            sample_mean.select(3).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(5))).multiply(3)
        )
        directions = directions.addBands(
            sample_mean.select(0).subtract(sample_mean.select(4))
            .gt(sample_mean.select(4).subtract(sample_mean.select(8))).multiply(4)
        )
    
        # The next 4 are the Not() of the previous 4.
        directions = directions.addBands(directions.select(0).Not().multiply(5))
        directions = directions.addBands(directions.select(1).Not().multiply(6))
        directions = directions.addBands(directions.select(2).Not().multiply(7))
        directions = directions.addBands(directions.select(3).Not().multiply(8))
    
        # Mask all values that are not 1-8.
        directions = directions.updateMask(gradmask)
    
        # "Collapse" the stack into a single band image.
        directions = directions.reduce(ee.Reducer.sum())
    
        # Calculate local noise variance.
        sample_stats = sample_var.divide(sample_mean.multiply(sample_mean))
        sigmaV = sample_stats.toArray() \
            .arraySort() \
            .arraySlice(0, 0, 5) \
            .arrayReduce(ee.Reducer.mean(), [0])
    
        # Set up the 9x9 kernels for directional statistics.
        rect_weights = ee.List.repeat(ee.List.repeat(0, 9), 4) \
            .cat(ee.List.repeat(ee.List.repeat(1, 9), 5))
        diag_weights = ee.List([
            [1, 0, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 0, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 0, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 0],
            [1, 1, 1, 1, 1, 1, 1, 1, 1]
        ])
    
        rect_kernel = ee.Kernel.fixed(9, 9, rect_weights, 4, 4, False)
        diag_kernel = ee.Kernel.fixed(9, 9, diag_weights, 4, 4, False)
    
        # Create stacks for mean and variance using the directional kernels.
        dir_mean = band.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
        dir_var = band.reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=rect_kernel
        ).updateMask(directions.eq(1))
    
        dir_mean = dir_mean.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.mean(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
        dir_var = dir_var.addBands(
            band.reduceNeighborhood(
                reducer=ee.Reducer.variance(),
                kernel=diag_kernel
            ).updateMask(directions.eq(2))
        )
    
        # Add bands for rotated kernels.
        for i in range(1, 4):
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=rect_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 1))
            )
            dir_mean = dir_mean.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.mean(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
            dir_var = dir_var.addBands(
                band.reduceNeighborhood(
                    reducer=ee.Reducer.variance(),
                    kernel=diag_kernel.rotate(i)
                ).updateMask(directions.eq(2 * i + 2))
            )
    
        # "Collapse" the stack into single band images.
        dir_mean = dir_mean.reduce(ee.Reducer.sum())
        dir_var = dir_var.reduce(ee.Reducer.sum())
    
        # Generate the filtered value.
        varX = dir_var.subtract(
            dir_mean.multiply(dir_mean).multiply(sigmaV)
        ).divide(sigmaV.add(1.0))
    
        b = varX.divide(dir_var)
        result = dir_mean.add(
            b.multiply(band.subtract(dir_mean))
        )
    
        # Return the result with the new band name.
        result = result.arrayFlatten([['sum']]).rename(bandName.cat('_lee'))
        return result

    # Map over the band names using the per_band function.
    filtered_images = band_names.map(per_band)

    # Convert the list of images into an ImageCollection.
    filtered_collection = ee.ImageCollection(filtered_images)

    # Convert the ImageCollection to a single image.
    filtered_image = filtered_collection.toBands()

    # Collect the new band names.
    new_band_names = band_names.map(lambda name: ee.String(name).cat('_lee'))

    # Rename the bands of the filtered image.
    filtered_image = filtered_image.rename(new_band_names)

    return filtered_image



def apply_RefinedLee(image):
    """Applies the Refined Lee filter to the 'HH' and 'HV' bands.

    Parameters:
    image (ee.Image): Input image containing radar bands.
    
    Returns:
    ee.Image: Image containing original 'HH' and 'HV' bands plus their filtered versions
              named 'HH_lee' and 'HV_lee'.
    """
    # Get the list of bands in the image
    band_names = image.bandNames()
    
    # Define the target bands
    target_bands = ee.List(['HH', 'HV'])
    
    # Filter the target bands to those that exist in the image
    #existing_bands = target_bands.filter(lambda b: band_names.contains(b))
    
    # Function to process each existing band
    def process_band(band_name):
        band_name = ee.String(band_name)
        band = image.select(band_name)
        #band_natural = toNatural(band)
        band_filtered = RefinedLee(band)
        #band_filtered_db = todb(band_filtered)
        return band_filtered
    
    # Map the processing function over the existing bands
    filtered_bands_list = target_bands.map(process_band)
    
    # Convert the list of images to a single image
    filtered_bands_image = ee.ImageCollection(filtered_bands_list).toBands()

    # Add the filtered bands to the original image
    return image.addBands(filtered_bands_image).rename(['HH','HV','HH_lee','HV_lee'])
    


def todb(image):
    """
    Converts filtered 'HH_lee' and 'HV_lee' bands from natural units back to decibel (dB) scale.
    
    The conversion formula used is: 10 * log10((DN)^2- 83.0 db
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee' and 'HV_lee' bands in natural units.
    
    Returns:
    ee.Image: Image with additional bands 'HH_lee_db' and 'HV_lee_db' in dB scale.
    """

    hh = image.select('HH_lee')
    hhdb = hh.pow(2).log10().multiply(10).subtract(83).rename('HH_lee_db')
    hv = image.select('HV_lee')
    hvdb = hv.pow(2).log10().multiply(10).subtract(83).rename('HV_lee_db')
    return image.addBands(hhdb).addBands(hvdb)



def toInt(image):
    """
    Converts 'HH_lee_db' and 'HV_lee_db' bands from float dB values to 16-bit integer scale.
    
    The conversion:
    - Scales values from the range [-50, 10] dB to [0, 65535] integer range.
    - Converts scaled values to 32-bit integers.
    
    Parameters:
    image (ee.Image): Image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with additional integer bands 'HH_lee_db_int' and 'HV_lee_db_int'.
    """

    hh = image.select('HH_lee_db')
    hhint = hh.unitScale(-50, 10).multiply(65535).toUint16().rename('HH_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    hv = image.select('HV_lee_db')
    hvint = hv.unitScale(-50, 10).multiply(65535).toUint16().rename('HV_lee_db_int')   # keep GLCM will be performed at a 16-bit gray level
    return image.addBands(hhint).addBands(hvint)



def addTexture(image, glcmsize=5):
    """
    Adds GLCM texture measures (average) to specified bands of an image.
    Applies texture calculation only to bands listed in 'bands_to_analyze'.
    
    Parameters:
    image (ee.Image): Input image containing bands to analyze.
    glcmsize (int): Size of the window (kernel) for GLCM calculation (default 5).
    
    Returns:
    ee.Image: Image with added texture bands for specified bands.
    """
    bands_to_analyze = ee.List(['HH_lee_db_int', 'HV_lee_db_int'])  # Bands to apply GLCM to
    
    def applyTexture(band, img):
        band = ee.String(band)
        condition = bands_to_analyze.contains(band)
        return ee.Image(ee.Algorithms.If(
            condition,
            ee.Image(img).addBands(
                ee.Image(img).select([band])
                .glcmTexture(size=glcmsize, average=True)
            ),
            img
        ))

    # Apply GLCM texture to HH and HV bands
    band_names = image.bandNames()
    textured_image = ee.Image(band_names.iterate(applyTexture, image))
    
    return textured_image


def HHHV_sum(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'HH&HV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH&HV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    hh = image.select('HH_lee_db')
    hv = image.select('HV_lee_db')
    hh_add_hv = hh.add(hv).rename(['HH&HV'])   
    return image.addBands(hh_add_hv)

def HHHV_sub(image):
    """
    Adds the 'HH_lee_db' and 'HV_lee_db' bands pixel-wise and appends the result as a new band named 'HH-HV'.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH-HV' representing the sum of 'HH_lee_db' and 'HV_lee_db'.
    """
    hh = image.select('HH_lee_db')
    hv = image.select('HV_lee_db')
    hhhv_sub = hh.subtract(hv).rename(['HH-HV'])   
    return image.addBands(hhhv_sub)


def HVHH_ratio(image):
    """
    Calculates the ratio of HH to HV bands and appends it as a new band.
    
    Parameters:
    image (ee.Image): Input image containing 'HH_lee_db' and 'HV_lee_db' bands.
    
    Returns:
    ee.Image: Image with an additional band 'HH_HV_ratio'.
    """
    # 1. Select the bands
    hh = image.select('HH_lee_db')
    hv = image.select('HV_lee_db')
    
    # 2. Perform the division (HH / HV)
    # Using .divide() handles the pixel-wise math on the server side
    hvhh_ratio = hv.divide(hh).rename('HV/HH')
    
    # 3. Return the image with the new band added
    return image.addBands(hvhh_ratio)

# New function to prefix computed indices with 'S2_' while keeping raw bands (B*) unchanged
def add_p2_prefix_to_all_bands(image):
    """
    Renames EVERY band in the image by adding 'P2_' prefix.
    This applies to both raw Sentinel-2 bands (e.g., 'HH' → 'P2_HH')
    """
    band_names = image.bandNames()
    new_names = band_names.map(lambda name: ee.String('P2_').cat(name))
    return image.rename(new_names)


    
if __name__ == "__main__":

    # define chunk size
    CHUNK_SIZE = 1
    
    # Output directory name on Google Drive for exported images  
    outdir = '_palsar2_yearly_summary_stats' 
    
    # gee assets path
    parent = 'projects/fates-mrv/assets'
    
    # Ask EE for the list of *direct* children of that folder
    resp = ee.data.listAssets({'parent': parent})  # dict with key 'assets'
    assets_meta = resp.get('assets', [])           # list of dictionaries
    
    # Keep only assets whose type == 'TABLE'   (vectors/shapefiles)
    table_ids = [a['name'] for a in assets_meta if a['type'] == 'TABLE']
    print('Found', len(table_ids), 'vector assets')

     # Print all found vector asset IDs with their index
    for i, aid in enumerate(table_ids):
        print(f'{i:2d}. {aid}')
    
    # 1. Define the semiannual
    year = 2019    
    START_DATE = f'{year}-01-01'
    END_DATE   = f'{year}-12-31'
        
    # Loop over all vector assets
    for asset_id in tqdm(table_ids):
        #print(table_ids)

        # Filter to process only assets "fishnet"
        if not asset_id.endswith(f'{year}'):
            continue  # Skip if it does not end with ''

        # Extract basename from asset ID for naming outputs
        basename = os.path.basename(asset_id)      # ← fishnet
        print('\n───────────────────────────────────────────────')
        print('Processing:', asset_id)

        # Load the vector asset as a FeatureCollection
        fishnet = ee.FeatureCollection(asset_id)
        total_cells = fishnet.size().getInfo()
        print('\nNumber of grid cells :', total_cells)
        print('Schema columns:', fishnet.first().propertyNames().getInfo()) 

        # spatial tiling : loop over the grid in chunks
        for offset in range(0, total_cells, CHUNK_SIZE):
        # for offset in range(1126, total_cells):
            chunk_idx = offset // CHUNK_SIZE
            chunk_fc  = ee.FeatureCollection(fishnet.toList(CHUNK_SIZE, offset))
            chunk_geom = chunk_fc.geometry()

            # Build the ImageCollection with processing steps
            sar_collection = (ee.ImageCollection('JAXA/ALOS/PALSAR-2/Level2_2/ScanSAR')
                              .filterBounds(chunk_geom)
                              .filterDate(START_DATE, END_DATE)
                              .filter(ee.Filter.And(ee.Filter.listContains('system:band_names', 'HH'), 
                                                    ee.Filter.listContains('system:band_names', 'HV')))
                              .select(['HH', 'HV'])
                              .map(apply_RefinedLee)
                              .map(todb)
                              .map(toInt)
                              .map(addTexture)
                              .map(HHHV_sum)
                              .map(HHHV_sub)
                              .map(HVHH_ratio)
                              # .map(resample_to_10m)
                              .map(add_p2_prefix_to_all_bands)
                             )
            
            # Check bands before reduction
            collection_size = sar_collection.size().getInfo()
            if collection_size == 0:
                # No images found for this feature, skip processing
                print(f"No images found for chunk {chunk_idx} in {year}. Skipping...")
                continue  # skip to next chunk

            else:

                # 1. Extract the year from the first image in the collection
                # use .first() to get one image, then format its date to 'YYYY'
                first_image = sar_collection.first()
                year_string = first_image.date().format('YYYY')

                # Define a combined reducer that computes min, max, mean, median, and stdDev for every band
                stats_reducer = (ee.Reducer.minMax()
                                 .combine(reducer2=ee.Reducer.mean(), sharedInputs=True)
                                 .combine(reducer2=ee.Reducer.median(), sharedInputs=True)
                                 .combine(reducer2=ee.Reducer.stdDev(), sharedInputs=True))
               
                # Reduce the ImageCollection to single image with stats_reducer, with each band in Float consistently
                sar_stats = sar_collection.reduce(stats_reducer)
                bands = sar_collection.first().bandNames()
                
                # # check band before reduction
                # print("\nBands before ee.Reducer:", sar_collection.first().bandNames().getInfo())
                # # Check bands after reduction
                # print("Bands after ee.Reducer:", sar_stats.bandNames().getInfo())
                
                # # 2. Perform reduceRegions on the yearly mean image
                # This calculates the mean pixel value within each polygon of the chunk
                sar_yearly_stats = sar_stats.reduceRegions(
                    collection=chunk_fc,
                    reducer=ee.Reducer.mean(), 
                    scale=10,
                    crs='EPSG:3857',
                    tileScale=1)
                
                # Add range = max - min for every band
                sar_yearly_stats = sar_yearly_stats.map(lambda f: f.set(
                    ee.Dictionary.fromLists(
                        # Keys: e.g. "S2_B4_range"
                        bands.map(lambda b: ee.String(b).cat('_range')),
                        # Values: max - min (null if no valid pixels)
                        bands.map(lambda b: 
                            ee.Number(f.get(ee.String(b).cat('_max'))).subtract(
                                ee.Number(f.get(ee.String(b).cat('_min')))
                            )
                        )
                    )
                ))

                # Add a property with the year
                sar_yearly_stats = sar_yearly_stats.map(lambda f: f.set('system:1styear', year_string))
                
                # Create a descriptive task name for export
                task_desc = f'palsar-2_{basename}_YearlyStats_chunk_{chunk_idx}'
                
                # Set up and start an export task to Google Drive
                task = ee.batch.Export.table.toDrive(**{
                    'collection': sar_yearly_stats,  # The combined FeatureCollection with time series
                    'description': task_desc,  # Set file name
                    'folder': outdir,          # Folder on Google Drive
                    'fileFormat': 'CSV',       # Format (use uppercase for consistency)
                    # 'selectors': ['date'] + list(s2_collection.first().bandNames().getInfo())  # Include date and bands
                })
                
                task.start()
                
                print(f'\n    → Task: {task_desc} submitted '
                      f'({offset:,} – {min(offset+CHUNK_SIZE, total_cells)-1:,})')
    
                # Optional: sleep to avoid submitting too many tasks too quickly
                # Each project's queue supports a maximum of 3,000 tasks
                import time
                time.sleep(3) # sleep interval to aviod creating multi outdir in google drive
                # do a pre-check of time/chunk to set this sleep value

        